<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>سیاست Golden Dataset فعال در Notebookهای بعدی</h2>

<p>
از این Notebook به بعد، توسعه و Benchmarkها با Golden Dataset موجود آغاز می‌شوند تا کار پروژه
تا دریافت نظر دکتر کمیل رودی متوقف نماند.
</p>

<p>
پس از دریافت نظر مؤلف و ساخت Golden Dataset نهایی جدید، Notebookهای بعدی بازنویسی نمی‌شوند.
فقط همان Pipeline با اجرای دوباره، Golden Dataset جدید را به‌عنوان ورودی فعال می‌خواند.
</p>

<p>
برای جلوگیری از اشتباه، تمام Notebookهای بعدی فقط با یک متغیر مرکزی کار می‌کنند:
</p>

<pre><code>golden_dataset</code></pre>

<p>
Code Cell 00 در ابتدای هر اجرای تازه، بالاترین نسخهٔ موجود Golden Dataset را از پوشۀ مشخص‌شده
پیدا و داخل همین متغیر بارگذاری می‌کند.
</p>

<ul>
<li>در وضعیت فعلی، Golden Dataset موجود وارد متغیر <code>golden_dataset</code> می‌شود.</li>
<li>پس از ساخته‌شدن Golden Dataset جدید، همان متغیر خودکار Dataset جدید را می‌خواند.</li>
<li>در Codeهای بعدی هیچ نام نسخه‌ای نوشته نمی‌شود.</li>
<li>برای اجرای دوباره، فقط <code>Restart Kernel → Run All</code> انجام می‌شود؛ نه بازنویسی Notebookها.</li>
</ul>

<p>
بنابراین تمام Benchmarkها، Indexها، Retrieval، Citation و Evaluationهای بعدی همیشه از متغیر واحد
<code>golden_dataset</code> استفاده می‌کنند.
</p>

</div>

In [8]:
# Code Cell 00

from pathlib import Path
import hashlib
import json
import re

golden_dataset_directory = Path.cwd().resolve() / "golden_dataset"

assert golden_dataset_directory.is_dir(), (
    f"Golden Dataset directory not found: {golden_dataset_directory}"
)

version_pattern = re.compile(
    r"^golden_dataset_v(?P<version>\d+)\.jsonl$"
)

dataset_candidates = []

for path in golden_dataset_directory.iterdir():
    match = version_pattern.fullmatch(path.name)

    if match and path.is_file():
        dataset_candidates.append(
            (int(match.group("version")), path)
        )

assert dataset_candidates, (
    "No versioned Golden Dataset JSONL file was found."
)

_, golden_dataset_path = max(
    dataset_candidates,
    key=lambda item: item[0],
)

dataset_hash = hashlib.sha256(
    golden_dataset_path.read_bytes()
).hexdigest()

golden_dataset = [
    json.loads(line)
    for line in golden_dataset_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert golden_dataset, "Golden Dataset is empty."
assert all(
    isinstance(record, dict)
    for record in golden_dataset
), "Golden Dataset contains an invalid record."

golden_dataset_context = {
    "path": golden_dataset_path.as_posix(),
    "sha256": dataset_hash,
    "record_count": len(golden_dataset),
}

print("Active Golden Dataset:", golden_dataset_context["path"])
print("SHA-256:", golden_dataset_context["sha256"])
print("Record count:", golden_dataset_context["record_count"])
print("\nACTIVE GOLDEN DATASET: PASS")

Active Golden Dataset: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_v1.jsonl
SHA-256: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
Record count: 573

ACTIVE GOLDEN DATASET: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h1>Notebook 13 — Embedding Benchmark</h1>

<h2>مقایسۀ مدل‌های Embedding بر اساس Golden Dataset فعال</h2>

<p>
هدف این Notebook انتخاب مدل Embedding بر اساس Benchmark واقعی روی Golden Dataset فعال پروژه است.
انتخاب مدل بر مبنای شهرت، تنظیم پیش‌فرض یا حدس انجام نمی‌شود.
</p>

<p>
در این مرحله، Candidate Modelها همراه با Candidateهای Chunking Freeze‌شده NB11
ارزیابی می‌شوند تا کیفیت بازیابی، هزینه، Latency و Failureهای واقعی هر ترکیب اندازه‌گیری شود.
</p>

<h3>جایگاه در Pipeline</h3>

<p>
این Notebook پس از Golden Dataset و پیش از ساخت Dense Index قرار دارد.
</p>

<h3>Inputهای اصلی</h3>

<ul>
<li>متغیر مرکزی <code>golden_dataset</code></li>
<li>اطلاعات همراه آن در <code>golden_dataset_context</code></li>
<li>Candidateهای Chunking Freeze‌شده NB11</li>
<li>Candidate Modelهای Embedding که در مراحل بعد Discovery خواهند شد</li>
</ul>

<h3>Outputهای مورد انتظار</h3>

<ul>
<li>Registry مدل‌های Candidate</li>
<li>Config Benchmark نسخه‌دار</li>
<li>Metricهای واقعی Retrieval برای هر ترکیب Model و Chunking</li>
<li>تحلیل کیفیت، Latency، Failure و Cost</li>
<li>Evidence لازم برای انتخاب Embedding پیش از Dense Index</li>
</ul>

<h3>خارج از Scope</h3>

<ul>
<li>ساخت Dense Index نهایی</li>
<li>انتخاب نهایی Chunking</li>
<li>Sparse Retrieval، Hybrid Retrieval و Reranking</li>
<li>Generation، Citation و Safety</li>
<li>هرگونه تغییر در Golden Dataset</li>
</ul>

<h3>معیار موفقیت نهایی</h3>

<p>
Notebook فقط زمانی کامل است که تمام Candidateها با Golden Dataset فعال Benchmark شوند،
Metricها واقعاً اندازه‌گیری و ثبت شوند و
<code>Restart Kernel → Run All → Final Health PASS</code>
با موفقیت انجام شود.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Starting Point / Execution Plan</h2>

<p>
Code Cell 00، Golden Dataset فعال را با موفقیت بارگذاری کرده است.
اکنون باید پیش از شناسایی مدل‌ها یا خواندن Candidateهای Chunking، وضعیت واقعی همین Dataset و Report همراه آن بررسی شود.
</p>

<p>
این Step فقط Discovery و Preflight است؛ هیچ Dataset، Index، Model، Config یا Artifact جدیدی نمی‌سازد یا تغییر نمی‌دهد.
</p>

<h3>کاری که انجام می‌شود</h3>

<ul>
<li>بررسی ساختار واقعی رکوردهای <code>golden_dataset</code></li>
<li>بررسی یکتایی شناسه‌های موجود بدون فرض‌کردن نام Field</li>
<li>خواندن Report همراه Dataset از روی Disk</li>
<li>ثبت Fieldها، Countها و Hashهای واقعی برای ادامه Benchmark</li>
</ul>

<h3>Validation</h3>

<ul>
<li>Golden Dataset فعال خالی نباشد.</li>
<li>تمام رکوردها JSON Object باشند.</li>
<li>Report همراه Dataset وجود داشته و قابل Parse باشد.</li>
<li>هیچ Path، Count، Hash یا Schema از قبل فرض نشود.</li>
</ul>

<p>
خروجی موفق این Step، Preflight واقعی Golden Dataset فعال برای ورود به Discovery ورودی‌های NB13 است.
</p>

</div>

In [9]:
# Code Cell 01

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import platform
import subprocess
import time

In [10]:
# Code Cell 02

assert golden_dataset, "golden_dataset is empty."

assert all(
    isinstance(record, dict)
    for record in golden_dataset
), "golden_dataset contains a non-object record."

golden_dataset_report_path = golden_dataset_path.with_name(
    f"{golden_dataset_path.stem}_final_report.json"
)

assert golden_dataset_report_path.is_file(), (
    f"Golden Dataset final report was not found: {golden_dataset_report_path}"
)

golden_dataset_report = json.loads(
    golden_dataset_report_path.read_text(
        encoding="utf-8"
    )
)

assert isinstance(
    golden_dataset_report,
    dict,
), "Golden Dataset final report is not a JSON object."

field_counts = Counter()

for record in golden_dataset:
    field_counts.update(record.keys())

candidate_id_field = next(
    (
        field_name
        for field_name in (
            "candidate_id",
            "golden_id",
            "id",
        )
        if field_name in field_counts
    ),
    None,
)

duplicate_identifier_count = None

if candidate_id_field is not None:
    identifier_values = [
        record.get(candidate_id_field)
        for record in golden_dataset
    ]

    duplicate_identifier_count = (
        len(identifier_values)
        - len(set(identifier_values))
    )

print("=== NB13 GOLDEN DATASET PREFLIGHT ===")
print("Dataset path:", golden_dataset_context["path"])
print("Dataset SHA-256:", golden_dataset_context["sha256"])
print("Dataset records:", golden_dataset_context["record_count"])
print("Report path:", golden_dataset_report_path.name)

print("\nDataset fields:")
for field_name in sorted(field_counts):
    print(
        f"- {field_name}: "
        f"present in {field_counts[field_name]} records"
    )

print("\nReport fields:")
for field_name in sorted(golden_dataset_report):
    print("-", field_name)

print("\nIdentifier field discovered:", candidate_id_field)

if candidate_id_field is not None:
    print(
        "Duplicate identifiers:",
        duplicate_identifier_count,
    )

print("\nNB13 GOLDEN DATASET PREFLIGHT: PASS")

=== NB13 GOLDEN DATASET PREFLIGHT ===
Dataset path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_v1.jsonl
Dataset SHA-256: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
Dataset records: 573
Report path: golden_dataset_v1_final_report.json

Dataset fields:
- candidate_id: present in 573 records
- competency_ids: present in 573 records
- expected_answer: present in 573 records
- generation_competency: present in 573 records
- generation_status: present in 573 records
- question: present in 573 records
- source_evidence: present in 573 records
- source_lineage: present in 573 records
- source_seed_id: present in 573 records
- taxonomy_topics: present in 573 records

Report fields:
- artifact
- coverage
- created_at
- dataset_version
- freeze_status
- records
- status
- target_size
- validation

Identifier field discovered: candidate_id
Duplicate identifiers: 0

NB13 GOLDEN DATASET PREFLIGHT: PAS

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی Golden Dataset فعال با Report و Integrity Erratum</h2>

<p>
Golden Dataset فعال از نظر نام فایل، تعداد رکورد و Target Size با Report همراه خود تطبیق دارد،
اما Hash فایل فعال با Hash ثبت‌شده در Report اولیه یکسان نیست.
</p>

<p>
در کنار Dataset، یک Integrity Erratum رسمی نیز وجود دارد. پیش از هر تصمیم،
باید محتوای واقعی این Erratum از روی Disk خوانده شود تا مشخص شود آیا تفاوت Hash
مستند، قابل‌ردیابی و پذیرفته‌شده است یا خیر.
</p>

<p>
این Step فقط Root Cause را بررسی می‌کند و هیچ فایل، Dataset، Report یا Artifactی را تغییر نمی‌دهد.
</p>

<h3>معیار PASS این مرحله</h3>

<p>
Erratum باید وجود داشته و قابل Parse باشد. سپس در Step بعد، Hash فعال فقط در صورتی پذیرفته می‌شود
که Erratum آن را به‌صورت صریح و قابل‌اعتبارسنجی پوشش دهد.
</p>

</div>

In [12]:
# Code Cell 03

golden_dataset_erratum_path = golden_dataset_path.with_name(
    f"{golden_dataset_path.stem}_integrity_erratum.json"
)

assert golden_dataset_erratum_path.is_file(), (
    "Golden Dataset hash differs from the final report, "
    "but no integrity erratum file was found."
)

golden_dataset_erratum = json.loads(
    golden_dataset_erratum_path.read_text(
        encoding="utf-8"
    )
)

assert isinstance(
    golden_dataset_erratum,
    dict,
), "Golden Dataset integrity erratum is not a JSON object."

erratum_hash = hashlib.sha256(
    golden_dataset_erratum_path.read_bytes()
).hexdigest()

print("=== GOLDEN DATASET INTEGRITY ERRATUM DISCOVERY ===")
print("Dataset path:", golden_dataset_path.name)
print("Active dataset SHA-256:", golden_dataset_context["sha256"])
print("Report SHA-256:", reported_dataset_sha256)

print("\nErratum path:", golden_dataset_erratum_path.name)
print("Erratum SHA-256:", erratum_hash)

print("\nErratum fields:")
for field_name in sorted(golden_dataset_erratum):
    print("-", field_name)

print("\nErratum content:")
print(
    json.dumps(
        golden_dataset_erratum,
        ensure_ascii=False,
        indent=2,
    )
)

print("\nGOLDEN DATASET INTEGRITY ERRATUM DISCOVERY: PASS")

=== GOLDEN DATASET INTEGRITY ERRATUM DISCOVERY ===
Dataset path: golden_dataset_v1.jsonl
Active dataset SHA-256: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
Report SHA-256: 25ed3732db5f03d3b771daa7b1051d61f8de7524644c4a055792b39a22493cc4

Erratum path: golden_dataset_v1_integrity_erratum.json
Erratum SHA-256: 5cc13580587d68c4f476125937d5f04701b56541557b73caa5061b381090788d

Erratum fields:
- actual_sha256
- artifact
- dataset_content_modified
- dataset_matches_git_head
- dataset_version
- integrity_status
- issue_type
- legacy_mismatch_exists_in_git_head
- legacy_report_path
- legacy_report_sha256
- records

Erratum content:
{
  "artifact": "golden_dataset_v1.jsonl",
  "dataset_version": "golden_dataset_v1",
  "records": 573,
  "actual_sha256": "df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729",
  "legacy_report_sha256": "25ed3732db5f03d3b771daa7b1051d61f8de7524644c4a055792b39a22493cc4",
  "legacy_report_path": "golden_dataset_v1_final_report.json",

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>پذیرش نهایی Integrity Golden Dataset فعال</h2>

<p>
Integrity Erratum رسمی تأیید کرد که تفاوت Hash میان Golden Dataset فعال و Report اولیه
ناشی از Metadata هاردکدشدهٔ قدیمی در Report است، نه تغییر در محتوای Dataset.
</p>

<p>
در این Step، Dataset فعال، Report اولیه و Erratum رسمی با هم Reconcilie می‌شوند.
پس از PASS شدن این Gate، Golden Dataset فعال به‌عنوان ورودی معتبر NB13 پذیرفته می‌شود.
</p>

<p>
این Step هیچ Artifactی را تغییر نمی‌دهد و فقط وضعیت Integrity را در حافظهٔ Notebook ثبت می‌کند.
</p>

<h3>معیار PASS</h3>

<ul>
<li>Hash فعال دقیقاً با Hash ثبت‌شده در Erratum برابر باشد.</li>
<li>Hash قدیمی Report با مقدار Legacy ثبت‌شده در Erratum برابر باشد.</li>
<li>تعداد رکوردها در Dataset و Erratum برابر باشد.</li>
<li>Erratum صراحتاً عدم تغییر محتوا را تأیید کند.</li>
<li>Erratum صراحتاً تطابق Dataset با Git Head را تأیید کند.</li>
</ul>

</div>

In [13]:
# Code Cell 04

integrity_gates = {
    "erratum_artifact_matches_active_file": (
        golden_dataset_erratum.get("artifact")
        == golden_dataset_path.name
    ),
    "active_hash_matches_erratum": (
        golden_dataset_context["sha256"]
        == golden_dataset_erratum.get("actual_sha256")
    ),
    "legacy_report_hash_matches_erratum": (
        reported_dataset_sha256
        == golden_dataset_erratum.get("legacy_report_sha256")
    ),
    "legacy_report_filename_matches_erratum": (
        golden_dataset_report_path.name
        == golden_dataset_erratum.get("legacy_report_path")
    ),
    "record_count_matches_erratum": (
        golden_dataset_context["record_count"]
        == golden_dataset_erratum.get("records")
    ),
    "dataset_matches_git_head": (
        golden_dataset_erratum.get("dataset_matches_git_head")
        is True
    ),
    "dataset_content_not_modified": (
        golden_dataset_erratum.get("dataset_content_modified")
        is False
    ),
    "integrity_status_accepted": (
        golden_dataset_erratum.get("integrity_status")
        == "PASS_WITH_LEGACY_REPORT_ERRATUM"
    ),
}

failed_integrity_gates = [
    gate_name
    for gate_name, passed in integrity_gates.items()
    if not passed
]

print("=== ACTIVE GOLDEN DATASET INTEGRITY RECONCILIATION ===")

for gate_name, passed in integrity_gates.items():
    print(
        f"- {gate_name}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

assert not failed_integrity_gates, (
    "Golden Dataset integrity reconciliation failed: "
    f"{failed_integrity_gates}"
)

golden_dataset_integrity = {
    "status": "PASS",
    "active_dataset_sha256": golden_dataset_context["sha256"],
    "integrity_status": golden_dataset_erratum["integrity_status"],
    "report_status": "LEGACY_REPORT_ERRATUM_APPLIED",
}

print("\nGolden Dataset integrity status:", golden_dataset_integrity["status"])
print("Dataset validation basis:", golden_dataset_integrity["integrity_status"])
print("\nACTIVE GOLDEN DATASET INTEGRITY: PASS")

=== ACTIVE GOLDEN DATASET INTEGRITY RECONCILIATION ===
- erratum_artifact_matches_active_file: PASS
- active_hash_matches_erratum: PASS
- legacy_report_hash_matches_erratum: PASS
- legacy_report_filename_matches_erratum: PASS
- record_count_matches_erratum: PASS
- dataset_matches_git_head: PASS
- dataset_content_not_modified: PASS
- integrity_status_accepted: PASS

Golden Dataset integrity status: PASS
Dataset validation basis: PASS_WITH_LEGACY_REPORT_ERRATUM

ACTIVE GOLDEN DATASET INTEGRITY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>کشف Artifactهای Freeze‌شدهٔ Chunking از NB11</h2>

<p>
Golden Dataset فعال با موفقیت پذیرفته شد. برای Benchmark Embedding، ورودی دوم
Candidateهای Chunking هستند که در NB11 با وضعیت
<code>FROZEN_FOR_EVALUATION</code>
ثبت شده‌اند.
</p>

<p>
در این Step ابتدا Manifest واقعی Freeze از روی Disk پیدا و خوانده می‌شود.
هدف فقط کشف Contract، وضعیت Freeze و مسیرهای ثبت‌شده در Manifest است.
</p>

<p>
این مرحله هیچ Chunk، Dataset، Model، Index یا Artifact جدیدی ایجاد یا تغییر نمی‌دهد.
</p>

<h3>معیار PASS</h3>

<ul>
<li>Repository Root بدون مسیر شخصی Hard-codeشده کشف شود.</li>
<li>Manifest Freeze یکتای NB11 روی Disk وجود داشته باشد.</li>
<li>Manifest قابل Parse باشد.</li>
<li>Contract و فیلدهای واقعی آن از روی فایل نمایش داده شوند.</li>
</ul>

</div>

In [14]:
# Code Cell 05

current_path = Path.cwd().resolve()

project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / ".git").exists()
        and (candidate / "notebooks" / "phase2").exists()
    ),
    None,
)

assert project_root is not None, (
    "Project root could not be discovered."
)

freeze_manifest_matches = sorted(
    project_root.rglob(
        "evaluation_freeze_manifest_v1.json"
    )
)

assert len(freeze_manifest_matches) == 1, (
    "Expected exactly one NB11 evaluation freeze manifest, "
    f"found {len(freeze_manifest_matches)}."
)

chunking_freeze_manifest_path = freeze_manifest_matches[0]

chunking_freeze_manifest = json.loads(
    chunking_freeze_manifest_path.read_text(
        encoding="utf-8"
    )
)

assert isinstance(
    chunking_freeze_manifest,
    dict,
), "NB11 evaluation freeze manifest is not a JSON object."

manifest_hash = hashlib.sha256(
    chunking_freeze_manifest_path.read_bytes()
).hexdigest()

print("=== NB11 CHUNKING FREEZE DISCOVERY ===")
print(
    "Manifest path:",
    chunking_freeze_manifest_path.relative_to(
        project_root
    ),
)
print("Manifest SHA-256:", manifest_hash)

print("\nManifest fields:")
for field_name in sorted(chunking_freeze_manifest):
    print("-", field_name)

print("\nManifest preview:")
print(
    json.dumps(
        chunking_freeze_manifest,
        ensure_ascii=False,
        indent=2,
    )[:5000]
)

print("\nNB11 CHUNKING FREEZE DISCOVERY: PASS")

=== NB11 CHUNKING FREEZE DISCOVERY ===
Manifest path: reports/phase2/chunking/evaluation_freeze_manifest_v1.json
Manifest SHA-256: c29c4f1840e1fbd313c4f27b3556ff84f2041cce86ba0d5e27ff278c450106cc

Manifest fields:
- accepted_corpus
- artifacts
- candidate_reread_validation
- final_production_strategy
- final_selection_status
- freeze_scope
- measurement_tokenizer
- nb12_handoff
- next_notebook
- not_production_selection
- notebook
- retrieval_metrics
- status
- structural_shortlist

Manifest preview:
{
  "accepted_corpus": {
    "path": "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl",
    "records": 19456,
    "sha256": "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40"
  },
  "artifacts": {
    "baseline_phase1_char_safe": {
      "artifact_type": "chunk_candidate",
      "bytes": 37213608,
      "freeze_status": "FROZEN_FOR_EVALUATION",
      "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
      "records": 3488,
   

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی مستقل Artifactهای Freeze‌شدهٔ NB11</h2>

<p>
Manifest Freeze وجود و ساختار Candidateهای Chunking را تأیید کرد.
اکنون همهٔ Artifactهای ثبت‌شده در همان Manifest مستقیماً از روی Disk بررسی می‌شوند.
</p>

<p>
این Step تضمین می‌کند NB13 دقیقاً روی Candidateهایی اجرا شود که در NB11 Benchmark و
<code>FROZEN_FOR_EVALUATION</code>
شده‌اند؛ نه روی فایل ناقص، تغییرکرده یا نامرتبط.
</p>

<h3>بررسی‌ها</h3>

<ul>
<li>وجود تمام Artifactهای ثبت‌شده</li>
<li>تطابق اندازهٔ فایل با Manifest</li>
<li>تطابق SHA-256 فایل با Manifest</li>
<li>اعتبار وضعیت Freeze</li>
<li>استخراج Registry Candidateهای دارای نقش Retrieval برای مراحل بعد</li>
</ul>

<p>
این مرحله هیچ Candidate، Manifest یا فایل دیگری را تغییر نمی‌دهد.
</p>

</div>

In [15]:
# Code Cell 06

assert (
    chunking_freeze_manifest.get("status")
    == "FROZEN_FOR_EVALUATION"
), "NB11 manifest does not have FROZEN_FOR_EVALUATION status."

assert (
    chunking_freeze_manifest.get("final_production_strategy")
    in {None, "NOT_SELECTED"}
), "NB11 manifest incorrectly declares a final production strategy."

manifest_artifacts = chunking_freeze_manifest.get("artifacts")

assert isinstance(
    manifest_artifacts,
    dict,
), "NB11 manifest artifacts field is not a JSON object."

artifact_validation_results = {}
artifact_validation_failures = []

for artifact_name, descriptor in sorted(manifest_artifacts.items()):
    relative_path = descriptor.get("path")
    expected_sha256 = descriptor.get("sha256")
    expected_bytes = descriptor.get("bytes")

    assert isinstance(relative_path, str), (
        f"Artifact path is invalid: {artifact_name}"
    )

    artifact_path = project_root / relative_path

    result = {
        "path": relative_path,
        "exists": artifact_path.is_file(),
        "size_matches": False,
        "sha256_matches": False,
    }

    if artifact_path.is_file():
        actual_bytes = artifact_path.stat().st_size

        artifact_digest = hashlib.sha256()

        with artifact_path.open("rb") as handle:
            for chunk in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                artifact_digest.update(chunk)

        actual_sha256 = artifact_digest.hexdigest()

        result["actual_bytes"] = actual_bytes
        result["actual_sha256"] = actual_sha256
        result["size_matches"] = (
            actual_bytes == expected_bytes
        )
        result["sha256_matches"] = (
            actual_sha256 == expected_sha256
        )

    artifact_validation_results[artifact_name] = result

    if not (
        result["exists"]
        and result["size_matches"]
        and result["sha256_matches"]
    ):
        artifact_validation_failures.append(
            artifact_name
        )

chunking_candidates = {
    artifact_name: {
        "path": descriptor["path"],
        "records": descriptor["records"],
        "role": descriptor["role"],
        "sha256": descriptor["sha256"],
    }
    for artifact_name, descriptor in manifest_artifacts.items()
    if descriptor.get("artifact_type") == "chunk_candidate"
}

print("=== NB11 FROZEN ARTIFACT VALIDATION ===")
print("Manifest artifacts:", len(manifest_artifacts))
print("Chunking candidates:", len(chunking_candidates))

print("\nCandidate registry:")
for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    print(
        f"- {candidate_name}: "
        f"role={candidate['role']}, "
        f"records={candidate['records']}, "
        f"path={candidate['path']}"
    )

print("\nFailed artifact validations:", len(artifact_validation_failures))

if artifact_validation_failures:
    print(artifact_validation_failures)

assert not artifact_validation_failures, (
    "NB11 frozen artifact validation failed: "
    f"{artifact_validation_failures}"
)

print("\nNB11 FROZEN ARTIFACT VALIDATION: PASS")

=== NB11 FROZEN ARTIFACT VALIDATION ===
Manifest artifacts: 15
Chunking candidates: 5

Candidate registry:
- baseline_phase1_char_safe: role=retrieval, records=3488, path=data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl
- parent_child_children: role=retrieval_child, records=2721, path=data/phase2/chunks/nb11_benchmark_v1/parent_child_children_v1.jsonl
- parent_child_parents: role=context_parent, records=1447, path=data/phase2/chunks/nb11_benchmark_v1/parent_child_parents_v1.jsonl
- structure_aware_medium: role=retrieval, records=1799, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_medium_v1.jsonl
- structure_aware_small: role=retrieval, records=2526, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_small_v1.jsonl

Failed artifact validations: 0

NB11 FROZEN ARTIFACT VALIDATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Discovery ساختار واقعی Candidateهای Chunking</h2>

<p>
Artifactهای Freeze‌شده از نظر فایل، Size و Hash معتبر هستند.
اکنون باید ساختار واقعی Recordهای JSONL هر Candidate بررسی شود.
</p>

<p>
هدف این Step کشف Fieldهای واقعی Chunk، Metadata، Locator، Lineage و شناسه‌ها است.
نام هیچ Fieldی از قبل فرض نمی‌شود.
</p>

<p>
این مرحله فقط اولین Record هر Candidate را می‌خواند و هیچ فایل یا Artifactی را تغییر نمی‌دهد.
</p>

<h3>خروجی مورد انتظار</h3>

<ul>
<li>Fieldهای واقعی هر Candidate</li>
<li>نوع دادهٔ هر Field در نمونهٔ واقعی</li>
<li>مقایسهٔ ابتدایی Contract میان Candidateها</li>
</ul>

</div>

In [16]:
# Code Cell 07

chunking_candidate_samples = {}

print("=== CHUNKING CANDIDATE RECORD DISCOVERY ===")

for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    first_record = None

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                first_record = json.loads(line)
                break

    assert isinstance(
        first_record,
        dict,
    ), (
        f"First non-empty record is not a JSON object: "
        f"{candidate_name}"
    )

    field_types = {
        field_name: type(value).__name__
        for field_name, value in sorted(
            first_record.items()
        )
    }

    chunking_candidate_samples[candidate_name] = {
        "fields": sorted(first_record.keys()),
        "field_types": field_types,
    }

    print(f"\n{candidate_name}")
    print("Role:", candidate["role"])
    print("Fields:")

    for field_name, field_type in field_types.items():
        print(f"  - {field_name}: {field_type}")

print("\nCHUNKING CANDIDATE RECORD DISCOVERY: PASS")

=== CHUNKING CANDIDATE RECORD DISCOVERY ===

baseline_phase1_char_safe
Role: retrieval
Fields:
  - attribution_context: dict
  - boundary_metadata: dict
  - char_count: int
  - chunk_contract_version: str
  - chunk_count: int
  - chunk_id: str
  - chunk_index: int
  - content: str
  - content_origin: list
  - lineage: dict
  - locator_end: dict
  - locator_start: dict
  - member_document_ids: list
  - member_locators: list
  - parent_chunk_id: NoneType
  - rights: dict
  - role: str
  - source_id: str
  - source_type: str
  - strategy_id: str
  - strategy_version: str
  - token_count: int

parent_child_children
Role: retrieval_child
Fields:
  - attribution_context: dict
  - boundary_metadata: dict
  - char_count: int
  - chunk_contract_version: str
  - chunk_count: int
  - chunk_id: str
  - chunk_index: int
  - content: str
  - content_origin: list
  - lineage: dict
  - locator_end: dict
  - locator_start: dict
  - member_document_ids: list
  - member_locators: list
  - parent_chunk_id

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی کامل Recordهای Candidateهای Chunking</h2>

<p>
Discovery ساختار نشان داد تمام Candidateها Contract یکسانی دارند و Fieldهای اصلی Chunk
در همه وجود دارند.
</p>

<p>
اکنون هر فایل Candidate به‌طور کامل خوانده می‌شود تا مشخص شود تمام Recordها JSON معتبر،
دارای <code>chunk_id</code> یکتا، محتوای غیرخالی و Fieldهای سازگار با Contract کشف‌شده هستند.
</p>

<p>
در پایان، فقط Candidateهایی که نقش Retrieval دارند در متغیر
<code>retrieval_chunking_candidates</code>
برای Benchmark Embedding ثبت می‌شوند.
</p>

<p>
این مرحله هیچ Chunk یا Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [17]:
# Code Cell 08

reference_fields = set(
    next(
        iter(
            chunking_candidate_samples.values()
        )
    )["fields"]
)

candidate_record_validation = {}
candidate_record_validation_failures = []

for candidate_name, candidate in sorted(
    chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    record_count = 0
    json_failures = []
    field_contract_failures = []
    empty_chunk_ids = []
    duplicate_chunk_ids = []
    empty_content_chunk_ids = []
    chunk_ids = set()

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                json_failures.append(
                    {
                        "line": line_number,
                        "message": str(error),
                    }
                )
                continue

            record_count += 1

            if not isinstance(record, dict):
                field_contract_failures.append(
                    {
                        "line": line_number,
                        "reason": "record_is_not_object",
                    }
                )
                continue

            if set(record.keys()) != reference_fields:
                field_contract_failures.append(
                    {
                        "line": line_number,
                        "reason": "field_contract_mismatch",
                    }
                )

            chunk_id = record.get("chunk_id")

            if not isinstance(chunk_id, str) or not chunk_id.strip():
                empty_chunk_ids.append(line_number)
            elif chunk_id in chunk_ids:
                duplicate_chunk_ids.append(chunk_id)
            else:
                chunk_ids.add(chunk_id)

            content = record.get("content")

            if not isinstance(content, str) or not content.strip():
                empty_content_chunk_ids.append(chunk_id)

    result = {
        "records": record_count,
        "expected_records": candidate["records"],
        "json_failures": len(json_failures),
        "field_contract_failures": len(field_contract_failures),
        "empty_chunk_ids": len(empty_chunk_ids),
        "duplicate_chunk_ids": len(duplicate_chunk_ids),
        "empty_content_chunk_ids": len(
            empty_content_chunk_ids
        ),
    }

    candidate_record_validation[candidate_name] = result

    candidate_passed = (
        record_count == candidate["records"]
        and not json_failures
        and not field_contract_failures
        and not empty_chunk_ids
        and not duplicate_chunk_ids
        and not empty_content_chunk_ids
    )

    if not candidate_passed:
        candidate_record_validation_failures.append(
            candidate_name
        )

retrieval_chunking_candidates = {
    candidate_name: candidate
    for candidate_name, candidate in chunking_candidates.items()
    if candidate["role"] in {
        "retrieval",
        "retrieval_child",
    }
}

print("=== FULL CHUNKING CANDIDATE RECORD VALIDATION ===")

for candidate_name, result in sorted(
    candidate_record_validation.items()
):
    print(f"\n{candidate_name}")
    for field_name, value in result.items():
        print(f"  {field_name}: {value}")

print(
    "\nRetrieval candidates:",
    sorted(retrieval_chunking_candidates),
)

print(
    "\nFailed candidates:",
    candidate_record_validation_failures,
)

assert not candidate_record_validation_failures, (
    "Chunking candidate record validation failed: "
    f"{candidate_record_validation_failures}"
)

print(
    "\nFULL CHUNKING CANDIDATE RECORD VALIDATION: PASS"
)

=== FULL CHUNKING CANDIDATE RECORD VALIDATION ===

baseline_phase1_char_safe
  records: 3488
  expected_records: 3488
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

parent_child_children
  records: 2721
  expected_records: 2721
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

parent_child_parents
  records: 1447
  expected_records: 1447
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

structure_aware_medium
  records: 1799
  expected_records: 1799
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

structure_aware_small
  records: 2526
  expected_records: 2526
  json_failures: 0
  field_contract_failures: 0
  empty_chunk_ids: 0
  duplicate_chunk_ids: 0
  empty_content_chunk_ids: 0

Retrieval 

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Discovery ساختار Gold Evidence در Golden Dataset فعال</h2>

<p>
Candidateهای Chunking آمادهٔ Benchmark هستند، اما پیش از اجرای هر مدل Embedding باید مشخص شود
هر سؤال Golden Dataset چگونه به Source Evidence، Lineage، Seed و Locator متصل شده است.
</p>

<p>
این Step ساختار واقعی Fieldهای
<code>source_evidence</code>،
<code>source_lineage</code>
و
<code>source_seed_id</code>
را در کل Golden Dataset بررسی می‌کند.
</p>

<p>
هدف این است که در مراحل بعد، معیارهای Retrieval دقیقاً براساس Gold Evidence واقعی ساخته شوند
و هیچ نگاشت Question به Chunk از روی حدس انجام نشود.
</p>

<p>
این مرحله فقط Discovery است و هیچ Dataset، Chunk، Model یا Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [18]:
# Code Cell 09

def describe_value_structure(value):
    if isinstance(value, dict):
        return {
            "type": "dict",
            "keys": sorted(value.keys()),
        }

    if isinstance(value, list):
        item_types = sorted(
            {
                type(item).__name__
                for item in value
            }
        )

        return {
            "type": "list",
            "length": len(value),
            "item_types": item_types,
        }

    return {
        "type": type(value).__name__,
    }


source_evidence_structures = Counter()
source_lineage_structures = Counter()
source_seed_id_types = Counter()

for record in golden_dataset:
    source_evidence_structure = json.dumps(
        describe_value_structure(
            record.get("source_evidence")
        ),
        ensure_ascii=False,
        sort_keys=True,
    )

    source_lineage_structure = json.dumps(
        describe_value_structure(
            record.get("source_lineage")
        ),
        ensure_ascii=False,
        sort_keys=True,
    )

    source_evidence_structures[
        source_evidence_structure
    ] += 1

    source_lineage_structures[
        source_lineage_structure
    ] += 1

    source_seed_id_types[
        type(
            record.get("source_seed_id")
        ).__name__
    ] += 1

print("=== GOLD EVIDENCE STRUCTURE DISCOVERY ===")

print("\nsource_seed_id value types:")
for structure, count in sorted(
    source_seed_id_types.items()
):
    print(f"- {structure}: {count}")

print("\nsource_evidence structures:")
for structure, count in sorted(
    source_evidence_structures.items()
):
    print(f"- {structure}: {count}")

print("\nsource_lineage structures:")
for structure, count in sorted(
    source_lineage_structures.items()
):
    print(f"- {structure}: {count}")

print("\nFirst three Golden Dataset evidence samples:")

for index, record in enumerate(
    golden_dataset[:3],
    start=1,
):
    print(f"\nSample {index}")
    print("candidate_id:", record["candidate_id"])
    print("source_seed_id:", record["source_seed_id"])
    print(
        "source_evidence:",
        json.dumps(
            record["source_evidence"],
            ensure_ascii=False,
            indent=2,
        ),
    )
    print(
        "source_lineage:",
        json.dumps(
            record["source_lineage"],
            ensure_ascii=False,
            indent=2,
        ),
    )

print("\nGOLD EVIDENCE STRUCTURE DISCOVERY: PASS")

=== GOLD EVIDENCE STRUCTURE DISCOVERY ===

source_seed_id value types:
- int: 573

source_evidence structures:
- {"type": "str"}: 573

source_lineage structures:
- {"type": "str"}: 573

First three Golden Dataset evidence samples:

Sample 1
candidate_id: 94fe2e71-b942-400a-8538-9b1988959d52
source_seed_id: 21
source_evidence: "بیمه‌گر کسی است که تعهد می‌کند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه‌گذار فردی است که خود یا دارایی‌اش را بیمه می‌کند. مبلغی که بیمه‌گذار برای دریافت این حمایت می‌پردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می‌کند که بیمه‌گر متعهد به جبران آن است و فرانشیز یا خودپرداخت بخشی از خسارت است که خود بیمه‌گذار باید آن را پرداخت کند."
source_lineage: "pilot_21"

Sample 2
candidate_id: f47efecc-ad01-44d2-8f0b-3c5d2ab6d1f7
source_seed_id: 21
source_evidence: "بیمه‌گر کسی است که تعهد می‌کند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه‌گذار فردی است که خود یا دارایی‌اش را بیمه می‌کند. مبلغی که بیمه‌گذار برای دریافت این حمایت می‌پردازد حق بیمه 

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>ساخت Registry واقعی Gold Evidence و Seedها</h2>

<p>
Discovery نشان داد Gold Evidence به‌صورت متن و Source Seed در Golden Dataset ذخیره شده است.
اکنون پیش از نگاشت Evidence به Chunkها، باید رابطهٔ واقعی میان Candidateها، Seedها و متن Evidence
به‌طور کامل اندازه‌گیری شود.
</p>

<p>
این Step مشخص می‌کند هر Seed چند Candidate Golden دارد، هر متن Evidence چند بار استفاده شده است
و آیا Evidence خالی یا ناسازگار وجود دارد یا نه.
</p>

<p>
هیچ Locator، Document ID یا Gold Chunk جدیدی در این مرحله ساخته نمی‌شود.
</p>

</div>

In [19]:
# Code Cell 10

gold_evidence_registry = {}
seed_candidate_counts = Counter()
seed_evidence_hashes = defaultdict(set)
lineage_counts = Counter()
empty_evidence_candidate_ids = []
duplicate_question_count = 0

normalized_questions = Counter()

for record in golden_dataset:
    candidate_id = record["candidate_id"]
    source_seed_id = record["source_seed_id"]
    source_evidence = record["source_evidence"]
    source_lineage = record["source_lineage"]
    question = record["question"]

    seed_candidate_counts[source_seed_id] += 1
    lineage_counts[source_lineage] += 1

    if not isinstance(source_evidence, str) or not source_evidence.strip():
        empty_evidence_candidate_ids.append(candidate_id)
        continue

    evidence_hash = hashlib.sha256(
        source_evidence.encode("utf-8")
    ).hexdigest()

    seed_evidence_hashes[source_seed_id].add(
        evidence_hash
    )

    if evidence_hash not in gold_evidence_registry:
        gold_evidence_registry[evidence_hash] = {
            "source_evidence": source_evidence,
            "source_seed_ids": set(),
            "source_lineages": set(),
            "candidate_ids": [],
        }

    gold_evidence_registry[evidence_hash][
        "source_seed_ids"
    ].add(source_seed_id)

    gold_evidence_registry[evidence_hash][
        "source_lineages"
    ].add(source_lineage)

    gold_evidence_registry[evidence_hash][
        "candidate_ids"
    ].append(candidate_id)

    normalized_question = " ".join(
        question.split()
    )

    normalized_questions[
        normalized_question
    ] += 1

duplicate_question_count = sum(
    count - 1
    for count in normalized_questions.values()
    if count > 1
)

seed_multiple_evidence = {
    seed_id: evidence_hashes
    for seed_id, evidence_hashes in seed_evidence_hashes.items()
    if len(evidence_hashes) > 1
}

print("=== GOLD EVIDENCE AND SEED REGISTRY ===")
print("Golden candidates:", len(golden_dataset))
print("Unique source seeds:", len(seed_candidate_counts))
print("Unique evidence texts:", len(gold_evidence_registry))
print("Empty evidence records:", len(empty_evidence_candidate_ids))
print("Duplicate normalized questions:", duplicate_question_count)
print("Seeds with multiple evidence texts:", len(seed_multiple_evidence))

print("\nCandidate count by source seed:")
for seed_id, count in sorted(seed_candidate_counts.items()):
    print(f"- seed {seed_id}: {count}")

print("\nCandidate count by source lineage:")
for lineage, count in sorted(lineage_counts.items()):
    print(f"- {lineage}: {count}")

assert not empty_evidence_candidate_ids, (
    "Golden Dataset contains empty source evidence."
)

assert not duplicate_question_count, (
    "Golden Dataset contains duplicate normalized questions."
)

print("\nGOLD EVIDENCE AND SEED REGISTRY: PASS")

=== GOLD EVIDENCE AND SEED REGISTRY ===
Golden candidates: 573
Unique source seeds: 28
Unique evidence texts: 28
Empty evidence records: 0
Duplicate normalized questions: 0
Seeds with multiple evidence texts: 0

Candidate count by source seed:
- seed 1: 21
- seed 2: 21
- seed 3: 21
- seed 4: 21
- seed 5: 21
- seed 6: 21
- seed 7: 21
- seed 8: 21
- seed 9: 21
- seed 10: 21
- seed 11: 21
- seed 12: 21
- seed 13: 20
- seed 14: 20
- seed 15: 20
- seed 16: 20
- seed 17: 20
- seed 18: 20
- seed 19: 20
- seed 20: 20
- seed 21: 21
- seed 22: 20
- seed 23: 20
- seed 24: 20
- seed 25: 20
- seed 26: 20
- seed 27: 20
- seed 28: 20

Candidate count by source lineage:
- pilot_1: 21
- pilot_10: 21
- pilot_11: 21
- pilot_12: 21
- pilot_13: 20
- pilot_14: 20
- pilot_15: 20
- pilot_16: 20
- pilot_17: 20
- pilot_18: 20
- pilot_19: 20
- pilot_2: 21
- pilot_20: 20
- pilot_21: 21
- pilot_22: 20
- pilot_23: 20
- pilot_24: 20
- pilot_25: 20
- pilot_26: 20
- pilot_27: 20
- pilot_28: 20
- pilot_3: 21
- pilot_4:

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>نگاشت اولیهٔ Gold Evidence به Candidateهای Chunking با تطابق متن خام</h2>

<p>
Golden Dataset دارای ۲۸ متن Evidence یکتا است. اکنون این متن‌ها مستقیماً با محتوای Chunkهای
Candidateهای Retrieval مقایسه می‌شوند.
</p>

<p>
این مرحله فقط تطابق دقیق متن خام را می‌سنجد. اگر بخشی از Evidence به‌دلیل تفاوت فاصله،
نیم‌فاصله، ساختار Chunk یا Boundary پیدا نشود، هنوز هیچ نگاشت تقریبی یا اصلاحی انجام نمی‌دهیم.
</p>

<p>
هدف، اندازه‌گیری پایه‌ای Coverage Evidence برای هر Candidate است.
</p>

</div>

In [20]:
# Code Cell 11

exact_evidence_matches = {
    evidence_hash: {
        candidate_name: []
        for candidate_name
        in retrieval_chunking_candidates
    }
    for evidence_hash in gold_evidence_registry
}

for candidate_name, candidate in sorted(
    retrieval_chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk_record = json.loads(line)
            chunk_content = chunk_record["content"]

            for evidence_hash, evidence_info in (
                gold_evidence_registry.items()
            ):
                source_evidence = evidence_info[
                    "source_evidence"
                ]

                if source_evidence in chunk_content:
                    exact_evidence_matches[
                        evidence_hash
                    ][
                        candidate_name
                    ].append(
                        chunk_record["chunk_id"]
                    )

exact_coverage = {}

for candidate_name in retrieval_chunking_candidates:
    matched_evidence_count = sum(
        1
        for evidence_hash in gold_evidence_registry
        if exact_evidence_matches[
            evidence_hash
        ][
            candidate_name
        ]
    )

    exact_coverage[candidate_name] = {
        "matched_evidence": matched_evidence_count,
        "total_evidence": len(gold_evidence_registry),
        "coverage_ratio": (
            matched_evidence_count
            / len(gold_evidence_registry)
        ),
    }

print("=== RAW EXACT GOLD EVIDENCE MATCHING ===")

for candidate_name, result in sorted(
    exact_coverage.items()
):
    print(f"\n{candidate_name}")
    print(
        "Matched evidence:",
        f"{result['matched_evidence']} "
        f"/ {result['total_evidence']}"
    )
    print(
        "Coverage ratio:",
        f"{result['coverage_ratio']:.4f}"
    )

print("\nUnmatched evidence by candidate:")

for candidate_name in sorted(
    retrieval_chunking_candidates
):
    unmatched_seed_ids = sorted(
        {
            next(
                iter(
                    gold_evidence_registry[
                        evidence_hash
                    ][
                        "source_seed_ids"
                    ]
                )
            )
            for evidence_hash in gold_evidence_registry
            if not exact_evidence_matches[
                evidence_hash
            ][
                candidate_name
            ]
        }
    )

    print(
        f"- {candidate_name}: "
        f"{unmatched_seed_ids}"
    )

print("\nRAW EXACT GOLD EVIDENCE MATCHING: PASS")

=== RAW EXACT GOLD EVIDENCE MATCHING ===

baseline_phase1_char_safe
Matched evidence: 0 / 28
Coverage ratio: 0.0000

parent_child_children
Matched evidence: 0 / 28
Coverage ratio: 0.0000

structure_aware_medium
Matched evidence: 0 / 28
Coverage ratio: 0.0000

structure_aware_small
Matched evidence: 0 / 28
Coverage ratio: 0.0000

Unmatched evidence by candidate:
- baseline_phase1_char_safe: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
- parent_child_children: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
- structure_aware_medium: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
- structure_aware_small: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]

RAW EXACT GOLD EVIDENCE MATCHING: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>بررسی تطابق سطحی Gold Evidence پس از Canonical Comparison</h2>

<p>
تطابق خام Gold Evidence با Chunkها صفر بود. این نتیجه به‌تنهایی ثابت نمی‌کند که Evidence در Corpus وجود ندارد،
زیرا تفاوت‌های Unicode، فاصله، نیم‌فاصله، نویسه‌های Formatting و شکل عربی/فارسی حروف می‌توانند
تطابق خام را از بین ببرند.
</p>

<p>
در این Step یک Normalizer فقط برای <strong>مقایسه</strong> ساخته می‌شود.
این تابع هیچ Dataset یا Chunkی را تغییر نمی‌دهد و جایگزین Policy نرمال‌سازی رسمی NB08 نیست.
</p>

<h3>قواعد Comparison</h3>

<ul>
<li>Unicode NFC</li>
<li>حذف Soft Hyphen و نویسه‌های Formatting</li>
<li>یکسان‌سازی فاصله‌ها</li>
<li>یکسان‌سازی صرفاً مقایسه‌ای ی/ک عربی و فارسی</li>
<li>حفظ متن اصلی Dataset و Chunk بدون تغییر</li>
</ul>

<p>
اگر بعد از این مرحله نیز Coverage صفر یا ناکافی باشد، باید Root Cause در Lineage و منبع Gold Evidence بررسی شود.
</p>

</div>

In [21]:
# Code Cell 12

import unicodedata


def canonical_comparison_text(text):
    normalized = unicodedata.normalize(
        "NFC",
        text,
    )

    normalized = normalized.replace(
        "\u00ad",
        "",
    )

    normalized = normalized.replace(
        "\u064a",
        "\u06cc",
    )

    normalized = normalized.replace(
        "\u0643",
        "\u06a9",
    )

    normalized = "".join(
        " "
        if unicodedata.category(character) == "Cf"
        else character
        for character in normalized
    )

    normalized = " ".join(
        normalized.split()
    )

    return normalized


comparison_evidence_matches = {
    evidence_hash: {
        candidate_name: []
        for candidate_name
        in retrieval_chunking_candidates
    }
    for evidence_hash in gold_evidence_registry
}

canonical_evidence_texts = {
    evidence_hash: canonical_comparison_text(
        evidence_info["source_evidence"]
    )
    for evidence_hash, evidence_info in (
        gold_evidence_registry.items()
    )
}

for candidate_name, candidate in sorted(
    retrieval_chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk_record = json.loads(line)

            canonical_chunk_content = (
                canonical_comparison_text(
                    chunk_record["content"]
                )
            )

            for evidence_hash, canonical_evidence in (
                canonical_evidence_texts.items()
            ):
                if canonical_evidence in canonical_chunk_content:
                    comparison_evidence_matches[
                        evidence_hash
                    ][
                        candidate_name
                    ].append(
                        chunk_record["chunk_id"]
                    )

comparison_coverage = {}

for candidate_name in retrieval_chunking_candidates:
    matched_evidence_count = sum(
        1
        for evidence_hash in gold_evidence_registry
        if comparison_evidence_matches[
            evidence_hash
        ][
            candidate_name
        ]
    )

    comparison_coverage[candidate_name] = {
        "matched_evidence": matched_evidence_count,
        "total_evidence": len(gold_evidence_registry),
        "coverage_ratio": (
            matched_evidence_count
            / len(gold_evidence_registry)
        ),
    }

print("=== CANONICAL GOLD EVIDENCE MATCHING ===")

for candidate_name, result in sorted(
    comparison_coverage.items()
):
    print(f"\n{candidate_name}")
    print(
        "Matched evidence:",
        f"{result['matched_evidence']} "
        f"/ {result['total_evidence']}"
    )
    print(
        "Coverage ratio:",
        f"{result['coverage_ratio']:.4f}"
    )

print(
    "\nCANONICAL GOLD EVIDENCE MATCHING: PASS"
)

=== CANONICAL GOLD EVIDENCE MATCHING ===

baseline_phase1_char_safe
Matched evidence: 0 / 28
Coverage ratio: 0.0000

parent_child_children
Matched evidence: 0 / 28
Coverage ratio: 0.0000

structure_aware_medium
Matched evidence: 0 / 28
Coverage ratio: 0.0000

structure_aware_small
Matched evidence: 0 / 28
Coverage ratio: 0.0000

CANONICAL GOLD EVIDENCE MATCHING: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>تحلیل واژگانی Root Cause برای نگاشت Gold Evidence به Chunkها</h2>

<p>
تطابق خام و Canonical هر دو صفر بودند. بنابراین Gold Evidence و Chunkها فقط اختلاف
فرمت سطحی ندارند و باید ارتباط آن‌ها به‌صورت واژگانی اندازه‌گیری شود.
</p>

<p>
در این Step برای هر متن Evidence، بیشترین هم‌پوشانی واژگانی با Chunkهای هر Candidate محاسبه می‌شود.
این یک Retrieval Benchmark نیست و هیچ مدل Embeddingی استفاده نمی‌شود؛ فقط یک تحلیل تشخیصی برای
کشف امکان یا عدم امکان ساخت Gold Mapping است.
</p>

<p>
این مرحله هیچ Gold Chunk را به‌طور قطعی انتخاب نمی‌کند و هیچ Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [22]:
# Code Cell 13

def comparison_token_set(text):
    canonical_text = canonical_comparison_text(text)

    return {
        token
        for token in re.findall(
            r"[\w\u0600-\u06ff]+",
            canonical_text,
        )
        if len(token) >= 2
    }


evidence_token_sets = {
    evidence_hash: comparison_token_set(
        evidence_info["source_evidence"]
    )
    for evidence_hash, evidence_info in (
        gold_evidence_registry.items()
    )
}

lexical_evidence_matches = {
    evidence_hash: {
        candidate_name: {
            "chunk_id": None,
            "evidence_token_coverage": 0.0,
            "jaccard_similarity": 0.0,
            "shared_token_count": 0,
        }
        for candidate_name
        in retrieval_chunking_candidates
    }
    for evidence_hash in gold_evidence_registry
}

for candidate_name, candidate in sorted(
    retrieval_chunking_candidates.items()
):
    candidate_path = project_root / candidate["path"]

    with candidate_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk_record = json.loads(line)
            chunk_tokens = comparison_token_set(
                chunk_record["content"]
            )

            if not chunk_tokens:
                continue

            for evidence_hash, evidence_tokens in (
                evidence_token_sets.items()
            ):
                if not evidence_tokens:
                    continue

                shared_tokens = (
                    evidence_tokens
                    & chunk_tokens
                )

                shared_token_count = len(shared_tokens)

                evidence_token_coverage = (
                    shared_token_count
                    / len(evidence_tokens)
                )

                jaccard_similarity = (
                    shared_token_count
                    / len(
                        evidence_tokens
                        | chunk_tokens
                    )
                )

                current_best = lexical_evidence_matches[
                    evidence_hash
                ][
                    candidate_name
                ]

                if (
                    evidence_token_coverage
                    > current_best[
                        "evidence_token_coverage"
                    ]
                ):
                    lexical_evidence_matches[
                        evidence_hash
                    ][
                        candidate_name
                    ] = {
                        "chunk_id": chunk_record["chunk_id"],
                        "evidence_token_coverage": (
                            evidence_token_coverage
                        ),
                        "jaccard_similarity": (
                            jaccard_similarity
                        ),
                        "shared_token_count": (
                            shared_token_count
                        ),
                    }

print("=== GOLD EVIDENCE LEXICAL ROOT-CAUSE ANALYSIS ===")

for candidate_name in sorted(
    retrieval_chunking_candidates
):
    coverage_values = [
        lexical_evidence_matches[
            evidence_hash
        ][
            candidate_name
        ][
            "evidence_token_coverage"
        ]
        for evidence_hash in gold_evidence_registry
    ]

    print(f"\n{candidate_name}")
    print(
        "Minimum best coverage:",
        f"{min(coverage_values):.4f}",
    )
    print(
        "Average best coverage:",
        f"{sum(coverage_values) / len(coverage_values):.4f}",
    )
    print(
        "Maximum best coverage:",
        f"{max(coverage_values):.4f}",
    )

print("\nBest overall match for each source seed:")

for evidence_hash, evidence_info in sorted(
    gold_evidence_registry.items(),
    key=lambda item: next(
        iter(item[1]["source_seed_ids"])
    ),
):
    best_candidate_name, best_match = max(
        lexical_evidence_matches[
            evidence_hash
        ].items(),
        key=lambda item: item[1][
            "evidence_token_coverage"
        ],
    )

    source_seed_id = next(
        iter(
            evidence_info["source_seed_ids"]
        )
    )

    print(
        f"- seed {source_seed_id}: "
        f"candidate={best_candidate_name}, "
        f"coverage={best_match['evidence_token_coverage']:.4f}, "
        f"shared_tokens={best_match['shared_token_count']}, "
        f"chunk_id={best_match['chunk_id']}"
    )

print(
    "\nGOLD EVIDENCE LEXICAL ROOT-CAUSE ANALYSIS: PASS"
)

=== GOLD EVIDENCE LEXICAL ROOT-CAUSE ANALYSIS ===

baseline_phase1_char_safe
Minimum best coverage: 0.3077
Average best coverage: 0.5953
Maximum best coverage: 1.0000

parent_child_children
Minimum best coverage: 0.3846
Average best coverage: 0.6210
Maximum best coverage: 1.0000

structure_aware_medium
Minimum best coverage: 0.4103
Average best coverage: 0.6632
Maximum best coverage: 1.0000

structure_aware_small
Minimum best coverage: 0.3846
Average best coverage: 0.6198
Maximum best coverage: 1.0000

Best overall match for each source seed:
- seed 1: candidate=baseline_phase1_char_safe, coverage=0.5517, shared_tokens=32, chunk_id=chunk:baseline_phase1_char_safe:retrieval:6788b17327885ac4bd3cb0ae
- seed 2: candidate=baseline_phase1_char_safe, coverage=0.5397, shared_tokens=34, chunk_id=chunk:baseline_phase1_char_safe:retrieval:ac0640058e3ce4dc1987f8da
- seed 3: candidate=structure_aware_medium, coverage=0.6522, shared_tokens=30, chunk_id=chunk:structure_aware_medium:retrieval:3a6094cb

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>کشف Artifactهای معتبر برای Gold Provenance Mapping</h2>

<p>
تحلیل واژگانی نشان داد Evidenceهای Golden Dataset با Chunkها مرتبط‌اند، اما متن آن‌ها دقیقاً یکسان نیست.
بنابراین انتخاب خودکار بهترین Chunk بر اساس شباهت واژگانی، به‌تنهایی Ground Truth معتبر ایجاد نمی‌کند.
</p>

<p>
در این Step، Artifactهای موجود پروژه که ممکن است ارتباط معتبر میان ۲۸ Seed،
Source Evidence، Locator و Pilot Review را نگه داشته باشند، از روی Disk کشف می‌شوند.
</p>

<p>
هدف، بازیابی Mapping واقعی از Artifactهای قبلی است؛ نه ساختن Mapping جدید بر اساس حدس.
</p>

<p>
این مرحله فقط Discovery است و هیچ Golden Label، Chunk، Dataset یا Artifactی را تغییر نمی‌دهد.
</p>

</div>

In [23]:
# Code Cell 14

provenance_keywords = (
    "author_review",
    "human_review",
    "pilot",
    "source_packet",
    "golden",
)

provenance_artifacts = []

for path in project_root.rglob("*"):
    if not path.is_file():
        continue

    normalized_name = path.name.lower()

    if any(
        keyword in normalized_name
        for keyword in provenance_keywords
    ):
        provenance_artifacts.append(path)

print("=== GOLD PROVENANCE ARTIFACT DISCOVERY ===")
print("Artifacts found:", len(provenance_artifacts))

for path in provenance_artifacts:
    artifact_hash = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    print("\nPath:", path.relative_to(project_root))
    print("Size bytes:", path.stat().st_size)
    print("SHA-256:", artifact_hash)

print("\nGOLD PROVENANCE ARTIFACT DISCOVERY: PASS")

=== GOLD PROVENANCE ARTIFACT DISCOVERY ===
Artifacts found: 24

Path: data/phase2/human_review/human_review_registry.json
Size bytes: 3675788
SHA-256: a851f1a20641f8627f12121bef6497e0fcfb72e6c0bc4e47096f4021e6604b04

Path: data/phase2/human_review/human_review_registry.before_pilots23_28_source_review_20260909T134140Z.json
Size bytes: 3596157
SHA-256: 52f576f7c930e3ddb985436e9c81f839832e4411cbd290982ec6cfda76ed48af

Path: data/phase2/human_review/human_review_registry.before_cell109_history_fidelity_repair_20260909T131657Z.json
Size bytes: 3595013
SHA-256: 7b61d69cf52119f2ef402ede382692f62fcf4cd967eec68732ae04b1b2c7e226

Path: data/phase2/human_review/human_review_registry.before_phase1_author_feedback_batch_20260909T130212Z.json
Size bytes: 3526169
SHA-256: 81d634841743021849fcc7f364d38c5b2e44849c969378c501325d4795208df0

Path: data/phase2/human_review/human_review_registry.before_author_review_28_promotion_20260909T130800Z.json
Size bytes: 3559756
SHA-256: 0893956d0553b4608ef0340c182

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Discovery ساختار Registry بازبینی انسانی و Source Packet</h2>

<p>
Artifactهای معتبر Provenance پیدا شدند. اکنون باید ساختار واقعی Registry اصلی Human Review
و Source Packet مربوط به Pilotهای ۲۳ تا ۲۸ بررسی شود.
</p>

<p>
هدف این Step پیدا کردن Fieldهای واقعیِ Case، Pilot، Seed، Target Evidence، Source، Locator
و تصمیم بازبینی است تا Gold Mapping از دادهٔ موجود بازیابی شود.
</p>

<p>
این مرحله فقط Read Only است و هیچ Registry، Source Packet، Golden Dataset یا Chunkی را تغییر نمی‌دهد.
</p>

</div>

In [24]:
# Code Cell 15

human_review_registry_path = next(
    (
        path
        for path in provenance_artifacts
        if path.name == "human_review_registry.json"
    ),
    None,
)

source_packet_path = next(
    (
        path
        for path in provenance_artifacts
        if path.name == "source_packet_pilots23_28_v1.json"
    ),
    None,
)

assert human_review_registry_path is not None, (
    "Active human review registry was not found."
)

assert source_packet_path is not None, (
    "Pilot 23-28 source packet was not found."
)

human_review_registry = json.loads(
    human_review_registry_path.read_text(
        encoding="utf-8"
    )
)

source_packet = json.loads(
    source_packet_path.read_text(
        encoding="utf-8"
    )
)

assert isinstance(
    human_review_registry,
    dict,
), "Human review registry is not a JSON object."

assert isinstance(
    source_packet,
    dict,
), "Source packet is not a JSON object."


def summarize_top_level_json_object(label, payload):
    print(f"\n{label}")
    print("Top-level fields:")

    for field_name, value in sorted(payload.items()):
        if isinstance(value, list):
            description = (
                f"list with {len(value)} items"
            )
        elif isinstance(value, dict):
            description = (
                f"object with {len(value)} fields"
            )
        else:
            description = type(value).__name__

        print(f"- {field_name}: {description}")


print("=== HUMAN REVIEW PROVENANCE CONTRACT DISCOVERY ===")
print(
    "Registry path:",
    human_review_registry_path.relative_to(
        project_root
    ),
)
print(
    "Source packet path:",
    source_packet_path.relative_to(
        project_root
    ),
)

summarize_top_level_json_object(
    "Human review registry",
    human_review_registry,
)

summarize_top_level_json_object(
    "Source packet",
    source_packet,
)

print(
    "\nHUMAN REVIEW PROVENANCE CONTRACT DISCOVERY: PASS"
)

=== HUMAN REVIEW PROVENANCE CONTRACT DISCOVERY ===
Registry path: data/phase2/human_review/human_review_registry.json
Source packet path: data/phase2/human_review/source_packet_pilots23_28_v1.json

Human review registry
Top-level fields:
- active_workspace: NoneType
- author_review: object with 7 fields
- completed_human_review_cases: list with 22 items
- continuation_state: object with 17 fields
- historical_completed_pilot: object with 2 fields
- human_review_summary: object with 9 fields
- phase: str
- pilot_human_review_status: str
- project: str
- purpose: str
- registry_version: str
- remaining_pilot_count: int
- retarget_candidate_discovery: object with 9 fields
- second_human_review: object with 2 fields
- workflow_policy: object with 6 fields

Source packet
Top-level fields:
- accepted_corpus_sha256: str
- author_facing_source_urls: bool
- author_filter_applied: bool
- authority_filter_applied: bool
- original_phase1_demo_provenance: str
- pilots: list with 6 items
- primary_d

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۴. کشف ساختار رکوردهای منشأ

در سل قبل مشخص شد که منشأ ۲۸ پایلوت در دو منبع نگهداری شده است:

- ۲۲ پایلوت تکمیل‌شده در رجیستری بازبینی انسانی؛
- ۶ پایلوت تکمیلی در بستهٔ منبع.

در این مرحله فقط قرارداد داده‌ای هر رکورد را بررسی می‌کنیم: شمارهٔ پایلوت، شناسه‌ها، متن منبع، و هر ارجاعی که بتواند بعداً به چانک‌ها وصل شود.

هنوز هیچ نگاشتی ساخته نمی‌شود و هیچ شباهت متنی به‌عنوان حقیقت در نظر گرفته نمی‌شود.

</div>

In [25]:
# Code Cell 16

completed_review_cases = human_review_registry["completed_human_review_cases"]
source_packet_pilots = source_packet["pilots"]

assert isinstance(completed_review_cases, list), (
    "completed_human_review_cases must be a list."
)
assert isinstance(source_packet_pilots, list), (
    "source_packet['pilots'] must be a list."
)

assert len(completed_review_cases) == 22, (
    f"Expected 22 completed human-review cases, found {len(completed_review_cases)}."
)
assert len(source_packet_pilots) == 6, (
    f"Expected 6 source-packet pilots, found {len(source_packet_pilots)}."
)

def describe_record_contract(label, records):
    assert records, f"{label} is empty."
    assert all(isinstance(record, dict) for record in records), (
        f"Every record in {label} must be a JSON object."
    )

    all_fields = sorted(
        {
            field_name
            for record in records
            for field_name in record.keys()
        }
    )

    print(f"\n{label}")
    print(f"Record count: {len(records)}")
    print("All observed fields:")

    for field_name in all_fields:
        value_types = sorted(
            {
                type(record[field_name]).__name__
                for record in records
                if field_name in record
            }
        )
        present_count = sum(
            field_name in record
            for record in records
        )
        print(
            f"- {field_name}: present in {present_count}/{len(records)}, "
            f"types={value_types}"
        )

    print("\nFirst record preview:")
    first_record = records[0]

    for field_name in sorted(first_record):
        value = first_record[field_name]

        if isinstance(value, dict):
            preview = f"object with fields {sorted(value.keys())}"
        elif isinstance(value, list):
            preview = f"list with {len(value)} items"
        else:
            preview = repr(value)
            if len(preview) > 300:
                preview = preview[:297] + "..."

        print(f"- {field_name}: {preview}")


print("=== HUMAN REVIEW RECORD-LEVEL CONTRACT DISCOVERY ===")

describe_record_contract(
    "Completed human-review cases",
    completed_review_cases,
)

describe_record_contract(
    "Source-packet pilots",
    source_packet_pilots,
)

print(
    "\nTotal provenance records:",
    len(completed_review_cases) + len(source_packet_pilots),
)

assert (
    len(completed_review_cases) + len(source_packet_pilots)
    == len(gold_evidence_registry)
), (
    "The number of provenance records does not match the number of "
    "Golden Dataset source seeds."
)

print("\nHUMAN REVIEW RECORD-LEVEL CONTRACT DISCOVERY: PASS")

=== HUMAN REVIEW RECORD-LEVEL CONTRACT DISCOVERY ===

Completed human-review cases
Record count: 22
All observed fields:
- author_final_review: present in 22/22, types=['dict']
- authority_expectation: present in 9/22, types=['str']
- authority_policy: present in 7/22, types=['dict']
- borrowing_evidence_search: present in 1/22, types=['dict']
- borrowing_hierarchy_audit: present in 1/22, types=['dict']
- citation_review: present in 22/22, types=['dict']
- combined_target_text: present in 18/22, types=['str']
- completion_summary: present in 14/22, types=['dict']
- corrected_target_interpretation: present in 1/22, types=['dict']
- deferred_question_candidate: present in 1/22, types=['dict']
- expected_answer_review: present in 22/22, types=['dict']
- final_question_draft_id: present in 4/22, types=['str']
- final_status: present in 22/22, types=['str']
- full_572_generation: present in 5/22, types=['str']
- golden_case: present in 5/22, types=['dict']
- golden_case_id: present in 22/22

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۵. کشف جزئیات شواهد و ادعاهای منبع

در این مرحله، ساختار داخلی شواهد منبع در پرونده‌های بازبینی انسانی و ادعاهای Source Packet بررسی می‌شود.

هدف، یافتن ارجاع‌های صریح و قابل‌ردیابی است؛ مانند شناسهٔ سند، شناسهٔ منبع، locator یا متن منبع.  
شباهت متنی به‌تنهایی برای ساخت نگاشت حقیقت استفاده نخواهد شد.

</div>

In [27]:
# Code Cell 17

def describe_object_records(label, records):
    assert records, f"{label} is empty."
    assert all(isinstance(record, dict) for record in records), (
        f"All entries in {label} must be JSON objects."
    )

    all_fields = sorted(
        {
            field_name
            for record in records
            for field_name in record
        }
    )

    print(f"\n{label}")
    print(f"Record count: {len(records)}")
    print("Observed fields:")

    for field_name in all_fields:
        present_count = sum(field_name in record for record in records)
        value_types = sorted(
            {
                type(record[field_name]).__name__
                for record in records
                if field_name in record
            }
        )
        print(
            f"- {field_name}: present in {present_count}/{len(records)}, "
            f"types={value_types}"
        )

    print("\nFirst record preview:")
    for field_name in sorted(records[0]):
        value = records[0][field_name]

        if isinstance(value, dict):
            preview = f"object with fields {sorted(value)}"
        elif isinstance(value, list):
            preview = f"list with {len(value)} items"
        else:
            preview = repr(value)
            if len(preview) > 350:
                preview = preview[:347] + "..."

        print(f"- {field_name}: {preview}")


review_source_evidence_records = [
    evidence_item
    for review_case in completed_review_cases
    for evidence_item in review_case.get("source_evidence", [])
]

review_citation_review_records = [
    review_case["citation_review"]
    for review_case in completed_review_cases
    if isinstance(review_case.get("citation_review"), dict)
]

review_target_evidence_records = [
    review_case["target_evidence"]
    for review_case in completed_review_cases
    if isinstance(review_case.get("target_evidence"), dict)
]

source_packet_claim_records = [
    claim
    for pilot in source_packet_pilots
    for claim in pilot.get("claims", [])
]

assert review_source_evidence_records, "No human-review source evidence was found."
assert review_citation_review_records, "No citation-review objects were found."
assert source_packet_claim_records, "No source-packet claims were found."

print("=== SOURCE EVIDENCE AND CLAIM CONTRACT DISCOVERY ===")

describe_object_records(
    "Human-review source_evidence entries",
    review_source_evidence_records,
)

describe_object_records(
    "Human-review citation_review objects",
    review_citation_review_records,
)

if review_target_evidence_records:
    describe_object_records(
        "Human-review target_evidence objects",
        review_target_evidence_records,
    )

describe_object_records(
    "Source-packet claim entries",
    source_packet_claim_records,
)

print("\nAccepted website document IDs by source-packet pilot:")
for pilot in sorted(
    source_packet_pilots,
    key=lambda item: item["reference_pilot_number"],
):
    print(
        f"- pilot {pilot['reference_pilot_number']}: "
        f"{pilot['accepted_website_document_ids']}"
    )

print("\nSOURCE EVIDENCE AND CLAIM CONTRACT DISCOVERY: PASS")

=== SOURCE EVIDENCE AND CLAIM CONTRACT DISCOVERY ===

Human-review source_evidence entries
Record count: 107
Observed fields:
- attribution: present in 28/107, types=['dict']
- char_count: present in 15/107, types=['int']
- char_end: present in 107/107, types=['int']
- char_start: present in 107/107, types=['int']
- document_id: present in 105/107, types=['str']
- human_attribution_overlay: present in 17/107, types=['dict']
- locator: present in 36/107, types=['dict']
- origin: present in 15/107, types=['str']
- part_index: present in 62/107, types=['int']
- role: present in 107/107, types=['str']
- source_id: present in 15/107, types=['str']
- source_identity: present in 14/107, types=['dict']
- source_locator: present in 14/107, types=['dict']
- source_path: present in 2/107, types=['str']
- source_type: present in 27/107, types=['str']
- speaker: present in 14/107, types=['dict']
- speaker_state: present in 30/107, types=['dict']
- text: present in 107/107, types=['str']
- text_sha2

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۶. ممیزی اتصال صریح پایلوت‌ها به پرونده‌های منشأ

Golden Dataset هر منبع را با نام‌هایی مانند `pilot_1` تا `pilot_28` معرفی می‌کند.

در این مرحله بررسی می‌کنیم آیا رجیستری بازبینی انسانی نیز شمارهٔ پایلوت یا شناسهٔ seed را به‌صورت صریح نگه داشته است یا نه.  
فقط اتصال‌های صریح پذیرفته می‌شوند؛ ترتیب رکوردها یا شباهت متن مبنای نگاشت نخواهد بود.

</div>

In [28]:
# Code Cell 18

import re

pilot_or_seed_pattern = re.compile(
    r"(pilot|seed|golden)",
    flags=re.IGNORECASE,
)

def discover_named_paths(value, path="root", findings=None):
    if findings is None:
        findings = []

    if isinstance(value, dict):
        for key, nested_value in value.items():
            nested_path = f"{path}.{key}"

            if pilot_or_seed_pattern.search(key):
                findings.append(
                    {
                        "path": nested_path,
                        "value_type": type(nested_value).__name__,
                        "value_preview": (
                            f"object with fields {sorted(nested_value)}"
                            if isinstance(nested_value, dict)
                            else (
                                f"list with {len(nested_value)} items"
                                if isinstance(nested_value, list)
                                else repr(nested_value)[:300]
                            )
                        ),
                    }
                )

            discover_named_paths(
                nested_value,
                path=nested_path,
                findings=findings,
            )

    elif isinstance(value, list):
        for index, nested_value in enumerate(value):
            discover_named_paths(
                nested_value,
                path=f"{path}[{index}]",
                findings=findings,
            )

    return findings


print("=== EXPLICIT PILOT / SEED LINKAGE DISCOVERY ===")

golden_lineages = sorted(
    {
        record["source_lineage"]
        for record in golden_dataset
    },
    key=lambda value: int(value.rsplit("_", 1)[1]),
)

print("Golden Dataset source lineages:")
print(golden_lineages)

registry_linkage_findings = discover_named_paths(
    completed_review_cases,
    path="completed_human_review_cases",
)

packet_linkage_findings = discover_named_paths(
    source_packet_pilots,
    path="source_packet_pilots",
)

print("\nRegistry fields or values with pilot/seed/golden in their path:")
for finding in registry_linkage_findings:
    print(
        f"- {finding['path']}: "
        f"{finding['value_type']} | {finding['value_preview']}"
    )

print("\nSource-packet fields or values with pilot/seed/golden in their path:")
for finding in packet_linkage_findings:
    print(
        f"- {finding['path']}: "
        f"{finding['value_type']} | {finding['value_preview']}"
    )

source_packet_numbers = sorted(
    pilot["reference_pilot_number"]
    for pilot in source_packet_pilots
)

print("\nExplicit Source-Packet pilot numbers:", source_packet_numbers)

assert source_packet_numbers == [23, 24, 25, 26, 27, 28], (
    "Source Packet must explicitly cover pilots 23 through 28."
)

print("\nEXPLICIT PILOT / SEED LINKAGE DISCOVERY: PASS")

=== EXPLICIT PILOT / SEED LINKAGE DISCOVERY ===
Golden Dataset source lineages:
['pilot_1', 'pilot_2', 'pilot_3', 'pilot_4', 'pilot_5', 'pilot_6', 'pilot_7', 'pilot_8', 'pilot_9', 'pilot_10', 'pilot_11', 'pilot_12', 'pilot_13', 'pilot_14', 'pilot_15', 'pilot_16', 'pilot_17', 'pilot_18', 'pilot_19', 'pilot_20', 'pilot_21', 'pilot_22', 'pilot_23', 'pilot_24', 'pilot_25', 'pilot_26', 'pilot_27', 'pilot_28']

Registry fields or values with pilot/seed/golden in their path:
- completed_human_review_cases[0].citation_review.human_review.golden_case_id: str | 'golden-case:3b93ce20ef4b617b9f922baddfbd1e0f'
- completed_human_review_cases[0].citation_review.human_review.pilot_case_id: str | 'question-pilot:3005b48518a7aeb532234332df23c952'
- completed_human_review_cases[0].expected_answer_review.human_review.golden_case_id: str | 'golden-case:3b93ce20ef4b617b9f922baddfbd1e0f'
- completed_human_review_cases[0].expected_answer_review.human_review.pilot_case_id: str | 'question-pilot:3005b48518a7aeb

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۷. کشف رجیسترهای تاریخچه و نگاشت اصلی

برای ۲۲ پروندهٔ اول، اتصال شمارهٔ پایلوت به `pilot_case_id` هنوز صریحاً پیدا نشده است.

در این مرحله بخش‌های تاریخچه، وضعیت ادامهٔ کار، خلاصهٔ بازبینی و بازبینی مؤلف در رجیستری بررسی می‌شوند.  
هدف فقط یافتن نگاشت ثبت‌شده است؛ ترتیب رکوردها مبنای نتیجه‌گیری نخواهد بود.

</div>

In [29]:
# Code Cell 19

def describe_top_level_object(label, payload):
    assert isinstance(payload, dict), f"{label} must be a JSON object."

    print(f"\n{label}")
    print(f"Field count: {len(payload)}")

    for field_name in sorted(payload):
        value = payload[field_name]

        if isinstance(value, dict):
            preview = f"object with fields {sorted(value)}"
        elif isinstance(value, list):
            preview = f"list with {len(value)} items"
        else:
            preview = repr(value)
            if len(preview) > 500:
                preview = preview[:497] + "..."

        print(f"- {field_name}: {preview}")


print("=== HUMAN REVIEW REGISTRY CONTROL-PLANE DISCOVERY ===")

for field_name in [
    "historical_completed_pilot",
    "continuation_state",
    "human_review_summary",
    "author_review",
    "retarget_candidate_discovery",
]:
    field_value = human_review_registry.get(field_name)

    if isinstance(field_value, dict):
        describe_top_level_object(
            f"Registry field: {field_name}",
            field_value,
        )
    else:
        print(
            f"\nRegistry field: {field_name}\n"
            f"- unavailable as an object; type={type(field_value).__name__}"
        )

print("\nHUMAN REVIEW REGISTRY CONTROL-PLANE DISCOVERY: PASS")

=== HUMAN REVIEW REGISTRY CONTROL-PLANE DISCOVERY ===

Registry field: historical_completed_pilot
Field count: 2
- history: object with fields ['cell82_first_question_draft_record', 'cell82_question_field_values', 'cell83_question_record', 'cell84_human_review_packet', 'cell84_question_record', 'cell85_human_adjudication_record', 'cell86_revised_question_draft_record', 'cell87_question_record', 'cell88_accepted_question_state', 'cell89_expected_answer_record', 'cell90_expected_answer_review_packet', 'cell91_accepted_qa_state', 'cell91_expected_answer_acceptance_record', 'cell92_citation_target_mapping_record', 'cell93_citation_human_review_packet', 'cell94_citation_acceptance_record', 'cell94_completed_pilot_qa_record']
- pilot_id: 'question-pilot:02ba2886f1b68521b93789becf741361'

Registry field: continuation_state
Field count: 17
- active_citation_mapping_id: None
- active_pilot_status: 'NONE'
- active_target_evidence_id: None
- author_final_review_status: 'NOT_STARTED'
- authority_p

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۸. کشف قرارداد ۲۸ پروندهٔ آمادهٔ بازبینی مؤلف

رجیستری یک فهرست مستقل از ۲۸ پروندهٔ آمادهٔ بازبینی مؤلف دارد.

این فهرست محتمل‌ترین محل برای نگاشت رسمی میان شمارهٔ پایلوت، شناسهٔ پرونده و منشأ شواهد است.  
در این مرحله فقط ساختار آن بررسی می‌شود و هیچ نگاشتی ساخته یا تغییر داده نمی‌شود.

</div>

In [30]:
# Code Cell 20

author_review_cases = human_review_registry["author_review"]["cases_ready_for_review"]

assert isinstance(author_review_cases, list), (
    "author_review.cases_ready_for_review must be a list."
)
assert len(author_review_cases) == 28, (
    f"Expected 28 author-review cases, found {len(author_review_cases)}."
)
assert all(isinstance(case, dict) for case in author_review_cases), (
    "Every author-review case must be a JSON object."
)

print("=== AUTHOR REVIEW CASE CONTRACT DISCOVERY ===")

all_fields = sorted(
    {
        field_name
        for case in author_review_cases
        for field_name in case
    }
)

print(f"Author-review case count: {len(author_review_cases)}")
print("Observed fields:")

for field_name in all_fields:
    present_count = sum(field_name in case for case in author_review_cases)
    value_types = sorted(
        {
            type(case[field_name]).__name__
            for case in author_review_cases
            if field_name in case
        }
    )
    print(
        f"- {field_name}: present in {present_count}/{len(author_review_cases)}, "
        f"types={value_types}"
    )

print("\nFirst author-review case preview:")
for field_name in sorted(author_review_cases[0]):
    value = author_review_cases[0][field_name]

    if isinstance(value, dict):
        preview = f"object with fields {sorted(value)}"
    elif isinstance(value, list):
        preview = f"list with {len(value)} items"
    else:
        preview = repr(value)
        if len(preview) > 500:
            preview = preview[:497] + "..."

    print(f"- {field_name}: {preview}")

print("\nPilot-related scalar values found in all author-review cases:")
for case_index, case in enumerate(author_review_cases, start=1):
    pilot_values = {
        field_name: value
        for field_name, value in case.items()
        if (
            "pilot" in field_name.lower()
            or "seed" in field_name.lower()
        )
        and not isinstance(value, (dict, list))
    }

    if pilot_values:
        print(f"- author-review record {case_index}: {pilot_values}")

print("\nAUTHOR REVIEW CASE CONTRACT DISCOVERY: PASS")

=== AUTHOR REVIEW CASE CONTRACT DISCOVERY ===
Author-review case count: 28
Observed fields:
- author_final_review: present in 28/28, types=['dict']
- author_review_queue_position: present in 6/28, types=['int']
- authority: present in 6/28, types=['dict']
- authority_expectation: present in 9/28, types=['str']
- authority_policy: present in 7/28, types=['dict']
- batch_version: present in 6/28, types=['str']
- borrowing_evidence_search: present in 1/28, types=['dict']
- borrowing_hierarchy_audit: present in 1/28, types=['dict']
- citation_review: present in 28/28, types=['dict']
- combined_target_text: present in 18/28, types=['str']
- completion_summary: present in 14/28, types=['dict']
- corrected_target_interpretation: present in 1/28, types=['dict']
- deferred_question_candidate: present in 1/28, types=['dict']
- derived_question: present in 6/28, types=['dict']
- expected_answer_review: present in 28/28, types=['dict']
- expected_behavior_review: present in 1/28, types=['dict']
- 

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۹. آزمون تطابق دقیق منشأ Golden با شواهد بازبینی

برای بازیابی نگاشت پایلوت‌های ۱ تا ۲۲، متن شواهد Golden فقط با تطابق دقیقِ نرمال‌شده بررسی می‌شود.

تطابق تقریبی، رتبه‌بندی شباهت یا ترتیب رکوردها در این مرحله استفاده نمی‌شود.  
فقط تطابق دقیق و یکتا می‌تواند به‌عنوان اتصال ثبت‌شده پذیرفته شود.

</div>

In [31]:
# Code Cell 21

import unicodedata

def canonical_exact_text(value):
    value = unicodedata.normalize("NFC", value)
    value = value.replace("\u200c", " ")
    value = value.replace("\u200f", " ")
    value = value.replace("\u200e", " ")
    value = value.replace("\u00ad", "")
    value = value.replace("ي", "ی")
    value = value.replace("ك", "ک")
    return " ".join(value.split())


gold_text_by_seed = {
    seed_id: canonical_exact_text(registry_record["source_evidence"])
    for seed_id, registry_record in gold_evidence_registry.items()
}

provenance_text_units = []

for review_index, review_case in enumerate(completed_review_cases, start=1):
    for evidence_index, evidence_item in enumerate(
        review_case.get("source_evidence", []),
        start=1,
    ):
        text = evidence_item.get("text")

        if isinstance(text, str) and text.strip():
            provenance_text_units.append(
                {
                    "origin": "human_review_registry",
                    "review_record_index": review_index,
                    "pilot_case_id": review_case["pilot_case_id"],
                    "evidence_index": evidence_index,
                    "document_id": evidence_item.get("document_id"),
                    "text": canonical_exact_text(text),
                }
            )

for packet_pilot in source_packet_pilots:
    for claim in packet_pilot.get("claims", []):
        description = claim.get("description")

        if isinstance(description, str) and description.strip():
            content_source = claim.get("content_source", {})

            provenance_text_units.append(
                {
                    "origin": "source_packet",
                    "reference_pilot_number": packet_pilot[
                        "reference_pilot_number"
                    ],
                    "pilot_case_id": packet_pilot["pilot_case_id"],
                    "claim_id": claim.get("claim_id"),
                    "document_id": (
                        content_source.get("document_id")
                        if isinstance(content_source, dict)
                        else None
                    ),
                    "text": canonical_exact_text(description),
                }
            )

assert provenance_text_units, "No provenance text units were collected."

print("=== STRICT GOLD-TO-PROVENANCE TEXT MATCHING ===")
print(f"Golden source texts: {len(gold_text_by_seed)}")
print(f"Provenance text units: {len(provenance_text_units)}")

strict_matches_by_seed = {}

for seed_id, gold_text in sorted(gold_text_by_seed.items()):
    exact_matches = [
        unit
        for unit in provenance_text_units
        if unit["text"] == gold_text
    ]

    strict_matches_by_seed[seed_id] = exact_matches

matched_seed_ids = [
    seed_id
    for seed_id, matches in strict_matches_by_seed.items()
    if matches
]

unique_match_seed_ids = [
    seed_id
    for seed_id, matches in strict_matches_by_seed.items()
    if len(matches) == 1
]

print(f"\nSeeds with one or more exact matches: {len(matched_seed_ids)}")
print(f"Seeds with exactly one exact match: {len(unique_match_seed_ids)}")

for seed_id in sorted(matched_seed_ids):
    print(f"\nseed {seed_id}: {len(strict_matches_by_seed[seed_id])} exact match(es)")

    for match in strict_matches_by_seed[seed_id]:
        identity = {
            key: value
            for key, value in match.items()
            if key != "text" and value is not None
        }
        print(f"- {identity}")

unmatched_seed_ids = sorted(
    set(gold_text_by_seed) - set(matched_seed_ids)
)

print("\nSeeds without an exact provenance-text match:")
print(unmatched_seed_ids)

print("\nSTRICT GOLD-TO-PROVENANCE TEXT MATCHING: PASS")

=== STRICT GOLD-TO-PROVENANCE TEXT MATCHING ===
Golden source texts: 28
Provenance text units: 129

Seeds with one or more exact matches: 0
Seeds with exactly one exact match: 0

Seeds without an exact provenance-text match:
['002b7b487dc2a05baa12c4121c092ff4ff19fccd24c1fad2e7d08d3595c876a6', '00a99af51bd7212fd003d1ee3475b1d42b21814f081513bd42165d725fb02a9c', '1695a706cb5d85e36c365af2542e478e4b4ca1ad6f526dc234e52dfe9013218b', '198458160870e17e100f8a280987167efc4a9e6baffcb7b01853366e79c5b1b6', '1f1f55bce79db1f11220e07e9fae6b19cdc492b0979d5e823aa92fba1b4175c8', '1ff6a0fe4296342e28c59e8b93d9e67d190f9eb01573d8bf4c23cb551d7c1651', '2066b0d9796587a80953d1d99612436cda296b5ecce48e12bbacdcaf5e5f7316', '2b1548a46b381a8b0ea610eeae302eb4ccfa5f924f41bed3a9d93732950d2ff1', '344c97fac66402b8513d959975f74342458bb1db1016fa7f6fb9f01de7dc0c23', '3b3eb00dc206806d143c84e5c7eb239208d18e03174462740df98ce67123431f', '3f660312f96d69f1659a01b85fb201345f6036c3e80c75c595faea07c2652761', '46a1acbf37ee1386f2365d97a

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۰. کشف ساختار فایل رسمی بازبینی ۲۸ پایلوت

تطابق دقیق متن Golden با قطعات شواهد، هیچ اتصال قابل‌قبولی ایجاد نکرد.

در این مرحله فایل رسمی بازبینی ۲۸ پایلوت بررسی می‌شود تا ببینیم آیا شمارهٔ پرونده، شمارهٔ پایلوت یا `pilot_case_id` به‌صورت صریح در آن ثبت شده است یا نه.

</div>

In [32]:
# Code Cell 22

author_review_markdown_path = next(
    (
        path
        for path in provenance_artifacts
        if path.name == "author_review_28_pilots.md"
    ),
    None,
)

assert author_review_markdown_path is not None, (
    "author_review_28_pilots.md was not found among provenance artifacts."
)

author_review_markdown = author_review_markdown_path.read_text(
    encoding="utf-8"
)

assert author_review_markdown.strip(), (
    "author_review_28_pilots.md is empty."
)

heading_pattern = re.compile(
    r"^\s{0,3}#{1,6}\s+.*?(?:case|pilot|پرونده|پایلوت).*?$",
    flags=re.IGNORECASE | re.MULTILINE,
)

identifier_pattern = re.compile(
    r"(?:question-pilot:[a-f0-9]+|pilot[_\s-]?\d+|source_seed_id)",
    flags=re.IGNORECASE,
)

heading_lines = heading_pattern.findall(author_review_markdown)
identifier_lines = [
    line.strip()
    for line in author_review_markdown.splitlines()
    if identifier_pattern.search(line)
]

print("=== AUTHOR REVIEW MARKDOWN STRUCTURE DISCOVERY ===")
print(
    "Path:",
    author_review_markdown_path.relative_to(project_root),
)
print("Character count:", len(author_review_markdown))
print("Heading lines mentioning case or pilot:", len(heading_lines))
print("Lines containing pilot-related identifiers:", len(identifier_lines))

print("\nHeading preview:")
for line in heading_lines[:40]:
    print("-", line.strip())

print("\nIdentifier-line preview:")
for line in identifier_lines[:60]:
    preview = line
    if len(preview) > 500:
        preview = preview[:497] + "..."
    print("-", preview)

print("\nAUTHOR REVIEW MARKDOWN STRUCTURE DISCOVERY: PASS")

=== AUTHOR REVIEW MARKDOWN STRUCTURE DISCOVERY ===
Path: notebooks/phase2/archive/author_review_28_pilots.md
Character count: 163318
Heading lines mentioning case or pilot: 0
Lines containing pilot-related identifiers: 0

Heading preview:

Identifier-line preview:

AUTHOR REVIEW MARKDOWN STRUCTURE DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۱. مشاهدهٔ ساختار واقعی فایل بازبینی مؤلف

جست‌وجوی مستقیم برای واژه‌های پایلوت و پرونده در فایل بازبینی نتیجه‌ای نداشت.

در این مرحله فقط تیترها، سرستون‌های جدول و ابتدای ساختار متن نمایش داده می‌شود تا روش ثبت هر مورد مشخص شود.  
این مشاهده هیچ تغییری در داده‌ها ایجاد نمی‌کند.

</div>

In [33]:
# Code Cell 23

markdown_lines = author_review_markdown.splitlines()

all_heading_lines = [
    line.strip()
    for line in markdown_lines
    if re.match(r"^\s{0,3}#{1,6}\s+\S", line)
]

table_header_lines = [
    line.strip()
    for line in markdown_lines
    if "|" in line and not re.fullmatch(r"[\s|:-]+", line)
]

nonempty_lines = [
    (line_number, line.strip())
    for line_number, line in enumerate(markdown_lines, start=1)
    if line.strip()
]

print("=== AUTHOR REVIEW MARKDOWN GENERAL STRUCTURE ===")
print("Markdown headings:", len(all_heading_lines))
print("Potential table rows:", len(table_header_lines))
print("Non-empty lines:", len(nonempty_lines))

print("\nHeading preview:")
for line in all_heading_lines[:60]:
    print("-", line)

print("\nPotential table-row preview:")
for line in table_header_lines[:20]:
    preview = line if len(line) <= 400 else line[:397] + "..."
    print("-", preview)

print("\nFirst 80 non-empty lines:")
for line_number, line in nonempty_lines[:80]:
    preview = line if len(line) <= 400 else line[:397] + "..."
    print(f"{line_number}: {preview}")

print("\nAUTHOR REVIEW MARKDOWN GENERAL STRUCTURE: PASS")

=== AUTHOR REVIEW MARKDOWN GENERAL STRUCTURE ===
Markdown headings: 0
Potential table rows: 0
Non-empty lines: 3402

Heading preview:

Potential table-row preview:

First 80 non-empty lines:
1: <div dir="rtl" align="right">
3: <h1 dir="rtl" align="right">بسته بازبینی مؤلف — ۲۸ مورد واقعی Golden Dataset</h1>
5: <p dir="rtl" align="right">این نسخه برای بازبینی دکتر کمیل رودی تهیه شده است. در هر مورد، ابتدا منبع و تاریخچه واقعی بازبینی نمایش داده می‌شود؛ اگر Target یا منبع عوض شده باشد، منبع اولیه و وضعیت سؤال/پاسخ آن قبل از منبع جایگزین آمده است.</p>
7: <p dir="rtl" align="right">ترتیب هر مورد ثابت است: متن منبع و تاریخچه منبع → روند سؤال و پاسخ → تصمیم‌های بازبینی → سؤال و پاسخ نهایی → نظر مؤلف.</p>
9: </div>
12: ---
15: <div dir="rtl" align="right">
17: <h2 dir="rtl" align="right">مورد 1 از ۲۸</h2>
19: <h3 dir="rtl" align="right">۱. متن منبع (بخشی از متن اصلی)</h3>
21: <p dir="rtl" align="right">تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواه

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۲. اتصال دقیق موارد فایل بازبینی به پرونده‌های رجیستری

فایل رسمی بازبینی، ۲۸ مورد شماره‌دار و متن منبع هر مورد را ثبت کرده است.

در این مرحله، متن منبع هر مورد فقط با تطابق دقیقِ نرمال‌شده به شواهد پرونده‌های رجیستری وصل می‌شود.  
اگر تطابق دقیق و یکتا نباشد، آن مورد بدون نگاشت باقی می‌ماند.

</div>

In [35]:
# Code Cell 24

import html

persian_digit_translation = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789",
)

case_heading_pattern = re.compile(
    r"""
    <h2\b[^>]*>
    \s*مورد\s+([۰-۹0-9]+)\s+از\s+[۰-۹0-9]+\s*
    </h2>
    """,
    flags=re.IGNORECASE | re.DOTALL | re.VERBOSE,
)

def html_to_plain_text(value):
    value = re.sub(r"<[^>]+>", " ", value)
    value = html.unescape(value)
    return canonical_exact_text(value)


case_heading_matches = list(
    case_heading_pattern.finditer(author_review_markdown)
)

assert len(case_heading_matches) == 28, (
    "Expected 28 author-review case headings; "
    f"found {len(case_heading_matches)}."
)

author_review_source_cases = []
cases_without_initial_paragraph = []

for match_index, heading_match in enumerate(case_heading_matches):
    case_number = int(
        heading_match.group(1).translate(persian_digit_translation)
    )

    block_start = heading_match.end()
    block_end = (
        case_heading_matches[match_index + 1].start()
        if match_index + 1 < len(case_heading_matches)
        else len(author_review_markdown)
    )
    case_block = author_review_markdown[block_start:block_end]

    first_paragraph_match = re.search(
        r"<p\b[^>]*>(.*?)</p>",
        case_block,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if first_paragraph_match is None:
        cases_without_initial_paragraph.append(case_number)
        continue

    source_text = html_to_plain_text(first_paragraph_match.group(1))

    author_review_source_cases.append(
        {
            "case_number": case_number,
            "source_text": source_text,
        }
    )

author_review_source_cases.sort(key=lambda item: item["case_number"])

human_review_text_units = [
    {
        "review_record_index": review_index,
        "pilot_case_id": review_case["pilot_case_id"],
        "document_id": evidence_item.get("document_id"),
        "source_text": canonical_exact_text(evidence_item["text"]),
    }
    for review_index, review_case in enumerate(
        completed_review_cases,
        start=1,
    )
    for evidence_item in review_case.get("source_evidence", [])
    if isinstance(evidence_item.get("text"), str)
    and evidence_item["text"].strip()
]

print("=== AUTHOR-REVIEW CASE TO REGISTRY EXACT LINKAGE ===")
print("Numbered author-review headings:", len(case_heading_matches))
print(
    "Cases with an initial paragraph:",
    len(author_review_source_cases),
)
print(
    "Cases without an initial paragraph:",
    cases_without_initial_paragraph,
)
print("Human-review evidence units:", len(human_review_text_units))

case_to_registry_exact_matches = {}

for author_case in author_review_source_cases:
    exact_matches = [
        evidence_unit
        for evidence_unit in human_review_text_units
        if evidence_unit["source_text"] == author_case["source_text"]
    ]

    case_to_registry_exact_matches[author_case["case_number"]] = exact_matches

matched_case_numbers = [
    case_number
    for case_number, matches in case_to_registry_exact_matches.items()
    if matches
]

unique_case_numbers = [
    case_number
    for case_number, matches in case_to_registry_exact_matches.items()
    if len(matches) == 1
]

print(f"\nCases with one or more exact registry matches: {len(matched_case_numbers)}")
print(f"Cases with exactly one registry match: {len(unique_case_numbers)}")

for case_number in sorted(matched_case_numbers):
    print(
        f"\nAuthor-review case {case_number}: "
        f"{len(case_to_registry_exact_matches[case_number])} exact match(es)"
    )

    for match in case_to_registry_exact_matches[case_number]:
        print(
            "- "
            f"review_record_index={match['review_record_index']}, "
            f"pilot_case_id={match['pilot_case_id']}, "
            f"document_id={match['document_id']}"
        )

unmatched_case_numbers = sorted(
    set(range(1, 29)) - set(matched_case_numbers)
)

print("\nCases without an exact registry-source match:")
print(unmatched_case_numbers)

print("\nAUTHOR-REVIEW CASE TO REGISTRY EXACT LINKAGE: PASS")

=== AUTHOR-REVIEW CASE TO REGISTRY EXACT LINKAGE ===
Numbered author-review headings: 28
Cases with an initial paragraph: 28
Cases without an initial paragraph: []
Human-review evidence units: 107

Cases with one or more exact registry matches: 16
Cases with exactly one registry match: 16

Author-review case 1: 1 exact match(es)
- review_record_index=1, pilot_case_id=question-pilot:3005b48518a7aeb532234332df23c952, document_id=website:5C92BB0D

Author-review case 2: 1 exact match(es)
- review_record_index=2, pilot_case_id=question-pilot:083ad832854722e126c05d0b89d00cb1, document_id=website:596080EF

Author-review case 3: 1 exact match(es)
- review_record_index=3, pilot_case_id=question-pilot:b801c37bf6fc78b00fcc0c79b368ef7d, document_id=website:C185157D

Author-review case 6: 1 exact match(es)
- review_record_index=6, pilot_case_id=question-pilot:672f5241db66879fb47096c5bb0f5568, document_id=book:adult-financial-literacy-course:fa-source:p001307

Author-review case 7: 1 exact match(es)

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۳. تکمیل اتصال دقیق با همهٔ پاراگراف‌های هر مورد

برخی موارد با پاراگراف نخست فایل بازبینی به شواهد رجیستری وصل نشدند.

در این مرحله همهٔ پاراگراف‌های هر مورد بررسی می‌شوند.  
فقط تطابق دقیقِ متن با شواهد ثبت‌شده پذیرفته می‌شود و هنوز از ترتیب برای ساخت نگاشت استفاده نخواهد شد.

</div>

In [36]:
# Code Cell 25

author_case_paragraph_units = []

for match_index, heading_match in enumerate(case_heading_matches):
    case_number = int(
        heading_match.group(1).translate(persian_digit_translation)
    )

    block_start = heading_match.end()
    block_end = (
        case_heading_matches[match_index + 1].start()
        if match_index + 1 < len(case_heading_matches)
        else len(author_review_markdown)
    )
    case_block = author_review_markdown[block_start:block_end]

    paragraph_html_values = re.findall(
        r"<p\b[^>]*>(.*?)</p>",
        case_block,
        flags=re.IGNORECASE | re.DOTALL,
    )

    for paragraph_index, paragraph_html in enumerate(
        paragraph_html_values,
        start=1,
    ):
        paragraph_text = html_to_plain_text(paragraph_html)

        if paragraph_text:
            author_case_paragraph_units.append(
                {
                    "case_number": case_number,
                    "paragraph_index": paragraph_index,
                    "source_text": paragraph_text,
                }
            )

assert author_case_paragraph_units, (
    "No paragraph text was extracted from the author-review cases."
)

print("=== FULL AUTHOR-REVIEW PARAGRAPH EXACT LINKAGE ===")
print("Author-review paragraph units:", len(author_case_paragraph_units))
print("Human-review evidence units:", len(human_review_text_units))

case_to_all_exact_matches = defaultdict(list)

for paragraph_unit in author_case_paragraph_units:
    for evidence_unit in human_review_text_units:
        if paragraph_unit["source_text"] == evidence_unit["source_text"]:
            case_to_all_exact_matches[paragraph_unit["case_number"]].append(
                {
                    "paragraph_index": paragraph_unit["paragraph_index"],
                    "review_record_index": evidence_unit[
                        "review_record_index"
                    ],
                    "pilot_case_id": evidence_unit["pilot_case_id"],
                    "document_id": evidence_unit["document_id"],
                }
            )

matched_case_numbers = sorted(case_to_all_exact_matches)
unmatched_case_numbers = sorted(
    set(range(1, 29)) - set(matched_case_numbers)
)

unique_registry_indices_by_case = {
    case_number: sorted(
        {
            match["review_record_index"]
            for match in matches
        }
    )
    for case_number, matches in case_to_all_exact_matches.items()
}

print(
    "\nCases with one or more exact paragraph-to-registry matches:",
    len(matched_case_numbers),
)

for case_number in matched_case_numbers:
    registry_indices = unique_registry_indices_by_case[case_number]

    print(
        f"\nAuthor-review case {case_number}: "
        f"registry indices={registry_indices}"
    )

    for match in case_to_all_exact_matches[case_number]:
        print(
            "- "
            f"paragraph={match['paragraph_index']}, "
            f"pilot_case_id={match['pilot_case_id']}, "
            f"document_id={match['document_id']}"
        )

print("\nCases without an exact paragraph-to-registry match:")
print(unmatched_case_numbers)

index_alignment_cases = [
    case_number
    for case_number, registry_indices in unique_registry_indices_by_case.items()
    if registry_indices == [case_number]
]

print(
    "\nCases whose exact match points to the same registry index:",
    index_alignment_cases,
)
print(
    "Count of same-index exact matches:",
    len(index_alignment_cases),
)

print("\nFULL AUTHOR-REVIEW PARAGRAPH EXACT LINKAGE: PASS")

=== FULL AUTHOR-REVIEW PARAGRAPH EXACT LINKAGE ===
Author-review paragraph units: 236
Human-review evidence units: 107

Cases with one or more exact paragraph-to-registry matches: 22

Author-review case 1: registry indices=[1]
- paragraph=1, pilot_case_id=question-pilot:3005b48518a7aeb532234332df23c952, document_id=website:5C92BB0D

Author-review case 2: registry indices=[2]
- paragraph=1, pilot_case_id=question-pilot:083ad832854722e126c05d0b89d00cb1, document_id=website:596080EF

Author-review case 3: registry indices=[3]
- paragraph=1, pilot_case_id=question-pilot:b801c37bf6fc78b00fcc0c79b368ef7d, document_id=website:C185157D

Author-review case 4: registry indices=[4]
- paragraph=6, pilot_case_id=question-pilot:909b695797935246c92c7a66ebe0f773, document_id=book:trainer-training-financial-literacy:fa-source:p000323
- paragraph=7, pilot_case_id=question-pilot:909b695797935246c92c7a66ebe0f773, document_id=book:trainer-training-financial-literacy:fa-source:p000324
- paragraph=8, pilot_c

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۴. آزمون اتصال دقیق موارد بازبینی به seedهای Golden

برای تکمیل زنجیرهٔ منشأ، سؤال‌های Golden با متن کامل هر مورد در فایل رسمی بازبینی مقایسه می‌شوند.

فقط سؤال‌هایی پذیرفته می‌شوند که به‌صورت دقیق و نرمال‌شده در متن همان مورد وجود داشته باشند.  
این آزمون، اتصال «مورد بازبینی → source_seed_id» را مستقل از ترتیب فهرست بررسی می‌کند.

</div>

In [37]:
# Code Cell 26

author_case_blocks = {}

for match_index, heading_match in enumerate(case_heading_matches):
    case_number = int(
        heading_match.group(1).translate(persian_digit_translation)
    )

    block_start = heading_match.end()
    block_end = (
        case_heading_matches[match_index + 1].start()
        if match_index + 1 < len(case_heading_matches)
        else len(author_review_markdown)
    )

    author_case_blocks[case_number] = html_to_plain_text(
        author_review_markdown[block_start:block_end]
    )

assert sorted(author_case_blocks) == list(range(1, 29)), (
    "Expected complete author-review blocks numbered 1 through 28."
)

gold_questions_by_seed = defaultdict(set)

for golden_record in golden_dataset:
    question = canonical_exact_text(golden_record["question"])

    if len(question) >= 25:
        gold_questions_by_seed[golden_record["source_seed_id"]].add(question)

assert gold_questions_by_seed, "No usable Golden Dataset questions were found."

print("=== STRICT AUTHOR-REVIEW TO GOLDEN QUESTION LINKAGE ===")
print("Author-review cases:", len(author_case_blocks))
print("Golden seeds with usable questions:", len(gold_questions_by_seed))

case_to_question_seed_matches = defaultdict(list)

for case_number, case_text in author_case_blocks.items():
    for source_seed_id, questions in gold_questions_by_seed.items():
        for question in questions:
            if question in case_text:
                case_to_question_seed_matches[case_number].append(
                    {
                        "source_seed_id": source_seed_id,
                        "question": question,
                    }
                )

case_to_unique_seed_ids = {
    case_number: sorted(
        {
            match["source_seed_id"]
            for match in matches
        }
    )
    for case_number, matches in case_to_question_seed_matches.items()
}

matched_case_numbers = sorted(case_to_unique_seed_ids)
unique_seed_case_numbers = [
    case_number
    for case_number, seed_ids in case_to_unique_seed_ids.items()
    if len(seed_ids) == 1
]

print("\nCases with one or more exact Golden-question matches:")
print(matched_case_numbers)

print("\nCases mapped to exactly one Golden seed:")
print(unique_seed_case_numbers)

for case_number in matched_case_numbers:
    seed_ids = case_to_unique_seed_ids[case_number]

    print(f"\nAuthor-review case {case_number}: seed IDs={seed_ids}")

    for match in case_to_question_seed_matches[case_number]:
        preview = match["question"]
        if len(preview) > 300:
            preview = preview[:297] + "..."

        print(
            f"- seed={match['source_seed_id']}, "
            f"question={preview!r}"
        )

unmatched_case_numbers = sorted(
    set(author_case_blocks) - set(matched_case_numbers)
)

print("\nCases without an exact Golden-question match:")
print(unmatched_case_numbers)

print("\nSTRICT AUTHOR-REVIEW TO GOLDEN QUESTION LINKAGE: PASS")

=== STRICT AUTHOR-REVIEW TO GOLDEN QUESTION LINKAGE ===
Author-review cases: 28
Golden seeds with usable questions: 28

Cases with one or more exact Golden-question matches:
[]

Cases mapped to exactly one Golden seed:
[]

Cases without an exact Golden-question match:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]

STRICT AUTHOR-REVIEW TO GOLDEN QUESTION LINKAGE: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۵. کشف نگاشت ثبت‌شده در نوت‌بوک یکپارچه‌سازی بازخورد مؤلف

تطابق دقیق متن منبع و سؤال، اتصال مستقیمی میان موارد بازبینی و seedهای Golden ایجاد نکرد.

در این مرحله نوت‌بوک یکپارچه‌سازی بازخورد مؤلف فقط خوانده و جست‌وجو می‌شود تا نگاشت ثبت‌شده میان پرونده‌های بازبینی، `source_seed_id` و پایلوت‌ها پیدا شود.

</div>

In [38]:
# Code Cell 27

author_feedback_notebook_candidates = sorted(
    project_root.rglob("*author*feedback*integration*.ipynb")
)

print("=== AUTHOR FEEDBACK NOTEBOOK MAPPING DISCOVERY ===")
print("Candidate notebooks found:")

for path in author_feedback_notebook_candidates:
    print("-", path.relative_to(project_root))

assert author_feedback_notebook_candidates, (
    "No author-feedback integration notebook was found in the project."
)
assert len(author_feedback_notebook_candidates) == 1, (
    "More than one author-feedback integration notebook was found; "
    "selecting one automatically would be unsafe."
)

author_feedback_notebook_path = author_feedback_notebook_candidates[0]

with author_feedback_notebook_path.open(encoding="utf-8") as handle:
    author_feedback_notebook = json.load(handle)

notebook_cells = author_feedback_notebook.get("cells", [])

assert isinstance(notebook_cells, list), (
    "The author-feedback notebook does not contain a valid cells list."
)

mapping_terms = [
    "source_seed_id",
    "source_lineage",
    "pilot_",
    "pilot_case_id",
    "review case",
    "مورد",
    "golden_dataset",
]

matching_cells = []

for cell_index, cell in enumerate(notebook_cells):
    source = "".join(cell.get("source", []))
    source_lower = source.lower()

    matched_terms = [
        term
        for term in mapping_terms
        if term.lower() in source_lower
    ]

    if matched_terms:
        matching_cells.append(
            {
                "cell_index": cell_index,
                "cell_type": cell.get("cell_type"),
                "matched_terms": matched_terms,
                "source": source,
            }
        )

print("\nNotebook path:", author_feedback_notebook_path.relative_to(project_root))
print("Notebook cells:", len(notebook_cells))
print("Cells mentioning mapping-related terms:", len(matching_cells))

for match in matching_cells:
    source_preview = match["source"].strip()

    if len(source_preview) > 1400:
        source_preview = source_preview[:1397] + "..."

    print(
        f"\n--- Cell {match['cell_index']} "
        f"({match['cell_type']}) ---"
    )
    print("Matched terms:", match["matched_terms"])
    print(source_preview)

print("\nAUTHOR FEEDBACK NOTEBOOK MAPPING DISCOVERY: PASS")

=== AUTHOR FEEDBACK NOTEBOOK MAPPING DISCOVERY ===
Candidate notebooks found:
- notebooks/phase2/12f_author_feedback_integration.ipynb

Notebook path: notebooks/phase2/12f_author_feedback_integration.ipynb
Notebook cells: 28
Cells mentioning mapping-related terms: 19

--- Cell 0 (markdown) ---
Matched terms: ['review case', 'مورد']
<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Notebook Introduction — Author Feedback Integration</h3>

<p>
این Notebook برای مرحله ادغام بازخورد Expert Review روی
<code>Golden Dataset v1</code>
ساخته شده است.
</p>

<p>
هدف این Notebook ایجاد یک مسیر کنترل‌شده، قابل‌ردیابی و Versioned برای بررسی بازخورد مؤلف، ثبت تصمیم‌های اصلاحی و آماده‌سازی تولید نسخه بعدی Golden Dataset است.
</p>

<p>
این مرحله بعد از تولید اولیه Golden Dataset v1 قرار دارد و قبل از ساخت Golden Dataset v2 اجرا می‌شود.
</p>

<h4>هدف اصلی Notebook</h4>

<ul>
<li>بررسی Artifactهای مربوط به Golden Dataset v1</li>
<li>بررسی Review Package ارسالشده برای Expert Review

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۶. بازیابی Lineage واقعی از نوت‌بوک تولید Golden

نوت‌بوک یکپارچه‌سازی بازخورد مؤلف نشان داد که منطق اصلی اتصال seedها به رکوردهای Golden در نوت‌بوک تولید Golden قرار دارد.

در این مرحله فقط سل‌های مرتبط NB12e خوانده می‌شوند تا مسیر واقعی
`Seed → Generated Records → Golden Dataset`
از خود کد تولید بازیابی شود.

</div>

In [39]:
# Code Cell 28

nb12e_candidates = sorted(
    project_root.rglob("*golden*dataset*full*generation*.ipynb")
)

print("=== NB12E LINEAGE CODE DISCOVERY ===")
print("Candidate notebooks found:")

for path in nb12e_candidates:
    print("-", path.relative_to(project_root))

assert nb12e_candidates, (
    "No Golden Dataset full-generation notebook was found."
)
assert len(nb12e_candidates) == 1, (
    "More than one full-generation notebook was found; "
    "selecting one automatically would be unsafe."
)

nb12e_path = nb12e_candidates[0]

with nb12e_path.open(encoding="utf-8") as handle:
    nb12e_notebook = json.load(handle)

nb12e_cells = nb12e_notebook.get("cells", [])

lineage_terms = [
    "source_seed_id",
    "source_lineage",
    "pilot_",
    "golden_dataset_v1",
    "golden_records",
    "candidate_id",
    "generation_status",
]

relevant_nb12e_cells = []

for cell_index, cell in enumerate(nb12e_cells):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    source_lower = source.lower()

    matched_terms = [
        term
        for term in lineage_terms
        if term.lower() in source_lower
    ]

    if matched_terms:
        relevant_nb12e_cells.append(
            {
                "cell_index": cell_index,
                "matched_terms": matched_terms,
                "source": source,
            }
        )

print("\nNotebook path:", nb12e_path.relative_to(project_root))
print("Total notebook cells:", len(nb12e_cells))
print("Relevant code cells:", len(relevant_nb12e_cells))

for item in relevant_nb12e_cells:
    source_preview = item["source"].strip()

    if len(source_preview) > 3000:
        source_preview = source_preview[:2997] + "..."

    print(f"\n--- Code cell {item['cell_index']} ---")
    print("Matched terms:", item["matched_terms"])
    print(source_preview)

print("\nNB12E LINEAGE CODE DISCOVERY: PASS")

=== NB12E LINEAGE CODE DISCOVERY ===
Candidate notebooks found:
- notebooks/phase2/12e_golden_dataset_full_generation.ipynb

Notebook path: notebooks/phase2/12e_golden_dataset_full_generation.ipynb
Total notebook cells: 154
Relevant code cells: 58

--- Code cell 7 ---
Matched terms: ['pilot_']
# Code Cell 04

pilot_headers = re.findall(
    r"مورد\s+\d+\s+از\s+۲۸",
    seed_text
)


unique_headers = list(
    dict.fromkeys(pilot_headers)
)


print("=== SEED STRUCTURE DISCOVERY ===")
print("Detected Pilot Headers:", len(pilot_headers))
print("Unique Pilot Headers:", len(unique_headers))


print("\n=== FIRST HEADERS ===")

for header in unique_headers[:10]:
    print("-", header)


missing_expected = 28 - len(unique_headers)


print("\nMissing Expected Pilot Headers:", missing_expected)


print("\nCODE CELL 04 GATE: PASS")

--- Code cell 9 ---
Matched terms: ['pilot_']
# Code Cell 05

pilot_blocks = re.split(
    r"(?=مورد\s+\d+\s+از\s+۲۸)",
    seed_text
)


pilot_blocks = [
    block.s

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۷. ساخت رجیستری قطعی منشأ Golden Dataset

منطق تولید NB12e مسیر زیر را به‌صورت مستقیم ثبت کرده است:

`مورد N بازبینی → pilot_number N → source_seed_id N → source_lineage pilot_N`

در این سل، برای هر seed فعال Golden Dataset، منشأ مستند آن شامل شناسهٔ پرونده، شناسهٔ سندها و نوع منبع ساخته می‌شود.

این رجیستری از Golden Dataset فعال ساخته می‌شود و در صورت ناسازگاری lineage، اجرای ارزیابی را متوقف می‌کند.

</div>

In [40]:
# Code Cell 29

review_case_by_pilot_id = {
    review_case["pilot_case_id"]: review_case
    for review_case in completed_review_cases
}

author_case_by_pilot_id = {
    author_case["pilot_case_id"]: author_case
    for author_case in author_review_cases
}

assert len(review_case_by_pilot_id) == 22, (
    "Human-review pilot_case_id values must be unique."
)
assert len(author_case_by_pilot_id) == 28, (
    "Author-review pilot_case_id values must be unique."
)

gold_records_by_seed = defaultdict(list)

for golden_record in golden_dataset:
    gold_records_by_seed[golden_record["source_seed_id"]].append(
        golden_record
    )

active_seed_ids = sorted(gold_records_by_seed)

assert active_seed_ids == list(range(1, 29)), (
    "The active Golden Dataset must contain source seeds 1 through 28. "
    f"Found: {active_seed_ids}"
)

for source_seed_id, records in gold_records_by_seed.items():
    observed_lineages = {
        record["source_lineage"]
        for record in records
    }

    assert observed_lineages == {f"pilot_{source_seed_id}"}, (
        f"Seed {source_seed_id} has inconsistent source lineage: "
        f"{observed_lineages}"
    )

gold_provenance_registry = {}

for case_number in range(1, 23):
    exact_matches = case_to_all_exact_matches.get(case_number, [])

    matched_pilot_case_ids = {
        match["pilot_case_id"]
        for match in exact_matches
    }

    matched_registry_indices = {
        match["review_record_index"]
        for match in exact_matches
    }

    assert len(matched_pilot_case_ids) == 1, (
        f"Review case {case_number} does not have exactly one "
        "exactly-linked human-review pilot_case_id."
    )
    assert matched_registry_indices == {case_number}, (
        f"Review case {case_number} does not map to the same "
        "human-review registry index."
    )

    pilot_case_id = next(iter(matched_pilot_case_ids))
    review_case = review_case_by_pilot_id[pilot_case_id]

    document_ids = sorted(
        {
            evidence_item["document_id"]
            for evidence_item in review_case.get("source_evidence", [])
            if isinstance(evidence_item.get("document_id"), str)
            and evidence_item["document_id"]
        }
    )

    gold_provenance_registry[case_number] = {
        "source_seed_id": case_number,
        "source_lineage": f"pilot_{case_number}",
        "review_case_number": case_number,
        "pilot_case_id": pilot_case_id,
        "provenance_source": "human_review_registry",
        "document_ids": document_ids,
    }

source_packet_by_number = {
    pilot["reference_pilot_number"]: pilot
    for pilot in source_packet_pilots
}

assert sorted(source_packet_by_number) == list(range(23, 29)), (
    "Source Packet must explicitly cover pilots 23 through 28."
)

for case_number in range(23, 29):
    packet_pilot = source_packet_by_number[case_number]
    pilot_case_id = packet_pilot["pilot_case_id"]

    assert pilot_case_id in author_case_by_pilot_id, (
        f"Source Packet pilot {case_number} is absent from author-review cases."
    )
    assert author_case_by_pilot_id[pilot_case_id].get(
        "reference_pilot_number"
    ) == case_number, (
        f"Author-review record for pilot {case_number} has inconsistent "
        "reference_pilot_number."
    )

    claim_document_ids = {
        claim.get("content_source", {}).get("document_id")
        for claim in packet_pilot.get("claims", [])
        if isinstance(claim.get("content_source"), dict)
    }

    document_ids = sorted(
        {
            document_id
            for document_id in (
                set(packet_pilot["accepted_website_document_ids"])
                | claim_document_ids
            )
            if isinstance(document_id, str) and document_id
        }
    )

    gold_provenance_registry[case_number] = {
        "source_seed_id": case_number,
        "source_lineage": f"pilot_{case_number}",
        "review_case_number": case_number,
        "pilot_case_id": pilot_case_id,
        "provenance_source": "source_packet",
        "document_ids": document_ids,
    }

assert sorted(gold_provenance_registry) == active_seed_ids, (
    "The provenance registry does not cover every active Golden seed."
)

print("=== ACTIVE GOLDEN PROVENANCE REGISTRY ===")
print("Active Golden seeds:", len(active_seed_ids))
print("Provenance registry entries:", len(gold_provenance_registry))

for source_seed_id in active_seed_ids:
    entry = gold_provenance_registry[source_seed_id]

    print(
        f"- seed {source_seed_id}: "
        f"source={entry['provenance_source']}, "
        f"documents={len(entry['document_ids'])}, "
        f"pilot_case_id={entry['pilot_case_id']}"
    )

print("\nACTIVE GOLDEN PROVENANCE REGISTRY: PASS")

=== ACTIVE GOLDEN PROVENANCE REGISTRY ===
Active Golden seeds: 28
Provenance registry entries: 28
- seed 1: source=human_review_registry, documents=1, pilot_case_id=question-pilot:3005b48518a7aeb532234332df23c952
- seed 2: source=human_review_registry, documents=1, pilot_case_id=question-pilot:083ad832854722e126c05d0b89d00cb1
- seed 3: source=human_review_registry, documents=1, pilot_case_id=question-pilot:b801c37bf6fc78b00fcc0c79b368ef7d
- seed 4: source=human_review_registry, documents=5, pilot_case_id=question-pilot:909b695797935246c92c7a66ebe0f773
- seed 5: source=human_review_registry, documents=8, pilot_case_id=question-pilot:3023c1c546df2fe2211f79702b6e3dbb
- seed 6: source=human_review_registry, documents=2, pilot_case_id=question-pilot:672f5241db66879fb47096c5bb0f5568
- seed 7: source=human_review_registry, documents=42, pilot_case_id=question-pilot:7334395fa6456f24ed250e2b354cc184
- seed 8: source=human_review_registry, documents=2, pilot_case_id=question-pilot:d7378ba1655fe8

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۸. بررسی پوشش سندهای منشأ در کاندیدهای چانکینگ

رجیستری منشأ برای هر seed، شناسهٔ سندهای معتبر را مشخص کرده است.

در این مرحله بررسی می‌شود که هر شناسهٔ سند در کدام چانک‌های هر کاندید چانکینگ وجود دارد.  
Seedهایی که شناسهٔ سند ندارند یا هیچ چانک متناظری ندارند، فعلاً وارد محاسبهٔ معیارهای retrieval نمی‌شوند.

</div>

In [43]:
# Code Cell 30

candidate_document_to_chunk_ids = {}
candidate_chunk_count_by_name = {}

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    assert candidate_path.is_file(), (
        f"Chunk candidate file is missing: {candidate_path}"
    )

    document_to_chunk_ids = defaultdict(set)
    chunk_count = 0

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            chunk_count += 1
            chunk_id = chunk["chunk_id"]

            document_ids = {
                document_id
                for document_id in chunk.get("member_document_ids", [])
                if isinstance(document_id, str) and document_id
            }

            if isinstance(chunk.get("source_id"), str) and chunk["source_id"]:
                document_ids.add(chunk["source_id"])

            for document_id in document_ids:
                document_to_chunk_ids[document_id].add(chunk_id)

    candidate_document_to_chunk_ids[candidate_name] = document_to_chunk_ids
    candidate_chunk_count_by_name[candidate_name] = chunk_count

print("=== PROVENANCE DOCUMENT-TO-CHUNK COVERAGE ===")
print("Active Golden provenance seeds:", len(gold_provenance_registry))

seeds_without_document_ids = [
    source_seed_id
    for source_seed_id, entry in gold_provenance_registry.items()
    if not entry["document_ids"]
]

print("Seeds without registered document IDs:", seeds_without_document_ids)

document_coverage_by_candidate = {}

for candidate_name, document_to_chunk_ids in (
    candidate_document_to_chunk_ids.items()
):
    seed_results = {}

    for source_seed_id, entry in gold_provenance_registry.items():
        expected_document_ids = entry["document_ids"]

        if not expected_document_ids:
            seed_results[source_seed_id] = {
                "status": "NO_REGISTERED_DOCUMENT_ID",
                "matched_document_ids": [],
                "positive_chunk_ids": set(),
            }
            continue

        matched_document_ids = sorted(
            document_id
            for document_id in expected_document_ids
            if document_id in document_to_chunk_ids
        )

        positive_chunk_ids = (
            set().union(
                *(
                    document_to_chunk_ids[document_id]
                    for document_id in matched_document_ids
                )
            )
            if matched_document_ids
            else set()
        )

        if len(matched_document_ids) == len(expected_document_ids):
            status = "FULL_DOCUMENT_COVERAGE"
        elif matched_document_ids:
            status = "PARTIAL_DOCUMENT_COVERAGE"
        else:
            status = "NO_DOCUMENT_COVERAGE"

        seed_results[source_seed_id] = {
            "status": status,
            "matched_document_ids": matched_document_ids,
            "positive_chunk_ids": positive_chunk_ids,
        }

    document_coverage_by_candidate[candidate_name] = seed_results

    status_counts = Counter(
        result["status"]
        for result in seed_results.values()
    )

    print(f"\n{candidate_name}")
    print("Chunk records indexed:", candidate_chunk_count_by_name[candidate_name])

    for status in [
        "FULL_DOCUMENT_COVERAGE",
        "PARTIAL_DOCUMENT_COVERAGE",
        "NO_DOCUMENT_COVERAGE",
        "NO_REGISTERED_DOCUMENT_ID",
    ]:
        print(f"- {status}: {status_counts[status]}")

    no_coverage_seed_ids = sorted(
        source_seed_id
        for source_seed_id, result in seed_results.items()
        if result["status"] == "NO_DOCUMENT_COVERAGE"
    )

    if no_coverage_seed_ids:
        print("- Seeds with no document coverage:", no_coverage_seed_ids)

print("\nPROVENANCE DOCUMENT-TO-CHUNK COVERAGE: PASS")

=== PROVENANCE DOCUMENT-TO-CHUNK COVERAGE ===
Active Golden provenance seeds: 28
Seeds without registered document IDs: [16]

baseline_phase1_char_safe
Chunk records indexed: 3488
- FULL_DOCUMENT_COVERAGE: 27
- PARTIAL_DOCUMENT_COVERAGE: 0
- NO_DOCUMENT_COVERAGE: 0
- NO_REGISTERED_DOCUMENT_ID: 1

parent_child_children
Chunk records indexed: 2721
- FULL_DOCUMENT_COVERAGE: 27
- PARTIAL_DOCUMENT_COVERAGE: 0
- NO_DOCUMENT_COVERAGE: 0
- NO_REGISTERED_DOCUMENT_ID: 1

structure_aware_medium
Chunk records indexed: 1799
- FULL_DOCUMENT_COVERAGE: 27
- PARTIAL_DOCUMENT_COVERAGE: 0
- NO_DOCUMENT_COVERAGE: 0
- NO_REGISTERED_DOCUMENT_ID: 1

structure_aware_small
Chunk records indexed: 2526
- FULL_DOCUMENT_COVERAGE: 27
- PARTIAL_DOCUMENT_COVERAGE: 0
- NO_DOCUMENT_COVERAGE: 0
- NO_REGISTERED_DOCUMENT_ID: 1

PROVENANCE DOCUMENT-TO-CHUNK COVERAGE: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۱۹. بازیابی شناسهٔ منشأ برای seed بدون document_id

۲۷ seed دارای شناسهٔ سند، در همهٔ کاندیدهای چانکینگ پوشش کامل دارند.

در این مرحله فقط شواهد ثبت‌شدهٔ seed بدون document_id بررسی می‌شود تا مشخص شود آیا شناسهٔ معتبر دیگری مانند `source_id` یا `source_path` برای آن وجود دارد یا نه.

</div>

In [44]:
# Code Cell 31

seed_ids_without_documents = [
    source_seed_id
    for source_seed_id, entry in gold_provenance_registry.items()
    if not entry["document_ids"]
]

print("=== MISSING PROVENANCE IDENTIFIER DISCOVERY ===")
print("Seeds without document IDs:", seed_ids_without_documents)

for source_seed_id in seed_ids_without_documents:
    provenance_entry = gold_provenance_registry[source_seed_id]
    pilot_case_id = provenance_entry["pilot_case_id"]
    review_case = review_case_by_pilot_id[pilot_case_id]

    print(f"\n--- Seed {source_seed_id} ---")
    print("Pilot case ID:", pilot_case_id)
    print("Source evidence entries:")

    for evidence_index, evidence_item in enumerate(
        review_case.get("source_evidence", []),
        start=1,
    ):
        print(f"\nEvidence entry {evidence_index}:")
        print(
            json.dumps(
                evidence_item,
                ensure_ascii=False,
                indent=2,
            )
        )

    identifier_candidates = sorted(
        {
            value
            for evidence_item in review_case.get("source_evidence", [])
            for field_name, value in evidence_item.items()
            if field_name in {"document_id", "source_id", "source_path"}
            and isinstance(value, str)
            and value
        }
    )

    print("\nRegistered identifier candidates:")
    print(identifier_candidates)

    for candidate_name, document_to_chunk_ids in (
        candidate_document_to_chunk_ids.items()
    ):
        matched_identifiers = [
            identifier
            for identifier in identifier_candidates
            if identifier in document_to_chunk_ids
        ]

        print(
            f"- {candidate_name}: "
            f"matched identifiers={matched_identifiers}"
        )

print("\nMISSING PROVENANCE IDENTIFIER DISCOVERY: PASS")

=== MISSING PROVENANCE IDENTIFIER DISCOVERY ===
Seeds without document IDs: [16]

--- Seed 16 ---
Pilot case ID: question-pilot:dc845f69631a7000938b161802bc3030
Source evidence entries:

Evidence entry 1:
{
  "char_end": 29,
  "char_start": 0,
  "human_attribution_overlay": {
    "basis": "EXPLICIT_NARRATIVE_CUE",
    "effective_canonical": null,
    "effective_label": "یکی از حاضران",
    "effective_primary": false,
    "raw_source_preserved": true,
    "status": "NON_PRIMARY_CONTEXT"
  },
  "locator": {
    "locator_origin": "RAW_SOURCE_VERIFIED_CELL79",
    "source_line_end": 225,
    "source_line_start": 225,
    "source_path": "data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/میزان.md"
  },
  "role": "ATTRIBUTION_CONTRAST_LEFT",
  "source_path": "data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/میزان.md",
  "source_type": "transcript",
  "speaker_state": {
    "base_raw": "یکی از حاضران",
    "base_status": "NARRATIVE_CUE",
    "effective_canonical": null,


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۰. جست‌وجوی دقیق مسیر transcript در metadata چانک‌ها

برای seed 16، مسیر فایل transcript و بازهٔ خطوط منشأ ثبت شده است، اما `document_id` ثبت نشده است.

در این مرحله همین مسیر به‌صورت دقیق در رکوردهای چانک جست‌وجو می‌شود تا شناسهٔ سند متناظر، در صورت وجود، از metadata واقعی بازیابی شود.

</div>

In [45]:
# Code Cell 32

seed_id_for_path_recovery = 16

seed16_review_case = review_case_by_pilot_id[
    gold_provenance_registry[seed_id_for_path_recovery]["pilot_case_id"]
]

seed16_source_paths = sorted(
    {
        evidence_item.get("source_path")
        for evidence_item in seed16_review_case.get("source_evidence", [])
        if isinstance(evidence_item.get("source_path"), str)
        and evidence_item["source_path"]
    }
)

assert len(seed16_source_paths) == 1, (
    "Expected exactly one registered source path for seed 16. "
    f"Found: {seed16_source_paths}"
)

seed16_source_path = seed16_source_paths[0]
seed16_source_filename = Path(seed16_source_path).name

print("=== TRANSCRIPT PATH-TO-CHUNK METADATA DISCOVERY ===")
print("Seed:", seed_id_for_path_recovery)
print("Registered source path:", seed16_source_path)
print("Registered source filename:", seed16_source_filename)

path_matches_by_candidate = {}

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    matches = []

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)

            serialized_chunk = json.dumps(
                chunk,
                ensure_ascii=False,
                sort_keys=True,
            )

            if (
                seed16_source_path in serialized_chunk
                or seed16_source_filename in serialized_chunk
            ):
                matches.append(
                    {
                        "chunk_id": chunk["chunk_id"],
                        "source_id": chunk.get("source_id"),
                        "source_type": chunk.get("source_type"),
                        "member_document_ids": chunk.get(
                            "member_document_ids",
                            [],
                        ),
                        "locator_start": chunk.get("locator_start"),
                        "locator_end": chunk.get("locator_end"),
                        "lineage": chunk.get("lineage"),
                        "content_preview": chunk.get("content", "")[:500],
                    }
                )

    path_matches_by_candidate[candidate_name] = matches

    print(f"\n{candidate_name}")
    print("Exact path/filename metadata matches:", len(matches))

    for match in matches[:10]:
        print(
            "- "
            f"chunk_id={match['chunk_id']}, "
            f"source_id={match['source_id']}, "
            f"member_document_ids={match['member_document_ids']}"
        )
        print(f"  locator_start={match['locator_start']}")
        print(f"  locator_end={match['locator_end']}")
        print(f"  lineage={match['lineage']}")
        print(f"  content_preview={match['content_preview']!r}")

print("\nTRANSCRIPT PATH-TO-CHUNK METADATA DISCOVERY: PASS")

=== TRANSCRIPT PATH-TO-CHUNK METADATA DISCOVERY ===
Seed: 16
Registered source path: data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/میزان.md
Registered source filename: میزان.md

baseline_phase1_char_safe
Exact path/filename metadata matches: 34
- chunk_id=chunk:baseline_phase1_char_safe:retrieval:e6c33c9783a5357ae6a5dce0, source_id=program:mizan:unit-001, member_document_ids=['program:mizan:unit-001:utterance:00001', 'program:mizan:unit-001:utterance:00002', 'program:mizan:unit-001:utterance:00003', 'program:mizan:unit-001:utterance:00004']
  locator_start={'benchmark_region_char_offset': 0, 'char_offset': 0, 'document_id': 'program:mizan:unit-001:utterance:00001', 'mode': 'SOURCE_DOCUMENT_CHAR_OFFSET'}
  locator_end={'benchmark_region_char_offset': 1200, 'char_offset': 319, 'document_id': 'program:mizan:unit-001:utterance:00004', 'mode': 'SOURCE_DOCUMENT_CHAR_OFFSET'}
  lineage={'member_lineage': [{'document_id': 'program:mizan:unit-001:utterance:00001', 'lineage': {'pa

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۱. بازیابی چانک‌های مثبت دقیق برای seed 16

مسیر منشأ seed 16 در metadata چانک‌ها پیدا شد، اما کل فایل transcript برای ارزیابی بسیار گسترده است.

در این مرحله فقط چانک‌هایی مثبت محسوب می‌شوند که متن بلندِ شاهد انسانی seed 16 را به‌صورت دقیق و نرمال‌شده در خود دارند.

</div>

In [46]:
# Code Cell 33

seed_id_for_exact_evidence = 16

seed16_review_case = review_case_by_pilot_id[
    gold_provenance_registry[seed_id_for_exact_evidence]["pilot_case_id"]
]

seed16_exact_evidence_texts = [
    canonical_exact_text(evidence_item["text"])
    for evidence_item in seed16_review_case.get("source_evidence", [])
    if isinstance(evidence_item.get("text"), str)
    and len(canonical_exact_text(evidence_item["text"])) >= 100
]

assert seed16_exact_evidence_texts, (
    "No sufficiently long exact evidence text was found for seed 16."
)

print("=== SEED 16 EXACT EVIDENCE-TO-CHUNK DISCOVERY ===")
print("Exact evidence texts tested:", len(seed16_exact_evidence_texts))

seed16_exact_chunk_matches = {}

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    matches = []

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            canonical_chunk_content = canonical_exact_text(
                chunk["content"]
            )

            matched_evidence_indexes = [
                evidence_index
                for evidence_index, evidence_text in enumerate(
                    seed16_exact_evidence_texts,
                    start=1,
                )
                if evidence_text in canonical_chunk_content
            ]

            if matched_evidence_indexes:
                matches.append(
                    {
                        "chunk_id": chunk["chunk_id"],
                        "matched_evidence_indexes": matched_evidence_indexes,
                        "source_id": chunk.get("source_id"),
                        "member_document_ids": chunk.get(
                            "member_document_ids",
                            [],
                        ),
                        "locator_start": chunk.get("locator_start"),
                        "locator_end": chunk.get("locator_end"),
                    }
                )

    seed16_exact_chunk_matches[candidate_name] = matches

    print(f"\n{candidate_name}")
    print("Exact evidence-matching chunks:", len(matches))

    for match in matches:
        print(
            "- "
            f"chunk_id={match['chunk_id']}, "
            f"evidence_indexes={match['matched_evidence_indexes']}, "
            f"source_id={match['source_id']}"
        )
        print(f"  member_document_ids={match['member_document_ids']}")
        print(f"  locator_start={match['locator_start']}")
        print(f"  locator_end={match['locator_end']}")

print("\nSEED 16 EXACT EVIDENCE-TO-CHUNK DISCOVERY: PASS")

=== SEED 16 EXACT EVIDENCE-TO-CHUNK DISCOVERY ===
Exact evidence texts tested: 1

baseline_phase1_char_safe
Exact evidence-matching chunks: 0

parent_child_children
Exact evidence-matching chunks: 0

structure_aware_medium
Exact evidence-matching chunks: 1
- chunk_id=chunk:structure_aware_medium:retrieval:b29c9b340d024d892f01bfb8, evidence_indexes=[1], source_id=program:mizan:unit-001
  member_document_ids=['program:mizan:unit-001:utterance:00104', 'program:mizan:unit-001:utterance:00105', 'program:mizan:unit-001:utterance:00106', 'program:mizan:unit-001:utterance:00107', 'program:mizan:unit-001:utterance:00108', 'program:mizan:unit-001:utterance:00109', 'program:mizan:unit-001:utterance:00110', 'program:mizan:unit-001:utterance:00111', 'program:mizan:unit-001:utterance:00112', 'program:mizan:unit-001:utterance:00113', 'program:mizan:unit-001:utterance:00114', 'program:mizan:unit-001:utterance:00115', 'program:mizan:unit-001:utterance:00116', 'program:mizan:unit-001:utterance:00117', '

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۲. ساخت رجیستری نهایی ارزیابی Retrieval

برای ۲۷ seed، چانک‌های مثبت از شناسهٔ سندهای ثبت‌شده ساخته شده‌اند.

برای seed 16، فقط در روش `structure_aware_medium` یک چانک مثبت با تطابق دقیق شاهد انسانی وجود دارد.  
بنابراین هر روش فقط با seedهایی ارزیابی می‌شود که برای همان روش، چانک مثبت معتبر دارند.

</div>

In [47]:
# Code Cell 34

retrieval_evaluation_registry = {}

for candidate_name in retrieval_chunking_candidates:
    seed_labels = {}

    for source_seed_id in active_seed_ids:
        provenance_entry = gold_provenance_registry[source_seed_id]

        if provenance_entry["document_ids"]:
            coverage_result = document_coverage_by_candidate[
                candidate_name
            ][source_seed_id]

            positive_chunk_ids = coverage_result["positive_chunk_ids"]
            eligible = (
                coverage_result["status"] == "FULL_DOCUMENT_COVERAGE"
                and bool(positive_chunk_ids)
            )
            label_basis = "REGISTERED_DOCUMENT_ID"

        else:
            exact_matches = seed16_exact_chunk_matches[candidate_name]
            positive_chunk_ids = {
                match["chunk_id"]
                for match in exact_matches
            }
            eligible = bool(positive_chunk_ids)
            label_basis = "EXACT_HUMAN_EVIDENCE"

        seed_labels[source_seed_id] = {
            "eligible": eligible,
            "positive_chunk_ids": positive_chunk_ids,
            "label_basis": label_basis,
        }

    retrieval_evaluation_registry[candidate_name] = seed_labels

print("=== FINAL RETRIEVAL EVALUATION REGISTRY ===")
print("Active Golden Dataset records:", golden_dataset_context["record_count"])
print("Active Golden Dataset hash:", golden_dataset_context["sha256"])

for candidate_name, seed_labels in retrieval_evaluation_registry.items():
    eligible_seed_ids = sorted(
        source_seed_id
        for source_seed_id, label in seed_labels.items()
        if label["eligible"]
    )

    excluded_seed_ids = sorted(
        source_seed_id
        for source_seed_id, label in seed_labels.items()
        if not label["eligible"]
    )

    exact_evidence_seed_ids = sorted(
        source_seed_id
        for source_seed_id in eligible_seed_ids
        if seed_labels[source_seed_id]["label_basis"]
        == "EXACT_HUMAN_EVIDENCE"
    )

    positive_chunk_total = sum(
        len(seed_labels[source_seed_id]["positive_chunk_ids"])
        for source_seed_id in eligible_seed_ids
    )

    print(f"\n{candidate_name}")
    print("- Eligible seeds:", len(eligible_seed_ids))
    print("- Excluded seeds:", excluded_seed_ids)
    print("- Seeds labelled by exact human evidence:", exact_evidence_seed_ids)
    print("- Total positive chunk labels:", positive_chunk_total)

    assert eligible_seed_ids, (
        f"{candidate_name} has no eligible evaluation seeds."
    )

assert (
    retrieval_evaluation_registry["structure_aware_medium"][16]["eligible"]
), "Seed 16 must be eligible for structure_aware_medium."

for candidate_name in [
    "baseline_phase1_char_safe",
    "parent_child_children",
    "structure_aware_small",
]:
    assert not retrieval_evaluation_registry[candidate_name][16]["eligible"], (
        f"Seed 16 must remain excluded for {candidate_name}."
    )

print("\nFINAL RETRIEVAL EVALUATION REGISTRY: PASS")

=== FINAL RETRIEVAL EVALUATION REGISTRY ===
Active Golden Dataset records: 573
Active Golden Dataset hash: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729

baseline_phase1_char_safe
- Eligible seeds: 27
- Excluded seeds: [16]
- Seeds labelled by exact human evidence: []
- Total positive chunk labels: 76

parent_child_children
- Eligible seeds: 27
- Excluded seeds: [16]
- Seeds labelled by exact human evidence: []
- Total positive chunk labels: 53

structure_aware_medium
- Eligible seeds: 28
- Excluded seeds: []
- Seeds labelled by exact human evidence: [16]
- Total positive chunk labels: 40

structure_aware_small
- Eligible seeds: 27
- Excluded seeds: [16]
- Seeds labelled by exact human evidence: []
- Total positive chunk labels: 52

FINAL RETRIEVAL EVALUATION REGISTRY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۳. بررسی محیط اجرای Embedding

رجیستری ارزیابی retrieval آماده است.

در این مرحله فقط محیط اجرای مدل‌های embedding بررسی می‌شود: کتابخانه‌های نصب‌شده، نسخه‌ها و دسترسی سخت‌افزاری.  
هیچ مدل embedding دانلود یا اجرا نمی‌شود.

</div>

In [48]:
# Code Cell 35

import importlib.util
import sys

embedding_runtime_packages = [
    "numpy",
    "torch",
    "transformers",
    "sentence_transformers",
    "faiss",
    "sklearn",
]

print("=== EMBEDDING RUNTIME DISCOVERY ===")
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

available_packages = {}

for package_name in embedding_runtime_packages:
    package_spec = importlib.util.find_spec(package_name)
    available = package_spec is not None
    available_packages[package_name] = available

    if available:
        package = __import__(package_name)
        package_version = getattr(package, "__version__", "version unavailable")
        print(f"- {package_name}: AVAILABLE ({package_version})")
    else:
        print(f"- {package_name}: NOT AVAILABLE")

if available_packages["torch"]:
    import torch

    print("\nPyTorch runtime:")
    print("- CUDA available:", torch.cuda.is_available())
    print(
        "- MPS available:",
        bool(
            hasattr(torch.backends, "mps")
            and torch.backends.mps.is_available()
        ),
    )

    if torch.cuda.is_available():
        print("- CUDA device:", torch.cuda.get_device_name(0))

assert available_packages["numpy"], (
    "numpy is required for embedding benchmark preparation."
)

print("\nEMBEDDING RUNTIME DISCOVERY: PASS")

=== EMBEDDING RUNTIME DISCOVERY ===
Python version: 3.11.3
Platform: macOS-12.7.6-x86_64-i386-64bit
- numpy: AVAILABLE (1.24.3)
- torch: AVAILABLE (2.2.2)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- transformers: AVAILABLE (4.53.2)
- sentence_transformers: AVAILABLE (5.0.0)
- faiss: NOT AVAILABLE
- sklearn: AVAILABLE (1.6.1)

PyTorch runtime:
- CUDA available: False
- MPS available: False

EMBEDDING RUNTIME DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۴. کشف مدل‌های embedding موجود در cache محلی

محیط لازم برای اجرای embedding فراهم است و اجرای benchmark روی CPU انجام خواهد شد.

در این مرحله فقط مدل‌های موجود در cache محلی بررسی می‌شوند تا پیش از هر دانلود یا انتخاب مدل، وضعیت واقعی دستگاه مشخص باشد.

</div>

In [49]:
# Code Cell 36

huggingface_cache_roots = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path.home() / "Library" / "Caches" / "huggingface" / "hub",
]

print("=== LOCAL EMBEDDING MODEL CACHE DISCOVERY ===")

local_model_snapshots = []

for cache_root in huggingface_cache_roots:
    print("\nCache root:", cache_root)
    print("Exists:", cache_root.is_dir())

    if not cache_root.is_dir():
        continue

    for model_directory in sorted(cache_root.glob("models--*")):
        snapshot_directory = model_directory / "snapshots"

        if not snapshot_directory.is_dir():
            continue

        for snapshot_path in sorted(snapshot_directory.iterdir()):
            if snapshot_path.is_dir():
                local_model_snapshots.append(
                    {
                        "model_cache_name": model_directory.name,
                        "snapshot_id": snapshot_path.name,
                        "path": snapshot_path,
                        "has_config": (snapshot_path / "config.json").is_file(),
                        "has_sentence_transformers_config": (
                            snapshot_path / "modules.json"
                        ).is_file(),
                    }
                )

print("\nLocal model snapshots found:", len(local_model_snapshots))

for model_snapshot in local_model_snapshots:
    print(
        "- "
        f"{model_snapshot['model_cache_name']} | "
        f"snapshot={model_snapshot['snapshot_id']} | "
        f"config={model_snapshot['has_config']} | "
        f"sentence_transformers={model_snapshot['has_sentence_transformers_config']}"
    )

print("\nLOCAL EMBEDDING MODEL CACHE DISCOVERY: PASS")

=== LOCAL EMBEDDING MODEL CACHE DISCOVERY ===

Cache root: /Users/macbookpro/.cache/huggingface/hub
Exists: True

Cache root: /Users/macbookpro/Library/Caches/huggingface/hub
Exists: False

Local model snapshots found: 3
- models--intfloat--multilingual-e5-small | snapshot=614241f622f53c4eeff9890bdc4f31cfecc418b3 | config=True | sentence_transformers=True
- models--intfloat--multilingual-e5-small | snapshot=c007d7ef6fd86656326059b28395a7a03a7c5846 | config=True | sentence_transformers=True
- models--sentence-transformers--all-MiniLM-L6-v2 | snapshot=c9745ed1d9f207416be6d2e6f8de32d1f16199bf | config=True | sentence_transformers=True

LOCAL EMBEDDING MODEL CACHE DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۵. اعتبارسنجی مدل‌های embedding محلی

دو مدل محلی برای benchmark در دسترس هستند.

در این مرحله هر مدل فقط با چند متن کوتاه بارگذاری و اعتبارسنجی می‌شود: بُعد embedding، شکل خروجی و نبود مقدار نامعتبر.  
هیچ جست‌وجو یا معیار retrieval محاسبه نمی‌شود.

</div>

In [51]:
# Code Cell 37

import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model_registry = {
    "multilingual_e5_small": {
        "model_id": "intfloat/multilingual-e5-small",
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
    },
    "all_minilm_l6_v2": {
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "query_prefix": "",
        "passage_prefix": "",
    },
}

embedding_models = {}

sanity_query = "چگونه می‌توان هزینه‌های غیرضروری را مدیریت کرد؟"
sanity_passages = [
    "مدیریت خرج یعنی هزینه کردن برای نیازها و هدف‌های واقعی.",
    "بازار سهام محل معامله اوراق بهادار است.",
]

print("=== LOCAL EMBEDDING MODEL VALIDATION ===")

for model_name, model_config in embedding_model_registry.items():
    print(f"\nLoading: {model_name}")
    print("Model ID:", model_config["model_id"])

    model = SentenceTransformer(
        model_config["model_id"],
        local_files_only=True,
        device="cpu",
    )

    encoded_query = model.encode(
        model_config["query_prefix"] + sanity_query,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    encoded_passages = model.encode(
        [
            model_config["passage_prefix"] + passage
            for passage in sanity_passages
        ],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    assert encoded_query.ndim == 1, (
        f"{model_name} query embedding must be one-dimensional."
    )
    assert encoded_passages.ndim == 2, (
        f"{model_name} passage embeddings must be two-dimensional."
    )
    assert encoded_passages.shape[0] == len(sanity_passages), (
        f"{model_name} returned an unexpected passage count."
    )
    assert encoded_query.shape[0] == encoded_passages.shape[1], (
        f"{model_name} query and passage dimensions do not match."
    )
    assert not np.isnan(encoded_query).any(), (
        f"{model_name} query embedding contains NaN."
    )
    assert not np.isnan(encoded_passages).any(), (
        f"{model_name} passage embeddings contain NaN."
    )

    embedding_models[model_name] = model

    print("Embedding dimension:", encoded_query.shape[0])
    print("Passage embedding shape:", encoded_passages.shape)

print("\nLOCAL EMBEDDING MODEL VALIDATION: PASS")

=== LOCAL EMBEDDING MODEL VALIDATION ===

Loading: multilingual_e5_small
Model ID: intfloat/multilingual-e5-small
Embedding dimension: 384
Passage embedding shape: (2, 384)

Loading: all_minilm_l6_v2
Model ID: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Passage embedding shape: (2, 384)

LOCAL EMBEDDING MODEL VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۶. بررسی امن تنظیمات Embedding اروان

مدل‌های محلی با موفقیت اعتبارسنجی شدند.

در این مرحله فقط وجود تنظیمات محلی اروان بررسی می‌شود: کلید API، آدرس Gateway و کتابخانهٔ لازم.  
هیچ مقدار محرمانه چاپ نمی‌شود و هیچ درخواست شبکه‌ای ارسال نخواهد شد.

</div>

In [53]:
# Code Cell 38

import os
from dotenv import load_dotenv
from urllib.parse import urlparse

arvan_env_path = project_root / ".env"

print("=== ARVAN EMBEDDING CONFIGURATION PREFLIGHT ===")
print("Environment file exists:", arvan_env_path.is_file())

if arvan_env_path.is_file():
    load_dotenv(arvan_env_path, override=False)

arvan_api_key = os.getenv("ARVAN_AI_API_KEY")
arvan_embedding_url = os.getenv("ARVAN_EMBEDDING_URL")

openai_package_available = importlib.util.find_spec("openai") is not None

print("OpenAI-compatible client available:", openai_package_available)
print("Arvan API key available:", bool(arvan_api_key))
print("Arvan embedding gateway available:", bool(arvan_embedding_url))

if arvan_embedding_url:
    parsed_gateway_url = urlparse(arvan_embedding_url)
    print("Gateway scheme:", parsed_gateway_url.scheme)
    print("Gateway host:", parsed_gateway_url.netloc)
    print("Gateway path configured:", bool(parsed_gateway_url.path))

assert openai_package_available, (
    "The openai package is required for the Arvan-compatible client."
)
assert arvan_api_key, (
    "ARVAN_AI_API_KEY is not available in the current environment."
)
assert arvan_embedding_url, (
    "ARVAN_EMBEDDING_URL is not available in the current environment."
)

print("\nARVAN EMBEDDING CONFIGURATION PREFLIGHT: PASS")

=== ARVAN EMBEDDING CONFIGURATION PREFLIGHT ===
Environment file exists: True
OpenAI-compatible client available: True
Arvan API key available: True
Arvan embedding gateway available: True
Gateway scheme: https
Gateway host: arvancloudai.ir
Gateway path configured: True

ARVAN EMBEDDING CONFIGURATION PREFLIGHT: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۷. اعتبارسنجی اتصال واقعی Embedding اروان

تنظیمات محلی اروان با موفقیت بررسی شد.

در این مرحله یک درخواست کوتاه به مدل قبلی ارسال می‌شود تا اتصال واقعی، شکل خروجی و بُعد embedding تأیید شود.  
هنوز هیچ چانک یا پرسش Golden Dataset به‌صورت انبوه ارسال نمی‌شود.

</div>

In [54]:
# Code Cell 39

from openai import OpenAI

arvan_embedding_model_id = "text-embedding-3-large"

arvan_embedding_client = OpenAI(
    api_key=arvan_api_key,
    base_url=arvan_embedding_url,
    timeout=30.0,
    max_retries=0,
)

print("=== ARVAN EMBEDDING LIVE CONNECTION VALIDATION ===")
print("Model:", arvan_embedding_model_id)

arvan_test_response = arvan_embedding_client.embeddings.create(
    model=arvan_embedding_model_id,
    input=sanity_query,
)

arvan_test_embedding = np.asarray(
    arvan_test_response.data[0].embedding,
    dtype=np.float32,
)

assert arvan_test_embedding.ndim == 1, (
    "Arvan embedding response must be one-dimensional."
)
assert arvan_test_embedding.size > 0, (
    "Arvan embedding response is empty."
)
assert not np.isnan(arvan_test_embedding).any(), (
    "Arvan embedding response contains NaN."
)

embedding_model_registry["arvan_text_embedding_3_large"] = {
    "model_id": arvan_embedding_model_id,
    "provider": "arvan",
    "embedding_dimension": int(arvan_test_embedding.size),
}

print("Embedding dimension:", arvan_test_embedding.size)
print("Response items:", len(arvan_test_response.data))
print("\nARVAN EMBEDDING LIVE CONNECTION VALIDATION: PASS")

=== ARVAN EMBEDDING LIVE CONNECTION VALIDATION ===
Model: text-embedding-3-large
Embedding dimension: 3072
Response items: 1

ARVAN EMBEDDING LIVE CONNECTION VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۸. برآورد حجم اجرای Benchmark

سه مدل آمادهٔ ارزیابی هستند: دو مدل محلی و یک مدل اروان.

در این مرحله حجم واقعی دادهٔ benchmark بررسی می‌شود تا تعداد درخواست‌ها و حجم تقریبی ورودی پیش از اجرای انبوه مشخص باشد.  
هیچ embedding جدیدی ساخته یا به سرویس اروان ارسال نمی‌شود.

</div>

In [55]:
# Code Cell 40

benchmark_query_records = [
    {
        "candidate_id": record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "question": canonical_exact_text(record["question"]),
    }
    for record in golden_dataset
]

assert len(benchmark_query_records) == golden_dataset_context["record_count"], (
    "Benchmark query count must match the active Golden Dataset."
)

candidate_workload_summary = {}
all_canonical_chunk_texts = set()

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    canonical_chunk_texts = []

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            canonical_chunk_text = canonical_exact_text(chunk["content"])
            canonical_chunk_texts.append(canonical_chunk_text)
            all_canonical_chunk_texts.add(canonical_chunk_text)

    candidate_workload_summary[candidate_name] = {
        "chunk_count": len(canonical_chunk_texts),
        "unique_chunk_text_count": len(set(canonical_chunk_texts)),
        "character_count": sum(
            len(text)
            for text in canonical_chunk_texts
        ),
    }

unique_query_texts = {
    record["question"]
    for record in benchmark_query_records
}

print("=== EMBEDDING BENCHMARK WORKLOAD DISCOVERY ===")
print("Golden query records:", len(benchmark_query_records))
print("Unique Golden query texts:", len(unique_query_texts))
print(
    "Golden query characters:",
    sum(len(question) for question in unique_query_texts),
)

print("\nChunk workload by candidate:")

for candidate_name, summary in candidate_workload_summary.items():
    print(f"\n{candidate_name}")
    print("- Chunk records:", summary["chunk_count"])
    print("- Unique chunk texts within candidate:", summary["unique_chunk_text_count"])
    print("- Chunk characters:", summary["character_count"])

print("\nCross-candidate unique chunk texts:", len(all_canonical_chunk_texts))
print(
    "Cross-candidate unique chunk characters:",
    sum(len(text) for text in all_canonical_chunk_texts),
)

estimated_remote_inputs = (
    len(unique_query_texts)
    + len(all_canonical_chunk_texts)
)

print("\nEstimated remote embedding inputs with text-level reuse:")
print(estimated_remote_inputs)

print("\nEMBEDDING BENCHMARK WORKLOAD DISCOVERY: PASS")

=== EMBEDDING BENCHMARK WORKLOAD DISCOVERY ===
Golden query records: 573
Unique Golden query texts: 573
Golden query characters: 42435

Chunk workload by candidate:

baseline_phase1_char_safe
- Chunk records: 3488
- Unique chunk texts within candidate: 3476
- Chunk characters: 3644876

parent_child_children
- Chunk records: 2721
- Unique chunk texts within candidate: 2709
- Chunk characters: 3138958

structure_aware_medium
- Chunk records: 1799
- Unique chunk texts within candidate: 1787
- Chunk characters: 3139880

structure_aware_small
- Chunk records: 2526
- Unique chunk texts within candidate: 2514
- Chunk characters: 3139153

Cross-candidate unique chunk texts: 7698
Cross-candidate unique chunk characters: 10064873

Estimated remote embedding inputs with text-level reuse:
8271

EMBEDDING BENCHMARK WORKLOAD DISCOVERY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۲۹. آماده‌سازی cache اجرای Benchmark

Benchmark embedding برای Golden Dataset فعال اجرا خواهد شد.

در این مرحله محل نگهداری خروجی‌ها، شناسهٔ اجرای فعلی و manifest قابل‌ردیابی ساخته می‌شود تا اجرای مدل‌ها قابل ادامه باشد و هر Golden Dataset فعال، خروجی مستقل خود را داشته باشد.

</div>

In [56]:
# Code Cell 41

embedding_benchmark_root = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding_benchmark"
)

active_benchmark_run_directory = (
    embedding_benchmark_root
    / golden_dataset_context["sha256"]
)

active_benchmark_run_directory.mkdir(
    parents=True,
    exist_ok=True,
)

embedding_benchmark_manifest_path = (
    active_benchmark_run_directory
    / "benchmark_manifest.json"
)

embedding_benchmark_manifest = {
    "active_golden_dataset": {
        "path": str(golden_dataset_context["path"]),
        "sha256": golden_dataset_context["sha256"],
        "record_count": golden_dataset_context["record_count"],
    },
    "chunking_candidates": {
        candidate_name: {
            "path": retrieval_chunking_candidates[candidate_name]["path"],
            "records": candidate_chunk_count_by_name[candidate_name],
        }
        for candidate_name in retrieval_chunking_candidates
    },
    "embedding_models": {
        model_name: {
            key: value
            for key, value in model_config.items()
            if key != "provider" or value != "arvan"
        }
        for model_name, model_config in embedding_model_registry.items()
    },
    "workload": {
        "query_records": len(benchmark_query_records),
        "unique_query_texts": len(unique_query_texts),
        "cross_candidate_unique_chunk_texts": len(
            all_canonical_chunk_texts
        ),
    },
}

if embedding_benchmark_manifest_path.is_file():
    existing_manifest = json.loads(
        embedding_benchmark_manifest_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        existing_manifest["active_golden_dataset"]["sha256"]
        == golden_dataset_context["sha256"]
    ), "Existing benchmark manifest belongs to a different active dataset."

else:
    embedding_benchmark_manifest_path.write_text(
        json.dumps(
            embedding_benchmark_manifest,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

print("=== EMBEDDING BENCHMARK CACHE INITIALIZATION ===")
print("Run directory:", active_benchmark_run_directory)
print("Manifest:", embedding_benchmark_manifest_path)
print("Active dataset records:", golden_dataset_context["record_count"])
print("Embedding models registered:", list(embedding_model_registry))
print("\nEMBEDDING BENCHMARK CACHE INITIALIZATION: PASS")

=== EMBEDDING BENCHMARK CACHE INITIALIZATION ===
Run directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
Manifest: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729/benchmark_manifest.json
Active dataset records: 573
Embedding models registered: ['multilingual_e5_small', 'all_minilm_l6_v2', 'arvan_text_embedding_3_large']

EMBEDDING BENCHMARK CACHE INITIALIZATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۰. ساخت کامل embeddingهای اروان با قابلیت ادامه

مدل اروان برای Benchmark انتخاب و اتصال آن تأیید شده است.

در این مرحله همهٔ متن‌های یکتای مورد نیاز Benchmark به‌صورت batch به اروان ارسال می‌شوند.  
هر batch بلافاصله ذخیره می‌شود؛ بنابراین در صورت قطع اجرا، ادامهٔ کار از batchهای باقی‌مانده انجام خواهد شد.

</div>

In [57]:
# Code Cell 42

arvan_embedding_cache_directory = (
    active_benchmark_run_directory
    / "arvan_text_embedding_3_large"
    / "batches"
)

arvan_embedding_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)

arvan_input_text_by_hash = {}

for query_record in benchmark_query_records:
    text = query_record["question"]
    text_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    arvan_input_text_by_hash[text_hash] = text

for chunk_text in all_canonical_chunk_texts:
    text_hash = hashlib.sha256(chunk_text.encode("utf-8")).hexdigest()
    arvan_input_text_by_hash[text_hash] = chunk_text

assert len(arvan_input_text_by_hash) == estimated_remote_inputs, (
    "The Arvan input registry has an unexpected unique-text count."
)

completed_arvan_input_hashes = set()

for batch_path in sorted(
    arvan_embedding_cache_directory.glob("batch_*.npz")
):
    with np.load(batch_path, allow_pickle=False) as batch_data:
        batch_hashes = batch_data["input_hashes"]
        batch_embeddings = batch_data["embeddings"]

        assert batch_embeddings.ndim == 2, (
            f"Invalid embedding matrix in {batch_path.name}."
        )
        assert batch_embeddings.shape[0] == len(batch_hashes), (
            f"Input and embedding counts differ in {batch_path.name}."
        )
        assert batch_embeddings.shape[1] == arvan_test_embedding.size, (
            f"Embedding dimension differs in {batch_path.name}."
        )

        completed_arvan_input_hashes.update(
            str(input_hash)
            for input_hash in batch_hashes
        )

pending_arvan_input_hashes = sorted(
    set(arvan_input_text_by_hash) - completed_arvan_input_hashes
)

max_batch_inputs = 64
max_batch_characters = 100_000

arvan_batches_to_run = []
current_batch_hashes = []
current_batch_character_count = 0

for input_hash in pending_arvan_input_hashes:
    input_text = arvan_input_text_by_hash[input_hash]

    exceeds_input_limit = (
        len(current_batch_hashes) >= max_batch_inputs
    )
    exceeds_character_limit = (
        current_batch_hashes
        and current_batch_character_count + len(input_text)
        > max_batch_characters
    )

    if exceeds_input_limit or exceeds_character_limit:
        arvan_batches_to_run.append(current_batch_hashes)
        current_batch_hashes = []
        current_batch_character_count = 0

    current_batch_hashes.append(input_hash)
    current_batch_character_count += len(input_text)

if current_batch_hashes:
    arvan_batches_to_run.append(current_batch_hashes)

print("=== ARVAN FULL EMBEDDING EXECUTION ===")
print("Unique inputs required:", len(arvan_input_text_by_hash))
print("Inputs already cached:", len(completed_arvan_input_hashes))
print("Inputs pending:", len(pending_arvan_input_hashes))
print("Batches pending:", len(arvan_batches_to_run))
print("Maximum inputs per batch:", max_batch_inputs)
print("Maximum characters per batch:", max_batch_characters)

for batch_number, batch_hashes in enumerate(
    arvan_batches_to_run,
    start=1,
):
    batch_texts = [
        arvan_input_text_by_hash[input_hash]
        for input_hash in batch_hashes
    ]

    try:
        response = arvan_embedding_client.embeddings.create(
            model=arvan_embedding_model_id,
            input=batch_texts,
        )
    except Exception as error:
        raise RuntimeError(
            "Arvan embedding batch failed. "
            f"Completed cache remains available at "
            f"{arvan_embedding_cache_directory}. "
            f"Failed batch: {batch_number}/{len(arvan_batches_to_run)}. "
            f"Original error: {error}"
        ) from error

    response_items = sorted(
        response.data,
        key=lambda item: item.index,
    )

    batch_embeddings = np.asarray(
        [
            response_item.embedding
            for response_item in response_items
        ],
        dtype=np.float32,
    )

    assert batch_embeddings.shape == (
        len(batch_hashes),
        arvan_test_embedding.size,
    ), (
        f"Unexpected Arvan response shape for batch {batch_number}: "
        f"{batch_embeddings.shape}"
    )
    assert not np.isnan(batch_embeddings).any(), (
        f"Arvan response contains NaN in batch {batch_number}."
    )

    batch_output_path = (
        arvan_embedding_cache_directory
        / f"batch_{len(completed_arvan_input_hashes):06d}.npz"
    )

    np.savez_compressed(
        batch_output_path,
        input_hashes=np.asarray(batch_hashes),
        embeddings=batch_embeddings,
    )

    completed_arvan_input_hashes.update(batch_hashes)

    print(
        f"Batch {batch_number}/{len(arvan_batches_to_run)} saved | "
        f"inputs cached: {len(completed_arvan_input_hashes)}"
    )

    time.sleep(0.1)

assert completed_arvan_input_hashes == set(arvan_input_text_by_hash), (
    "Arvan embedding cache is incomplete after execution."
)

arvan_embedding_execution_report = {
    "model_id": arvan_embedding_model_id,
    "embedding_dimension": int(arvan_test_embedding.size),
    "unique_inputs_required": len(arvan_input_text_by_hash),
    "unique_inputs_cached": len(completed_arvan_input_hashes),
    "batch_directory": str(arvan_embedding_cache_directory),
    "status": "PASS",
}

arvan_embedding_execution_report_path = (
    active_benchmark_run_directory
    / "arvan_embedding_execution_report.json"
)

arvan_embedding_execution_report_path.write_text(
    json.dumps(
        arvan_embedding_execution_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nARVAN FULL EMBEDDING EXECUTION: PASS")
print("Report:", arvan_embedding_execution_report_path)

=== ARVAN FULL EMBEDDING EXECUTION ===
Unique inputs required: 8271
Inputs already cached: 0
Inputs pending: 8271
Batches pending: 130
Maximum inputs per batch: 64
Maximum characters per batch: 100000
Batch 1/130 saved | inputs cached: 64
Batch 2/130 saved | inputs cached: 128
Batch 3/130 saved | inputs cached: 192
Batch 4/130 saved | inputs cached: 256
Batch 5/130 saved | inputs cached: 320
Batch 6/130 saved | inputs cached: 384
Batch 7/130 saved | inputs cached: 448
Batch 8/130 saved | inputs cached: 512
Batch 9/130 saved | inputs cached: 576
Batch 10/130 saved | inputs cached: 640
Batch 11/130 saved | inputs cached: 704
Batch 12/130 saved | inputs cached: 768
Batch 13/130 saved | inputs cached: 832
Batch 14/130 saved | inputs cached: 896
Batch 15/130 saved | inputs cached: 960
Batch 16/130 saved | inputs cached: 1024
Batch 17/130 saved | inputs cached: 1088
Batch 18/130 saved | inputs cached: 1152
Batch 19/130 saved | inputs cached: 1216
Batch 20/130 saved | inputs cached: 1280
Batc

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۱. ساخت کامل embeddingهای مدل‌های محلی

Embeddingهای اروان برای همهٔ ورودی‌های Benchmark ساخته و ذخیره شدند.

در این مرحله همان داده‌ها با دو مدل محلی اجرا می‌شوند. برای مدل E5، پیشوندهای استاندارد query و passage به‌صورت جداگانه اعمال می‌شوند.

</div>

In [58]:
# Code Cell 43

local_embedding_input_registry = {}

query_text_by_hash = {
    hashlib.sha256(
        query_record["question"].encode("utf-8")
    ).hexdigest(): query_record["question"]
    for query_record in benchmark_query_records
}

chunk_text_by_hash = {
    hashlib.sha256(
        chunk_text.encode("utf-8")
    ).hexdigest(): chunk_text
    for chunk_text in all_canonical_chunk_texts
}

local_embedding_execution_reports = {}

for model_name, model in embedding_models.items():
    model_config = embedding_model_registry[model_name]

    if model_config["query_prefix"] or model_config["passage_prefix"]:
        input_text_by_key = {
            **{
                f"query:{text_hash}": (
                    model_config["query_prefix"] + text
                )
                for text_hash, text in query_text_by_hash.items()
            },
            **{
                f"passage:{text_hash}": (
                    model_config["passage_prefix"] + text
                )
                for text_hash, text in chunk_text_by_hash.items()
            },
        }
    else:
        input_text_by_key = {
            f"text:{text_hash}": text
            for text_hash, text in {
                **query_text_by_hash,
                **chunk_text_by_hash,
            }.items()
        }

    local_embedding_input_registry[model_name] = input_text_by_key

    model_cache_directory = (
        active_benchmark_run_directory
        / model_name
        / "batches"
    )

    model_cache_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    completed_input_keys = set()

    for batch_path in sorted(model_cache_directory.glob("batch_*.npz")):
        with np.load(batch_path, allow_pickle=False) as batch_data:
            batch_input_keys = batch_data["input_keys"]
            batch_embeddings = batch_data["embeddings"]

            assert batch_embeddings.ndim == 2, (
                f"Invalid embedding matrix in {batch_path.name}."
            )
            assert batch_embeddings.shape[0] == len(batch_input_keys), (
                f"Input and embedding counts differ in {batch_path.name}."
            )

            completed_input_keys.update(
                str(input_key)
                for input_key in batch_input_keys
            )

    pending_input_keys = sorted(
        set(input_text_by_key) - completed_input_keys
    )

    batch_size = 128
    input_key_batches = [
        pending_input_keys[index:index + batch_size]
        for index in range(0, len(pending_input_keys), batch_size)
    ]

    print(f"\n=== LOCAL FULL EMBEDDING: {model_name} ===")
    print("Inputs required:", len(input_text_by_key))
    print("Inputs already cached:", len(completed_input_keys))
    print("Inputs pending:", len(pending_input_keys))
    print("Batches pending:", len(input_key_batches))

    for batch_number, input_key_batch in enumerate(
        input_key_batches,
        start=1,
    ):
        input_text_batch = [
            input_text_by_key[input_key]
            for input_key in input_key_batch
        ]

        batch_embeddings = model.encode(
            input_text_batch,
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        assert batch_embeddings.shape[0] == len(input_key_batch), (
            f"Unexpected result count for {model_name}, "
            f"batch {batch_number}."
        )
        assert not np.isnan(batch_embeddings).any(), (
            f"{model_name} returned NaN in batch {batch_number}."
        )

        batch_output_path = (
            model_cache_directory
            / f"batch_{len(completed_input_keys):06d}.npz"
        )

        np.savez_compressed(
            batch_output_path,
            input_keys=np.asarray(input_key_batch),
            embeddings=batch_embeddings,
        )

        completed_input_keys.update(input_key_batch)

        if (
            batch_number == 1
            or batch_number == len(input_key_batches)
            or batch_number % 10 == 0
        ):
            print(
                f"Batch {batch_number}/{len(input_key_batches)} saved | "
                f"inputs cached: {len(completed_input_keys)}"
            )

    assert completed_input_keys == set(input_text_by_key), (
        f"{model_name} embedding cache is incomplete."
    )

    local_embedding_execution_reports[model_name] = {
        "model_id": model_config["model_id"],
        "embedding_dimension": model.get_sentence_embedding_dimension(),
        "unique_inputs_required": len(input_text_by_key),
        "unique_inputs_cached": len(completed_input_keys),
        "batch_directory": str(model_cache_directory),
        "status": "PASS",
    }

    report_path = (
        active_benchmark_run_directory
        / f"{model_name}_embedding_execution_report.json"
    )

    report_path.write_text(
        json.dumps(
            local_embedding_execution_reports[model_name],
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

print("\nLOCAL FULL EMBEDDING EXECUTION: PASS")


=== LOCAL FULL EMBEDDING: multilingual_e5_small ===
Inputs required: 8271
Inputs already cached: 0
Inputs pending: 8271
Batches pending: 65
Batch 1/65 saved | inputs cached: 128
Batch 10/65 saved | inputs cached: 1280
Batch 20/65 saved | inputs cached: 2560
Batch 30/65 saved | inputs cached: 3840
Batch 40/65 saved | inputs cached: 5120
Batch 50/65 saved | inputs cached: 6400
Batch 60/65 saved | inputs cached: 7680
Batch 65/65 saved | inputs cached: 8271

=== LOCAL FULL EMBEDDING: all_minilm_l6_v2 ===
Inputs required: 8271
Inputs already cached: 0
Inputs pending: 8271
Batches pending: 65
Batch 1/65 saved | inputs cached: 128
Batch 10/65 saved | inputs cached: 1280
Batch 20/65 saved | inputs cached: 2560
Batch 30/65 saved | inputs cached: 3840
Batch 40/65 saved | inputs cached: 5120
Batch 50/65 saved | inputs cached: 6400
Batch 60/65 saved | inputs cached: 7680
Batch 65/65 saved | inputs cached: 8271

LOCAL FULL EMBEDDING EXECUTION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۲. بارگذاری و اعتبارسنجی cacheهای embedding

Embeddingهای اروان و دو مدل محلی با موفقیت ساخته شدند.

در این مرحله cacheهای هر مدل بارگذاری می‌شوند و بررسی می‌شود که برای همهٔ پرسش‌های Golden و همهٔ متن‌های چانک، بردار معتبر وجود دارد.

</div>

In [61]:
# Code Cell 44

def load_embedding_batches(
    batch_directory,
    key_field_name,
):
    vector_by_key = {}

    for batch_path in sorted(batch_directory.glob("batch_*.npz")):
        with np.load(batch_path, allow_pickle=False) as batch_data:
            batch_keys = batch_data[key_field_name]
            batch_embeddings = batch_data["embeddings"]

            assert batch_embeddings.shape[0] == len(batch_keys), (
                f"Invalid cache batch: {batch_path.name}"
            )

            for key, embedding in zip(batch_keys, batch_embeddings):
                key = str(key)

                assert key not in vector_by_key, (
                    f"Duplicate embedding cache key: {key}"
                )

                vector_by_key[key] = embedding.astype(
                    np.float32,
                    copy=False,
                )

    return vector_by_key


arvan_vector_by_text_hash = load_embedding_batches(
    arvan_embedding_cache_directory,
    key_field_name="input_hashes",
)

expected_arvan_hashes = (
    set(query_text_by_hash)
    | set(chunk_text_by_hash)
)

assert set(arvan_vector_by_text_hash) == expected_arvan_hashes, (
    "Arvan cache does not exactly match the current benchmark input registry."
)

benchmark_embeddings_by_model = {
    "arvan_text_embedding_3_large": {
        "query_vectors_by_text_hash": {
            text_hash: arvan_vector_by_text_hash[text_hash]
            for text_hash in query_text_by_hash
        },
        "passage_vectors_by_text_hash": {
            text_hash: arvan_vector_by_text_hash[text_hash]
            for text_hash in chunk_text_by_hash
        },
        "dimension": int(arvan_test_embedding.size),
    },
}

for model_name, input_text_by_key in local_embedding_input_registry.items():
    model_cache_directory = (
        active_benchmark_run_directory
        / model_name
        / "batches"
    )

    vector_by_input_key = load_embedding_batches(
        model_cache_directory,
        key_field_name="input_keys",
    )

    model_config = embedding_model_registry[model_name]

    if model_config["query_prefix"] or model_config["passage_prefix"]:
        query_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"query:{text_hash}"]
            for text_hash in query_text_by_hash
        }
        passage_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"passage:{text_hash}"]
            for text_hash in chunk_text_by_hash
        }
    else:
        query_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"text:{text_hash}"]
            for text_hash in query_text_by_hash
        }
        passage_vectors_by_text_hash = {
            text_hash: vector_by_input_key[f"text:{text_hash}"]
            for text_hash in chunk_text_by_hash
        }

    benchmark_embeddings_by_model[model_name] = {
        "query_vectors_by_text_hash": query_vectors_by_text_hash,
        "passage_vectors_by_text_hash": passage_vectors_by_text_hash,
        "dimension": int(
            next(iter(query_vectors_by_text_hash.values())).shape[0]
        ),
    }

print("=== EMBEDDING CACHE ASSEMBLY VALIDATION ===")

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_vectors = embedding_lookup["query_vectors_by_text_hash"]
    passage_vectors = embedding_lookup["passage_vectors_by_text_hash"]

    assert set(query_vectors) == set(query_text_by_hash), (
        f"{model_name} query cache is incomplete."
    )
    assert set(passage_vectors) == set(chunk_text_by_hash), (
        f"{model_name} passage cache is incomplete."
    )

    print(f"\n{model_name}")
    print("- Embedding dimension:", embedding_lookup["dimension"])
    print("- Query vectors:", len(query_vectors))
    print("- Passage vectors:", len(passage_vectors))

print("\nEMBEDDING CACHE ASSEMBLY VALIDATION: PASS")

=== EMBEDDING CACHE ASSEMBLY VALIDATION ===

arvan_text_embedding_3_large
- Embedding dimension: 3072
- Query vectors: 573
- Passage vectors: 7698

multilingual_e5_small
- Embedding dimension: 384
- Query vectors: 573
- Passage vectors: 7698

all_minilm_l6_v2
- Embedding dimension: 384
- Query vectors: 573
- Passage vectors: 7698

EMBEDDING CACHE ASSEMBLY VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۳. Benchmark واقعی Retrieval

برای هر مدل و هر روش چانکینگ، similarity میان پرسش‌های Golden و چانک‌ها محاسبه می‌شود.

معیار اصلی مقایسه فقط روی seedهای مشترک و دارای label معتبر اجرا می‌شود.  
بنابراین تفاوت تعداد چانک‌های مثبت یا پوشش اضافی یک روش، مقایسهٔ اصلی را ناعادلانه نمی‌کند.

</div>

In [62]:
# Code Cell 45

evaluation_cutoffs = (1, 3, 5, 10)

common_eligible_seed_ids = sorted(
    set.intersection(
        *(
            {
                source_seed_id
                for source_seed_id, label in seed_labels.items()
                if label["eligible"]
            }
            for seed_labels in retrieval_evaluation_registry.values()
        )
    )
)

assert common_eligible_seed_ids == [
    source_seed_id
    for source_seed_id in active_seed_ids
    if source_seed_id != 16
], (
    "The common evaluation seed set is not the expected fully-labelled set."
)

evaluation_query_records = [
    query_record
    for query_record in benchmark_query_records
    if query_record["source_seed_id"] in common_eligible_seed_ids
]

assert evaluation_query_records, "No eligible Golden queries were found."

def l2_normalize_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    assert np.all(norms > 0), "Cannot normalize zero-length embeddings."
    return matrix / norms


candidate_chunk_registry = {}

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    chunk_ids = []
    chunk_text_hashes = []

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)
            chunk_ids.append(chunk["chunk_id"])
            chunk_text_hashes.append(
                hashlib.sha256(
                    canonical_exact_text(chunk["content"]).encode("utf-8")
                ).hexdigest()
            )

    chunk_index_by_id = {
        chunk_id: index
        for index, chunk_id in enumerate(chunk_ids)
    }

    positive_chunk_indexes_by_seed = {}

    for source_seed_id in common_eligible_seed_ids:
        positive_chunk_ids = retrieval_evaluation_registry[
            candidate_name
        ][source_seed_id]["positive_chunk_ids"]

        positive_indexes = sorted(
            chunk_index_by_id[chunk_id]
            for chunk_id in positive_chunk_ids
            if chunk_id in chunk_index_by_id
        )

        assert positive_indexes, (
            f"{candidate_name} has no indexed positive chunk for "
            f"seed {source_seed_id}."
        )

        positive_chunk_indexes_by_seed[source_seed_id] = set(
            positive_indexes
        )

    candidate_chunk_registry[candidate_name] = {
        "chunk_ids": chunk_ids,
        "chunk_text_hashes": chunk_text_hashes,
        "positive_chunk_indexes_by_seed": positive_chunk_indexes_by_seed,
    }

print("=== FULL RETRIEVAL BENCHMARK ===")
print("Common eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(evaluation_query_records))
print("Evaluation cutoffs:", evaluation_cutoffs)

embedding_benchmark_results = {}

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    embedding_benchmark_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][
                    text_hash
                ]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        first_positive_ranks = []
        retrieval_batch_size = 16

        started_at = time.perf_counter()

        for batch_start in range(
            0,
            len(evaluation_query_records),
            retrieval_batch_size,
        ):
            batch_end = min(
                batch_start + retrieval_batch_size,
                len(evaluation_query_records),
            )

            similarity_matrix = (
                query_matrix[batch_start:batch_end]
                @ passage_matrix.T
            )

            for local_index, query_record in enumerate(
                evaluation_query_records[batch_start:batch_end]
            ):
                ranked_chunk_indexes = np.argsort(
                    similarity_matrix[local_index]
                )[::-1]

                positive_chunk_indexes = candidate_data[
                    "positive_chunk_indexes_by_seed"
                ][query_record["source_seed_id"]]

                first_positive_rank = next(
                    rank
                    for rank, chunk_index in enumerate(
                        ranked_chunk_indexes,
                        start=1,
                    )
                    if chunk_index in positive_chunk_indexes
                )

                first_positive_ranks.append(first_positive_rank)

        elapsed_seconds = time.perf_counter() - started_at

        first_positive_ranks = np.asarray(
            first_positive_ranks,
            dtype=np.int32,
        )

        metrics = {
            f"recall_at_{cutoff}": float(
                np.mean(first_positive_ranks <= cutoff)
            )
            for cutoff in evaluation_cutoffs
        }

        metrics.update(
            {
                "mrr": float(np.mean(1.0 / first_positive_ranks)),
                "mean_first_positive_rank": float(
                    np.mean(first_positive_ranks)
                ),
                "median_first_positive_rank": float(
                    np.median(first_positive_ranks)
                ),
                "query_count": int(len(first_positive_ranks)),
                "candidate_chunk_count": int(
                    len(candidate_data["chunk_ids"])
                ),
                "elapsed_seconds": round(elapsed_seconds, 3),
            }
        )

        embedding_benchmark_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print("- Recall@1:", f"{metrics['recall_at_1']:.4f}")
        print("- Recall@3:", f"{metrics['recall_at_3']:.4f}")
        print("- Recall@5:", f"{metrics['recall_at_5']:.4f}")
        print("- Recall@10:", f"{metrics['recall_at_10']:.4f}")
        print("- MRR:", f"{metrics['mrr']:.4f}")
        print(
            "- Mean first-positive rank:",
            f"{metrics['mean_first_positive_rank']:.2f}",
        )
        print("- Seconds:", metrics["elapsed_seconds"])

print("\nFULL RETRIEVAL BENCHMARK: PASS")

=== FULL RETRIEVAL BENCHMARK ===
Common eligible seeds: 27
Eligible Golden queries: 553
Evaluation cutoffs: (1, 3, 5, 10)

arvan_text_embedding_3_large × baseline_phase1_char_safe
- Recall@1: 0.0090
- Recall@3: 0.0163
- Recall@5: 0.0253
- Recall@10: 0.0362
- MRR: 0.0214
- Mean first-positive rank: 949.61
- Seconds: 0.528

arvan_text_embedding_3_large × parent_child_children
- Recall@1: 0.0072
- Recall@3: 0.0163
- Recall@5: 0.0235
- Recall@10: 0.0325
- MRR: 0.0202
- Mean first-positive rank: 809.93
- Seconds: 0.407

arvan_text_embedding_3_large × structure_aware_medium
- Recall@1: 0.0072
- Recall@3: 0.0163
- Recall@5: 0.0181
- Recall@10: 0.0325
- MRR: 0.0191
- Mean first-positive rank: 579.79
- Seconds: 0.257

arvan_text_embedding_3_large × structure_aware_small
- Recall@1: 0.0036
- Recall@3: 0.0127
- Recall@5: 0.0163
- Recall@10: 0.0380
- MRR: 0.0170
- Mean first-positive rank: 773.16
- Seconds: 0.337

multilingual_e5_small × baseline_phase1_char_safe
- Recall@1: 0.0090
- Recall@3: 0.0

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۴. آزمون تشخیصی با متن شاهد Golden

نتیجهٔ retrieval روی سؤال‌های Golden برای انتخاب نهایی مدل کافی نیست.

در این آزمون، به‌جای سؤال تولیدشده، خود متن شاهد Golden به‌عنوان query استفاده می‌شود.  
هدف تشخیص این است که افت معیارها از نوع سؤال‌ها ناشی می‌شود یا از مدل embedding، چانکینگ و labelهای retrieval.

</div>

In [63]:
# Code Cell 46

gold_evidence_text_by_seed = {}

for source_seed_id in common_eligible_seed_ids:
    observed_evidence_texts = {
        canonical_exact_text(record["source_evidence"])
        for record in gold_records_by_seed[source_seed_id]
    }

    assert len(observed_evidence_texts) == 1, (
        f"Seed {source_seed_id} has inconsistent Golden evidence text."
    )

    gold_evidence_text_by_seed[source_seed_id] = next(
        iter(observed_evidence_texts)
    )

evidence_probe_arvan_path = (
    active_benchmark_run_directory
    / "evidence_probe_arvan.npz"
)

if evidence_probe_arvan_path.is_file():
    with np.load(evidence_probe_arvan_path, allow_pickle=False) as probe_data:
        evidence_probe_seed_ids = probe_data["source_seed_ids"].astype(int)
        arvan_evidence_probe_matrix = probe_data["embeddings"].astype(
            np.float32
        )
else:
    evidence_probe_seed_ids = np.asarray(
        common_eligible_seed_ids,
        dtype=np.int32,
    )

    arvan_evidence_probe_response = arvan_embedding_client.embeddings.create(
        model=arvan_embedding_model_id,
        input=[
            gold_evidence_text_by_seed[source_seed_id]
            for source_seed_id in evidence_probe_seed_ids
        ],
    )

    arvan_evidence_probe_matrix = np.asarray(
        [
            item.embedding
            for item in sorted(
                arvan_evidence_probe_response.data,
                key=lambda item: item.index,
            )
        ],
        dtype=np.float32,
    )

    np.savez_compressed(
        evidence_probe_arvan_path,
        source_seed_ids=evidence_probe_seed_ids,
        embeddings=arvan_evidence_probe_matrix,
    )

assert evidence_probe_seed_ids.tolist() == common_eligible_seed_ids, (
    "Evidence-probe seed ordering is inconsistent."
)

evidence_probe_query_matrices = {
    "arvan_text_embedding_3_large": l2_normalize_rows(
        arvan_evidence_probe_matrix
    ),
}

for model_name, model in embedding_models.items():
    model_config = embedding_model_registry[model_name]

    local_probe_matrix = model.encode(
        [
            model_config["query_prefix"]
            + gold_evidence_text_by_seed[source_seed_id]
            for source_seed_id in common_eligible_seed_ids
        ],
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    evidence_probe_query_matrices[model_name] = l2_normalize_rows(
        local_probe_matrix
    )

evidence_probe_results = {}

print("=== GOLD EVIDENCE RETRIEVAL DIAGNOSTIC ===")
print("Diagnostic seeds:", len(common_eligible_seed_ids))

for model_name, probe_query_matrix in evidence_probe_query_matrices.items():
    evidence_probe_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        embedding_lookup = benchmark_embeddings_by_model[model_name]

        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        similarity_matrix = probe_query_matrix @ passage_matrix.T
        first_positive_ranks = []

        for row_index, source_seed_id in enumerate(
            common_eligible_seed_ids
        ):
            ranked_chunk_indexes = np.argsort(
                similarity_matrix[row_index]
            )[::-1]

            positive_chunk_indexes = candidate_data[
                "positive_chunk_indexes_by_seed"
            ][source_seed_id]

            first_positive_ranks.append(
                next(
                    rank
                    for rank, chunk_index in enumerate(
                        ranked_chunk_indexes,
                        start=1,
                    )
                    if chunk_index in positive_chunk_indexes
                )
            )

        first_positive_ranks = np.asarray(
            first_positive_ranks,
            dtype=np.int32,
        )

        metrics = {
            f"recall_at_{cutoff}": float(
                np.mean(first_positive_ranks <= cutoff)
            )
            for cutoff in evaluation_cutoffs
        }

        metrics["mrr"] = float(
            np.mean(1.0 / first_positive_ranks)
        )

        evidence_probe_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print("- Recall@1:", f"{metrics['recall_at_1']:.4f}")
        print("- Recall@3:", f"{metrics['recall_at_3']:.4f}")
        print("- Recall@5:", f"{metrics['recall_at_5']:.4f}")
        print("- Recall@10:", f"{metrics['recall_at_10']:.4f}")
        print("- MRR:", f"{metrics['mrr']:.4f}")

print("\nGOLD EVIDENCE RETRIEVAL DIAGNOSTIC: PASS")

=== GOLD EVIDENCE RETRIEVAL DIAGNOSTIC ===
Diagnostic seeds: 27

arvan_text_embedding_3_large × baseline_phase1_char_safe
- Recall@1: 0.4074
- Recall@3: 0.5556
- Recall@5: 0.6667
- Recall@10: 0.7407
- MRR: 0.5139

arvan_text_embedding_3_large × parent_child_children
- Recall@1: 0.4074
- Recall@3: 0.6296
- Recall@5: 0.7407
- Recall@10: 0.8148
- MRR: 0.5411

arvan_text_embedding_3_large × structure_aware_medium
- Recall@1: 0.3333
- Recall@3: 0.6296
- Recall@5: 0.7407
- Recall@10: 0.7778
- MRR: 0.4960

arvan_text_embedding_3_large × structure_aware_small
- Recall@1: 0.4074
- Recall@3: 0.7037
- Recall@5: 0.7407
- Recall@10: 0.7778
- MRR: 0.5536

multilingual_e5_small × baseline_phase1_char_safe
- Recall@1: 0.4074
- Recall@3: 0.5556
- Recall@5: 0.5926
- Recall@10: 0.6296
- MRR: 0.5057

multilingual_e5_small × parent_child_children
- Recall@1: 0.3704
- Recall@3: 0.5556
- Recall@5: 0.5556
- Recall@10: 0.6296
- MRR: 0.4799

multilingual_e5_small × structure_aware_medium
- Recall@1: 0.2963
- Re

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۵. خواندن معیار فریز‌شدهٔ انتخاب نهایی

Benchmark نشان داد که نتیجهٔ سؤال‌های Golden و نتیجهٔ متن شاهد یکسان نیست.

پیش از هر انتخاب نهایی، معیارهای فریز‌شدهٔ پروژه برای انتخاب مدل و روش چانکینگ خوانده می‌شوند تا تصمیم بر اساس قرارداد پروژه گرفته شود، نه یک آستانهٔ دلخواه.

</div>

In [64]:
# Code Cell 47

selection_requirements_artifact = chunking_freeze_manifest["artifacts"][
    "final_selection_requirements"
]

selection_requirements_path = Path(
    selection_requirements_artifact["path"]
)

if not selection_requirements_path.is_absolute():
    selection_requirements_path = project_root / selection_requirements_path

assert selection_requirements_path.is_file(), (
    "Frozen final-selection requirements artifact is missing: "
    f"{selection_requirements_path}"
)

selection_requirements = json.loads(
    selection_requirements_path.read_text(
        encoding="utf-8"
    )
)

print("=== FROZEN FINAL-SELECTION REQUIREMENTS ===")
print("Path:", selection_requirements_path.relative_to(project_root))
print(
    json.dumps(
        selection_requirements,
        ensure_ascii=False,
        indent=2,
    )
)

print("\nFROZEN FINAL-SELECTION REQUIREMENTS: PASS")

=== FROZEN FINAL-SELECTION REQUIREMENTS ===
Path: reports/phase2/chunking/final_selection_requirements_v1.json
{
  "already_passed": [
    "source_structure_discovery",
    "tokenizer_measurement_freeze",
    "full_corpus_token_calibration",
    "candidate_generation",
    "coverage_integrity",
    "lineage_integrity",
    "locator_integrity",
    "rights_propagation_integrity",
    "hard_max_validation",
    "reconstruction",
    "content_preservation",
    "structural_benchmark",
    "attribution_risk_benchmark",
    "root_cause_analysis"
  ],
  "current_status": "BLOCKED_PENDING_RETRIEVAL_AND_EVIDENCE_EVALUATION",
  "final_selection": "NOT_YET",
  "still_required": [
    "golden_dataset",
    "retrieval_benchmark",
    "citation_evaluation",
    "attribution_correctness_evaluation",
    "context_quality_evaluation",
    "final_tradeoff_decision"
  ]
}

FROZEN FINAL-SELECTION REQUIREMENTS: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۶. ارزیابی Citation و Attribution در خروجی Retrieval

در این مرحله خروجی‌های Top-10 هر مدل و روش چانکینگ بررسی می‌شوند.

دو موضوع سنجیده می‌شود:

- کامل‌بودن قرارداد citation در چانک‌های بازیابی‌شده؛
- انطباق provenance چانک‌های بازیابی‌شده با سندهای معتبر همان پرسش Golden.

این ارزیابی ساختاری و خودکار است؛ ارزیابی انسانی کیفیت پاسخ و context در مرحلهٔ بعد انجام می‌شود.

</div>

In [65]:
# Code Cell 48

citation_evaluation_cutoff = 10

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = Path(candidate["path"])

    if not candidate_path.is_absolute():
        candidate_path = project_root / candidate_path

    chunk_metadata_by_id = {}

    with candidate_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            chunk = json.loads(line)

            chunk_metadata_by_id[chunk["chunk_id"]] = {
                "source_id": chunk.get("source_id"),
                "source_type": chunk.get("source_type"),
                "locator_start": chunk.get("locator_start"),
                "locator_end": chunk.get("locator_end"),
                "attribution_context": chunk.get(
                    "attribution_context"
                ),
            }

    candidate_chunk_registry[candidate_name][
        "chunk_metadata_by_id"
    ] = chunk_metadata_by_id

print("=== CITATION AND ATTRIBUTION EVALUATION ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Top-K cutoff:", citation_evaluation_cutoff)

citation_attribution_evaluation_results = {}

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    citation_attribution_evaluation_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)

        similarity_matrix = query_matrix @ passage_matrix.T

        top_k_contract_passes = []
        top_k_positive_precisions = []
        top_k_has_positive = []
        top_1_is_positive = []

        for row_index, query_record in enumerate(
            evaluation_query_records
        ):
            top_k_indexes_unsorted = np.argpartition(
                similarity_matrix[row_index],
                -citation_evaluation_cutoff,
            )[-citation_evaluation_cutoff:]

            top_k_indexes = top_k_indexes_unsorted[
                np.argsort(
                    similarity_matrix[row_index][top_k_indexes_unsorted]
                )[::-1]
            ]

            top_k_chunk_ids = [
                candidate_data["chunk_ids"][chunk_index]
                for chunk_index in top_k_indexes
            ]

            positive_chunk_ids = retrieval_evaluation_registry[
                candidate_name
            ][query_record["source_seed_id"]][
                "positive_chunk_ids"
            ]

            top_k_positive_flags = [
                chunk_id in positive_chunk_ids
                for chunk_id in top_k_chunk_ids
            ]

            top_k_metadata = [
                candidate_data["chunk_metadata_by_id"][chunk_id]
                for chunk_id in top_k_chunk_ids
            ]

            top_k_contract_passes.append(
                all(
                    isinstance(metadata["source_id"], str)
                    and metadata["source_id"]
                    and isinstance(metadata["source_type"], str)
                    and metadata["source_type"]
                    and isinstance(metadata["locator_start"], dict)
                    and isinstance(metadata["locator_end"], dict)
                    and isinstance(
                        metadata["attribution_context"],
                        dict,
                    )
                    for metadata in top_k_metadata
                )
            )

            top_k_positive_precisions.append(
                np.mean(top_k_positive_flags)
            )
            top_k_has_positive.append(
                any(top_k_positive_flags)
            )
            top_1_is_positive.append(
                top_k_positive_flags[0]
            )

        metrics = {
            "citation_contract_pass_rate_at_10": float(
                np.mean(top_k_contract_passes)
            ),
            "attribution_correct_at_1": float(
                np.mean(top_1_is_positive)
            ),
            "attribution_correct_at_10": float(
                np.mean(top_k_has_positive)
            ),
            "mean_provenance_precision_at_10": float(
                np.mean(top_k_positive_precisions)
            ),
            "query_count": len(evaluation_query_records),
        }

        citation_attribution_evaluation_results[model_name][
            candidate_name
        ] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print(
            "- Citation contract pass@10:",
            f"{metrics['citation_contract_pass_rate_at_10']:.4f}",
        )
        print(
            "- Attribution correct@1:",
            f"{metrics['attribution_correct_at_1']:.4f}",
        )
        print(
            "- Attribution correct@10:",
            f"{metrics['attribution_correct_at_10']:.4f}",
        )
        print(
            "- Mean provenance precision@10:",
            f"{metrics['mean_provenance_precision_at_10']:.4f}",
        )

print("\nCITATION AND ATTRIBUTION EVALUATION: PASS")

=== CITATION AND ATTRIBUTION EVALUATION ===
Queries evaluated: 553
Top-K cutoff: 10

arvan_text_embedding_3_large × baseline_phase1_char_safe
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.0090
- Attribution correct@10: 0.0362
- Mean provenance precision@10: 0.0042

arvan_text_embedding_3_large × parent_child_children
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.0072
- Attribution correct@10: 0.0325
- Mean provenance precision@10: 0.0038

arvan_text_embedding_3_large × structure_aware_medium
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.0072
- Attribution correct@10: 0.0325
- Mean provenance precision@10: 0.0033

arvan_text_embedding_3_large × structure_aware_small
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.0036
- Attribution correct@10: 0.0380
- Mean provenance precision@10: 0.0040

multilingual_e5_small × baseline_phase1_char_safe
- Citation contract pass@10: 1.0000
- Attribution correct@1: 0.0090
- Attribution co

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۷. پروفایل خودکار کیفیت Context

قرارداد citation در همهٔ خروجی‌ها سالم است.

در این مرحله contextهای Top-10 از نظر ساختاری بررسی می‌شوند: حجم متن، تعداد token، تنوع سندهای منبع و میزان تکرار چانک‌ها.  
این مرحله کیفیت ساختاری context را می‌سنجد؛ داوری انسانیِ مفیدبودن پاسخ در مراحل بعد انجام می‌شود.

</div>

In [66]:
# Code Cell 49

context_quality_results = {}

print("=== AUTOMATED CONTEXT QUALITY PROFILE ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Top-K cutoff:", citation_evaluation_cutoff)

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    context_quality_results[model_name] = {}

    for candidate_name, candidate_data in candidate_chunk_registry.items():
        candidate_path = Path(
            retrieval_chunking_candidates[candidate_name]["path"]
        )

        if not candidate_path.is_absolute():
            candidate_path = project_root / candidate_path

        chunk_context_metadata_by_id = {}

        with candidate_path.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue

                chunk = json.loads(line)

                chunk_context_metadata_by_id[chunk["chunk_id"]] = {
                    "source_id": chunk.get("source_id"),
                    "token_count": chunk.get("token_count"),
                    "char_count": chunk.get("char_count"),
                    "content_hash": hashlib.sha256(
                        canonical_exact_text(
                            chunk["content"]
                        ).encode("utf-8")
                    ).hexdigest(),
                }

        passage_matrix = np.vstack(
            [
                embedding_lookup["passage_vectors_by_text_hash"][text_hash]
                for text_hash in candidate_data["chunk_text_hashes"]
            ]
        ).astype(np.float32)

        passage_matrix = l2_normalize_rows(passage_matrix)
        similarity_matrix = query_matrix @ passage_matrix.T

        context_token_totals = []
        context_char_totals = []
        unique_source_counts = []
        unique_content_counts = []

        for row_index in range(len(evaluation_query_records)):
            top_k_indexes_unsorted = np.argpartition(
                similarity_matrix[row_index],
                -citation_evaluation_cutoff,
            )[-citation_evaluation_cutoff:]

            top_k_indexes = top_k_indexes_unsorted[
                np.argsort(
                    similarity_matrix[row_index][top_k_indexes_unsorted]
                )[::-1]
            ]

            top_k_chunk_ids = [
                candidate_data["chunk_ids"][chunk_index]
                for chunk_index in top_k_indexes
            ]

            top_k_metadata = [
                chunk_context_metadata_by_id[chunk_id]
                for chunk_id in top_k_chunk_ids
            ]

            context_token_totals.append(
                sum(
                    metadata["token_count"]
                    for metadata in top_k_metadata
                    if isinstance(metadata["token_count"], int)
                )
            )
            context_char_totals.append(
                sum(
                    metadata["char_count"]
                    for metadata in top_k_metadata
                    if isinstance(metadata["char_count"], int)
                )
            )
            unique_source_counts.append(
                len(
                    {
                        metadata["source_id"]
                        for metadata in top_k_metadata
                        if isinstance(metadata["source_id"], str)
                        and metadata["source_id"]
                    }
                )
            )
            unique_content_counts.append(
                len(
                    {
                        metadata["content_hash"]
                        for metadata in top_k_metadata
                    }
                )
            )

        metrics = {
            "mean_context_tokens_at_10": float(
                np.mean(context_token_totals)
            ),
            "p95_context_tokens_at_10": float(
                np.percentile(context_token_totals, 95)
            ),
            "mean_context_characters_at_10": float(
                np.mean(context_char_totals)
            ),
            "mean_unique_sources_at_10": float(
                np.mean(unique_source_counts)
            ),
            "mean_unique_chunks_at_10": float(
                np.mean(unique_content_counts)
            ),
        }

        context_quality_results[model_name][candidate_name] = metrics

        print(f"\n{model_name} × {candidate_name}")
        print(
            "- Mean context tokens@10:",
            f"{metrics['mean_context_tokens_at_10']:.1f}",
        )
        print(
            "- P95 context tokens@10:",
            f"{metrics['p95_context_tokens_at_10']:.1f}",
        )
        print(
            "- Mean context characters@10:",
            f"{metrics['mean_context_characters_at_10']:.1f}",
        )
        print(
            "- Mean unique sources@10:",
            f"{metrics['mean_unique_sources_at_10']:.2f}",
        )
        print(
            "- Mean unique chunks@10:",
            f"{metrics['mean_unique_chunks_at_10']:.2f}",
        )

print("\nAUTOMATED CONTEXT QUALITY PROFILE: PASS")

=== AUTOMATED CONTEXT QUALITY PROFILE ===
Queries evaluated: 553
Top-K cutoff: 10

arvan_text_embedding_3_large × baseline_phase1_char_safe
- Mean context tokens@10: 3149.4
- P95 context tokens@10: 3445.4
- Mean context characters@10: 11380.7
- Mean unique sources@10: 8.20
- Mean unique chunks@10: 9.98

arvan_text_embedding_3_large × parent_child_children
- Mean context tokens@10: 3301.8
- P95 context tokens@10: 4025.6
- Mean context characters@10: 11935.4
- Mean unique sources@10: 8.47
- Mean unique chunks@10: 9.95

arvan_text_embedding_3_large × structure_aware_medium
- Mean context tokens@10: 5284.7
- P95 context tokens@10: 6477.0
- Mean context characters@10: 19138.5
- Mean unique sources@10: 8.72
- Mean unique chunks@10: 9.89

arvan_text_embedding_3_large × structure_aware_small
- Mean context tokens@10: 3562.9
- P95 context tokens@10: 4240.2
- Mean context characters@10: 12851.3
- Mean unique sources@10: 8.58
- Mean unique chunks@10: 9.94

multilingual_e5_small × baseline_phase1_

<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۸. اعتبارسنجی اتصال Parent–Child

در روش Parent–Child، جست‌وجو روی child انجام می‌شود اما context نهایی باید از parent متناظر ساخته شود.

در این مرحله ارتباط هر child با parent آن بررسی می‌شود تا ارزیابی context این روش بر اساس ساختار واقعی آن انجام شود.

</div>

In [67]:
# Code Cell 50

parent_candidate = chunking_candidates["parent_child_parents"]
parent_candidate_path = Path(parent_candidate["path"])

if not parent_candidate_path.is_absolute():
    parent_candidate_path = project_root / parent_candidate_path

child_candidate = retrieval_chunking_candidates[
    "parent_child_children"
]
child_candidate_path = Path(child_candidate["path"])

if not child_candidate_path.is_absolute():
    child_candidate_path = project_root / child_candidate_path

parent_chunks_by_id = {}

with parent_candidate_path.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        parent_chunk = json.loads(line)
        parent_chunks_by_id[parent_chunk["chunk_id"]] = parent_chunk

child_to_parent_ids = {}
children_without_parent_id = []
children_with_missing_parent = []

with child_candidate_path.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue

        child_chunk = json.loads(line)
        child_chunk_id = child_chunk["chunk_id"]
        parent_chunk_id = child_chunk.get("parent_chunk_id")

        if not isinstance(parent_chunk_id, str) or not parent_chunk_id:
            children_without_parent_id.append(child_chunk_id)
            continue

        child_to_parent_ids[child_chunk_id] = parent_chunk_id

        if parent_chunk_id not in parent_chunks_by_id:
            children_with_missing_parent.append(
                {
                    "child_chunk_id": child_chunk_id,
                    "parent_chunk_id": parent_chunk_id,
                }
            )

print("=== PARENT–CHILD CONTEXT LINKAGE VALIDATION ===")
print("Parent chunks:", len(parent_chunks_by_id))
print("Child chunks:", len(child_to_parent_ids))
print("Children without parent_chunk_id:", len(children_without_parent_id))
print("Children with missing parent record:", len(children_with_missing_parent))

assert not children_without_parent_id, (
    "Some parent-child retrieval chunks do not declare a parent_chunk_id."
)
assert not children_with_missing_parent, (
    "Some parent-child retrieval chunks point to missing parent records."
)

sample_child_chunk_id = next(iter(child_to_parent_ids))
sample_parent_chunk_id = child_to_parent_ids[sample_child_chunk_id]
sample_parent_chunk = parent_chunks_by_id[sample_parent_chunk_id]

print("\nSample linkage:")
print("- Child chunk ID:", sample_child_chunk_id)
print("- Parent chunk ID:", sample_parent_chunk_id)
print("- Parent token count:", sample_parent_chunk["token_count"])
print("- Parent source ID:", sample_parent_chunk["source_id"])
print(
    "- Parent locator start:",
    sample_parent_chunk["locator_start"],
)
print(
    "- Parent locator end:",
    sample_parent_chunk["locator_end"],
)

print("\nPARENT–CHILD CONTEXT LINKAGE VALIDATION: PASS")

=== PARENT–CHILD CONTEXT LINKAGE VALIDATION ===
Parent chunks: 1447
Child chunks: 2721
Children without parent_chunk_id: 0
Children with missing parent record: 0

Sample linkage:
- Child chunk ID: chunk:parent_child:child:e634d6ef46057e1f28b2a03f
- Parent chunk ID: chunk:parent_child:parent:eb013baa5da4520e962baef2
- Parent token count: 361
- Parent source ID: website:008E61CC
- Parent locator start: {'char_offset': 0, 'document_id': 'website:008E61CC'}
- Parent locator end: {'char_offset': 1316, 'document_id': 'website:008E61CC'}

PARENT–CHILD CONTEXT LINKAGE VALIDATION: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۳۹. ارزیابی Context واقعی Parent–Child

در روش Parent–Child، childها برای retrieval استفاده می‌شوند و parentهای متناظر context نهایی را می‌سازند.

در این مرحله حجم، تنوع و قرارداد citation context والدها برای هر مدل اندازه‌گیری می‌شود.

</div>

In [68]:
# Code Cell 51

parent_child_context_quality_results = {}

parent_child_candidate_data = candidate_chunk_registry[
    "parent_child_children"
]

print("=== PARENT–CHILD REAL CONTEXT QUALITY ===")
print("Queries evaluated:", len(evaluation_query_records))
print("Retrieved children per query:", citation_evaluation_cutoff)

for model_name, embedding_lookup in benchmark_embeddings_by_model.items():
    query_matrix = np.vstack(
        [
            embedding_lookup["query_vectors_by_text_hash"][
                hashlib.sha256(
                    query_record["question"].encode("utf-8")
                ).hexdigest()
            ]
            for query_record in evaluation_query_records
        ]
    ).astype(np.float32)

    query_matrix = l2_normalize_rows(query_matrix)

    child_passage_matrix = np.vstack(
        [
            embedding_lookup["passage_vectors_by_text_hash"][text_hash]
            for text_hash in parent_child_candidate_data[
                "chunk_text_hashes"
            ]
        ]
    ).astype(np.float32)

    child_passage_matrix = l2_normalize_rows(
        child_passage_matrix
    )

    similarity_matrix = query_matrix @ child_passage_matrix.T

    parent_context_token_totals = []
    parent_context_char_totals = []
    parent_context_unique_source_counts = []
    parent_context_parent_counts = []
    parent_context_contract_passes = []

    for row_index in range(len(evaluation_query_records)):
        top_child_indexes_unsorted = np.argpartition(
            similarity_matrix[row_index],
            -citation_evaluation_cutoff,
        )[-citation_evaluation_cutoff:]

        top_child_indexes = top_child_indexes_unsorted[
            np.argsort(
                similarity_matrix[row_index][top_child_indexes_unsorted]
            )[::-1]
        ]

        top_child_chunk_ids = [
            parent_child_candidate_data["chunk_ids"][child_index]
            for child_index in top_child_indexes
        ]

        parent_chunk_ids = list(
            dict.fromkeys(
                child_to_parent_ids[child_chunk_id]
                for child_chunk_id in top_child_chunk_ids
            )
        )

        parent_context_chunks = [
            parent_chunks_by_id[parent_chunk_id]
            for parent_chunk_id in parent_chunk_ids
        ]

        parent_context_token_totals.append(
            sum(
                parent_chunk["token_count"]
                for parent_chunk in parent_context_chunks
            )
        )
        parent_context_char_totals.append(
            sum(
                parent_chunk["char_count"]
                for parent_chunk in parent_context_chunks
            )
        )
        parent_context_unique_source_counts.append(
            len(
                {
                    parent_chunk["source_id"]
                    for parent_chunk in parent_context_chunks
                }
            )
        )
        parent_context_parent_counts.append(
            len(parent_context_chunks)
        )
        parent_context_contract_passes.append(
            all(
                isinstance(parent_chunk["source_id"], str)
                and parent_chunk["source_id"]
                and isinstance(parent_chunk["source_type"], str)
                and parent_chunk["source_type"]
                and isinstance(parent_chunk["locator_start"], dict)
                and isinstance(parent_chunk["locator_end"], dict)
                and isinstance(
                    parent_chunk["attribution_context"],
                    dict,
                )
                for parent_chunk in parent_context_chunks
            )
        )

    metrics = {
        "mean_parent_context_tokens": float(
            np.mean(parent_context_token_totals)
        ),
        "p95_parent_context_tokens": float(
            np.percentile(parent_context_token_totals, 95)
        ),
        "mean_parent_context_characters": float(
            np.mean(parent_context_char_totals)
        ),
        "mean_unique_parent_sources": float(
            np.mean(parent_context_unique_source_counts)
        ),
        "mean_unique_parent_chunks": float(
            np.mean(parent_context_parent_counts)
        ),
        "parent_context_citation_contract_pass_rate": float(
            np.mean(parent_context_contract_passes)
        ),
    }

    parent_child_context_quality_results[model_name] = metrics

    print(f"\n{model_name}")
    print(
        "- Mean parent-context tokens:",
        f"{metrics['mean_parent_context_tokens']:.1f}",
    )
    print(
        "- P95 parent-context tokens:",
        f"{metrics['p95_parent_context_tokens']:.1f}",
    )
    print(
        "- Mean parent-context characters:",
        f"{metrics['mean_parent_context_characters']:.1f}",
    )
    print(
        "- Mean unique parent sources:",
        f"{metrics['mean_unique_parent_sources']:.2f}",
    )
    print(
        "- Mean unique parent chunks:",
        f"{metrics['mean_unique_parent_chunks']:.2f}",
    )
    print(
        "- Citation contract pass:",
        f"{metrics['parent_context_citation_contract_pass_rate']:.4f}",
    )

print("\nPARENT–CHILD REAL CONTEXT QUALITY: PASS")

=== PARENT–CHILD REAL CONTEXT QUALITY ===
Queries evaluated: 553
Retrieved children per query: 10

arvan_text_embedding_3_large
- Mean parent-context tokens: 8177.3
- P95 parent-context tokens: 9904.2
- Mean parent-context characters: 29444.7
- Mean unique parent sources: 8.47
- Mean unique parent chunks: 9.60
- Citation contract pass: 1.0000

multilingual_e5_small
- Mean parent-context tokens: 7359.8
- P95 parent-context tokens: 9574.4
- Mean parent-context characters: 27506.2
- Mean unique parent sources: 9.13
- Mean unique parent chunks: 9.63
- Citation contract pass: 1.0000

all_minilm_l6_v2
- Mean parent-context tokens: 4878.0
- P95 parent-context tokens: 6637.0
- Mean parent-context characters: 17591.3
- Mean unique parent sources: 8.94
- Mean unique parent chunks: 10.00
- Citation contract pass: 1.0000

PARENT–CHILD REAL CONTEXT QUALITY: PASS


<div style="direction: rtl; text-align: right; line-height: 1.8;">

## ۴۰. ثبت تصمیم Trade-off و وضعیت آمادگی انتخاب

همهٔ ارزیابی‌های خودکار retrieval، citation، attribution و context اجرا شده‌اند.

در این مرحله نتیجهٔ واقعی ثبت می‌شود.  
اگر یک ترکیب در همهٔ ابعاد برنده نباشد، انتخاب نهایی انجام نمی‌شود و علت آن به‌صورت شفاف ثبت خواهد شد.

</div>

In [69]:
# Code Cell 52

question_retrieval_ranking = sorted(
    [
        {
            "model": model_name,
            "candidate": candidate_name,
            **metrics,
        }
        for model_name, candidate_results
        in embedding_benchmark_results.items()
        for candidate_name, metrics
        in candidate_results.items()
    ],
    key=lambda item: (
        -item["recall_at_10"],
        -item["mrr"],
    ),
)

evidence_retrieval_ranking = sorted(
    [
        {
            "model": model_name,
            "candidate": candidate_name,
            **metrics,
        }
        for model_name, candidate_results
        in evidence_probe_results.items()
        for candidate_name, metrics
        in candidate_results.items()
    ],
    key=lambda item: (
        -item["recall_at_10"],
        -item["mrr"],
    ),
)

best_question_retrieval = question_retrieval_ranking[0]
best_evidence_retrieval = evidence_retrieval_ranking[0]

embedding_benchmark_tradeoff = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "selection_status": "NO_FINAL_SELECTION",
    "decision_basis": {
        "best_question_retrieval": best_question_retrieval,
        "best_evidence_retrieval": best_evidence_retrieval,
        "citation_contract_status": "PASS_FOR_ALL_COMBINATIONS",
        "parent_child_context_contract_status": "PASS_FOR_ALL_MODELS",
    },
    "blocking_findings": [
        (
            "Question-based retrieval is materially weaker than "
            "source-evidence retrieval across the evaluated combinations."
        ),
        (
            "No single model-and-chunking combination leads both "
            "question retrieval and source-evidence retrieval."
        ),
        (
            "Parent-child context requires a context-budget policy "
            "before production use."
        ),
    ],
    "required_next_work": [
        "query_side_retrieval_remediation",
        "context_budget_and_assembly_policy",
        "post_remediation_retrieval_benchmark",
        "final_tradeoff_decision",
    ],
}

tradeoff_report_path = (
    active_benchmark_run_directory
    / "embedding_benchmark_tradeoff.json"
)

tradeoff_report_path.write_text(
    json.dumps(
        embedding_benchmark_tradeoff,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== EMBEDDING BENCHMARK TRADE-OFF DECISION ===")
print("Selection status:", embedding_benchmark_tradeoff["selection_status"])

print("\nBest question-retrieval combination:")
print(
    f"- model={best_question_retrieval['model']}, "
    f"candidate={best_question_retrieval['candidate']}, "
    f"Recall@10={best_question_retrieval['recall_at_10']:.4f}, "
    f"MRR={best_question_retrieval['mrr']:.4f}"
)

print("\nBest source-evidence-retrieval combination:")
print(
    f"- model={best_evidence_retrieval['model']}, "
    f"candidate={best_evidence_retrieval['candidate']}, "
    f"Recall@10={best_evidence_retrieval['recall_at_10']:.4f}, "
    f"MRR={best_evidence_retrieval['mrr']:.4f}"
)

print("\nBlocking findings:")
for finding in embedding_benchmark_tradeoff["blocking_findings"]:
    print("-", finding)

print("\nRequired next work:")
for next_step in embedding_benchmark_tradeoff["required_next_work"]:
    print("-", next_step)

print("\nReport:", tradeoff_report_path)
print("\nEMBEDDING BENCHMARK TRADE-OFF DECISION: PASS")

=== EMBEDDING BENCHMARK TRADE-OFF DECISION ===
Selection status: NO_FINAL_SELECTION

Best question-retrieval combination:
- model=multilingual_e5_small, candidate=baseline_phase1_char_safe, Recall@10=0.0434, MRR=0.0225

Best source-evidence-retrieval combination:
- model=arvan_text_embedding_3_large, candidate=parent_child_children, Recall@10=0.8148, MRR=0.5411

Blocking findings:
- Question-based retrieval is materially weaker than source-evidence retrieval across the evaluated combinations.
- No single model-and-chunking combination leads both question retrieval and source-evidence retrieval.
- Parent-child context requires a context-budget policy before production use.

Required next work:
- query_side_retrieval_remediation
- context_budget_and_assembly_policy
- post_remediation_retrieval_benchmark
- final_tradeoff_decision

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## اصلاح قرارداد ورودی مدل E5

مدل `multilingual-e5-small` برای بازیابی معنایی با دو پیشوند استاندارد طراحی شده است:

- برای پرسش: `query:`
- برای متن بازیابی‌شونده: `passage:`

بنابراین پیش از تغییر Chunking یا انتخاب مدل نهایی، همین مدل را یک‌بار با قرارداد صحیح خودش اجرا می‌کنیم و نتیجه را جدا از اجرای قبلی نگه می‌داریم.

این آزمایش:

- Golden Dataset فعال را از متغیر اصلی می‌خواند؛
- هیچ فایل یا نتیجهٔ قبلی را جایگزین نمی‌کند؛
- خروجی را با هش Golden Dataset فعال ذخیره می‌کند؛
- در اجرای آینده با Golden Dataset جدید، خودکار در پوشهٔ جداگانهٔ همان نسخه اجرا می‌شود.

</div>

In [ ]:
# Code Cell 53
# NB13 — E5 instructional embedding cache
# Uses the active Golden Dataset only; does not overwrite prior embeddings.

from pathlib import Path
import json
import numpy as np

assert "golden_dataset_path" in globals(), (
    "Run the Active Golden Dataset loader cell first."
)
assert "query_text_by_hash" in globals(), (
    "Run the embedding workload-discovery cell first."
)
assert "chunk_text_by_hash" in globals(), (
    "Run the embedding workload-discovery cell first."
)
assert "embedding_models" in globals(), (
    "Run the local embedding-model validation cell first."
)
assert "multilingual_e5_small" in embedding_models, (
    "The multilingual_e5_small model is not loaded."
)

e5_model = embedding_models["multilingual_e5_small"]

project_root = golden_dataset_path.parents[3]
e5_instructional_run_directory = (
    project_root
    / "artifacts"
    / "phase2"
    / "embedding_benchmark"
    / golden_dataset_context["sha256"]
)
e5_instructional_run_directory.mkdir(parents=True, exist_ok=True)

e5_instructional_cache_path = (
    e5_instructional_run_directory
    / "multilingual_e5_small_instructional_embeddings.npz"
)
e5_instructional_manifest_path = (
    e5_instructional_run_directory
    / "multilingual_e5_small_instructional_manifest.json"
)

query_hashes = list(query_text_by_hash.keys())
passage_hashes = list(chunk_text_by_hash.keys())

def encode_e5_instructional(text_by_hash, prefix, batch_size=128):
    hashes = list(text_by_hash.keys())
    prefixed_texts = [f"{prefix}: {text_by_hash[text_hash]}" for text_hash in hashes]

    vectors = e5_model.encode(
        prefixed_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    return hashes, vectors

cache_is_reusable = False

if e5_instructional_cache_path.is_file() and e5_instructional_manifest_path.is_file():
    cached_manifest = json.loads(
        e5_instructional_manifest_path.read_text(encoding="utf-8")
    )

    cache_is_reusable = (
        cached_manifest.get("golden_dataset_sha256")
        == golden_dataset_context["sha256"]
        and cached_manifest.get("query_hashes") == query_hashes
        and cached_manifest.get("passage_hashes") == passage_hashes
        and cached_manifest.get("query_prefix") == "query"
        and cached_manifest.get("passage_prefix") == "passage"
    )

if cache_is_reusable:
    cached_arrays = np.load(e5_instructional_cache_path)

    e5_instructional_query_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            cached_arrays["query_hashes"].tolist(),
            cached_arrays["query_vectors"],
        )
    }
    e5_instructional_passage_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            cached_arrays["passage_hashes"].tolist(),
            cached_arrays["passage_vectors"],
        )
    }

    print("Existing E5 instructional cache loaded.")
else:
    print("Encoding E5 queries with prefix: query:")
    encoded_query_hashes, encoded_query_vectors = encode_e5_instructional(
        query_text_by_hash,
        prefix="query",
    )

    print("\nEncoding E5 passages with prefix: passage:")
    encoded_passage_hashes, encoded_passage_vectors = encode_e5_instructional(
        chunk_text_by_hash,
        prefix="passage",
    )

    np.savez_compressed(
        e5_instructional_cache_path,
        query_hashes=np.array(encoded_query_hashes, dtype=object),
        query_vectors=encoded_query_vectors,
        passage_hashes=np.array(encoded_passage_hashes, dtype=object),
        passage_vectors=encoded_passage_vectors,
    )

    e5_instructional_manifest = {
        "artifact": e5_instructional_cache_path.name,
        "golden_dataset_path": str(golden_dataset_path),
        "golden_dataset_sha256": golden_dataset_context["sha256"],
        "model_name": "intfloat/multilingual-e5-small",
        "query_prefix": "query",
        "passage_prefix": "passage",
        "normalization": "L2",
        "query_count": len(encoded_query_hashes),
        "passage_count": len(encoded_passage_hashes),
        "embedding_dimension": int(encoded_query_vectors.shape[1]),
    }

    e5_instructional_manifest_path.write_text(
        json.dumps(e5_instructional_manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    e5_instructional_query_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            encoded_query_hashes,
            encoded_query_vectors,
        )
    }
    e5_instructional_passage_vectors = {
        text_hash: vector
        for text_hash, vector in zip(
            encoded_passage_hashes,
            encoded_passage_vectors,
        )
    }

e5_instructional_embeddings = {
    "model_name": "multilingual_e5_small_instructional",
    "model_id": "intfloat/multilingual-e5-small",
    "query_prefix": "query:",
    "passage_prefix": "passage:",
    "dimension": 384,
    "query_vectors_by_text_hash": e5_instructional_query_vectors,
    "passage_vectors_by_text_hash": e5_instructional_passage_vectors,
    "cache_path": e5_instructional_cache_path,
}

print("\n=== E5 INSTRUCTIONAL EMBEDDING CACHE: PASS ===")
print("Active Golden Dataset hash:", golden_dataset_context["sha256"])
print("Queries:", len(e5_instructional_query_vectors))
print("Passages:", len(e5_instructional_passage_vectors))
print("Dimension:", e5_instructional_embeddings["dimension"])
print("Cache:", e5_instructional_cache_path)

Encoding E5 queries with prefix: query:


Batches: 100%|██████████| 5/5 [00:06<00:00,  1.20s/it]



Encoding E5 passages with prefix: passage:


Batches: 100%|██████████| 61/61 [29:11<00:00, 28.72s/it]



=== E5 INSTRUCTIONAL EMBEDDING CACHE: PASS ===
Active Golden Dataset hash: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
Queries: 573
Passages: 7698
Dimension: 384
Cache: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729/multilingual_e5_small_instructional_embeddings.npz


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کنترل آماده‌بودن Benchmark اصلاح‌شدهٔ E5

در این مرحله بررسی می‌کنیم که embeddingهای جدید E5 کامل، نرمال‌سازی‌شده و آمادهٔ ورود به همان Benchmark قبلی هستند.

هیچ فایل، نتیجه یا Benchmark قبلی تغییر نمی‌کند.

</div>

In [71]:
# Code Cell 54
# NB13 — E5 instructional embedding validation and benchmark preflight

expected_query_hashes = set(query_text_by_hash)
expected_passage_hashes = set(chunk_text_by_hash)

actual_query_hashes = set(
    e5_instructional_embeddings["query_vectors_by_text_hash"]
)
actual_passage_hashes = set(
    e5_instructional_embeddings["passage_vectors_by_text_hash"]
)

assert actual_query_hashes == expected_query_hashes, (
    "Instructional E5 query cache is incomplete or has unexpected keys."
)
assert actual_passage_hashes == expected_passage_hashes, (
    "Instructional E5 passage cache is incomplete or has unexpected keys."
)

query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][text_hash]
        for text_hash in query_text_by_hash
    ]
)
passage_matrix = np.vstack(
    [
        e5_instructional_embeddings["passage_vectors_by_text_hash"][text_hash]
        for text_hash in chunk_text_by_hash
    ]
)

assert query_matrix.shape == (len(query_text_by_hash), 384)
assert passage_matrix.shape == (len(chunk_text_by_hash), 384)
assert np.isfinite(query_matrix).all()
assert np.isfinite(passage_matrix).all()

assert np.allclose(
    np.linalg.norm(query_matrix, axis=1),
    1.0,
    atol=1e-4,
), "Instructional E5 query vectors are not L2-normalized."

assert np.allclose(
    np.linalg.norm(passage_matrix, axis=1),
    1.0,
    atol=1e-4,
), "Instructional E5 passage vectors are not L2-normalized."

print("=== E5 INSTRUCTIONAL BENCHMARK PREFLIGHT ===")
print("Query vectors:", query_matrix.shape)
print("Passage vectors:", passage_matrix.shape)
print("Query prefix:", e5_instructional_embeddings["query_prefix"])
print("Passage prefix:", e5_instructional_embeddings["passage_prefix"])
print("\nE5 INSTRUCTIONAL BENCHMARK PREFLIGHT: PASS")

=== E5 INSTRUCTIONAL BENCHMARK PREFLIGHT ===
Query vectors: (573, 384)
Passage vectors: (7698, 384)
Query prefix: query:
Passage prefix: passage:

E5 INSTRUCTIONAL BENCHMARK PREFLIGHT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کشف قرارداد Benchmark بازیابی

پیش از اجرای Benchmark جدید، ساختار دقیق برچسب‌های مثبت و پرسش‌های واجد شرایط را از همان اجرای قبلی بررسی می‌کنیم.

هدف این است که E5 اصلاح‌شده دقیقاً با همان داده، همان چهار Chunking و همان معیارهای قبلی ارزیابی شود؛ بنابراین مقایسه کاملاً منصفانه خواهد بود.

</div>

In [72]:
# Code Cell 55
# NB13 — Retrieval benchmark contract discovery

def summarize_benchmark_object(name, value, max_keys=15):
    print(f"\n{name}")
    print("Type:", type(value).__name__)

    if isinstance(value, dict):
        print("Key count:", len(value))
        print("Key preview:", list(value.keys())[:max_keys])

        if value:
            first_key = next(iter(value))
            first_value = value[first_key]
            print("First key:", repr(first_key))
            print("First value type:", type(first_value).__name__)

            if isinstance(first_value, dict):
                print(
                    "First nested-key preview:",
                    list(first_value.keys())[:max_keys],
                )

    elif isinstance(value, (list, tuple, set)):
        print("Item count:", len(value))

        if value:
            first_value = next(iter(value))
            print("First item type:", type(first_value).__name__)

            if isinstance(first_value, dict):
                print(
                    "First-item key preview:",
                    list(first_value.keys())[:max_keys],
                )

    else:
        print("Value preview:", repr(value)[:500])


benchmark_contract_candidates = {
    name: value
    for name, value in globals().items()
    if any(
        token in name.lower()
        for token in (
            "evaluation_registry",
            "retrieval_registry",
            "positive",
            "eligible",
            "benchmark_query",
            "golden_query",
        )
    )
}

print("=== RETRIEVAL BENCHMARK CONTRACT DISCOVERY ===")

print("\nRetrieval candidates:")
for candidate_name, candidate in retrieval_chunking_candidates.items():
    print(
        f"- {candidate_name}: "
        f"records={candidate['records']}, "
        f"path={candidate['path']}"
    )

print("\nRelevant evaluation objects discovered:")
assert benchmark_contract_candidates, (
    "No benchmark-contract object was found. "
    "Run the Final Retrieval Evaluation Registry cell again."
)

for object_name, object_value in benchmark_contract_candidates.items():
    summarize_benchmark_object(object_name, object_value)

print("\nRETRIEVAL BENCHMARK CONTRACT DISCOVERY: PASS")

=== RETRIEVAL BENCHMARK CONTRACT DISCOVERY ===

Retrieval candidates:
- baseline_phase1_char_safe: records=3488, path=data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl
- parent_child_children: records=2721, path=data/phase2/chunks/nb11_benchmark_v1/parent_child_children_v1.jsonl
- structure_aware_medium: records=1799, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_medium_v1.jsonl
- structure_aware_small: records=2526, path=data/phase2/chunks/nb11_benchmark_v1/structure_aware_small_v1.jsonl

Relevant evaluation objects discovered:

positive_chunk_ids
Type: set
Item count: 3
First item type: str

retrieval_evaluation_registry
Type: dict
Key count: 4
Key preview: ['baseline_phase1_char_safe', 'parent_child_children', 'structure_aware_medium', 'structure_aware_small']
First key: 'baseline_phase1_char_safe'
First value type: dict
First nested-key preview: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

eligible
Type: bool
Value preview: True

eligible_s

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## Benchmark بازیابی پرسش با E5 اصلاح‌شده

در این آزمایش، E5 با پیشوندهای استاندارد `query:` و `passage:` روی همان پرسش‌ها، همان برچسب‌های مثبت و همان چهار روش Chunking قبلی ارزیابی می‌شود.

نتیجه مستقیماً با اجرای قبلی `multilingual_e5_small` مقایسه خواهد شد.

</div>

In [74]:
# Code Cell 56
# NB13 — Instructional E5 cache-to-chunk key reconciliation

candidate_content_reconciliation = {}

print("=== INSTRUCTIONAL E5 CACHE-TO-CHUNK RECONCILIATION ===")
print("Cached passage texts:", len(passage_hash_by_text))

for candidate_name, candidate in retrieval_chunking_candidates.items():
    candidate_path = project_root / candidate["path"]

    assert candidate_path.is_file(), (
        f"Missing frozen chunk candidate: {candidate_path}"
    )

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    matched_records = [
        record
        for record in candidate_records
        if record["content"] in passage_hash_by_text
    ]
    unmatched_records = [
        record
        for record in candidate_records
        if record["content"] not in passage_hash_by_text
    ]

    candidate_content_reconciliation[candidate_name] = {
        "records": len(candidate_records),
        "matched_records": len(matched_records),
        "unmatched_records": len(unmatched_records),
        "first_unmatched_record": (
            unmatched_records[0]
            if unmatched_records
            else None
        ),
    }

    print(f"\n{candidate_name}")
    print("Records:", len(candidate_records))
    print("Exact cache-text matches:", len(matched_records))
    print("Unmatched records:", len(unmatched_records))

    if unmatched_records:
        sample = unmatched_records[0]["content"]
        sample_prefix = sample[:180]

        prefix_matches = [
            cached_text
            for cached_text in chunk_text_by_hash.values()
            if cached_text.startswith(sample_prefix)
            or sample.startswith(cached_text[:180])
        ]

        print("\nFirst unmatched-content diagnostic:")
        print("Raw content length:", len(sample))
        print("Raw content preview:", repr(sample[:300]))
        print("Possible cached-text prefix matches:", len(prefix_matches))

        if prefix_matches:
            matched_text = prefix_matches[0]
            print("Cached content length:", len(matched_text))
            print("Cached content preview:", repr(matched_text[:300]))
            print("Texts exactly equal:", sample == matched_text)

print("\nAvailable text-normalization functions:")
normalization_function_names = sorted(
    name
    for name, value in globals().items()
    if callable(value)
    and any(
        token in name.lower()
        for token in (
            "normalize",
            "canonical",
            "clean_text",
            "text_hash",
            "hash_text",
        )
    )
)

for function_name in normalization_function_names:
    print("-", function_name)

print("\nINSTRUCTIONAL E5 CACHE-TO-CHUNK RECONCILIATION: PASS")

=== INSTRUCTIONAL E5 CACHE-TO-CHUNK RECONCILIATION ===
Cached passage texts: 7698

baseline_phase1_char_safe
Records: 3488
Exact cache-text matches: 22
Unmatched records: 3466

First unmatched-content diagnostic:
Raw content length: 1200
Raw content preview: 'دویدم و دویدم،\nسر کوهی رسیدم،\nدو تا خاتونُ دیدم ...\nاین شعر از معروف\u200cترین شعرهای دوران کودکی ماست. ماجرای پسر بچه\u200cای که در مسیری طولانی با افرادی گوناگون مواجه می\u200cشود و از هر کس چیزی می\u200cگیرد و چیزی به آ\u200cن\u200cها می\u200cدهد. چیزهایی که خودش احتیاجی به آن\u200cها ندارد، ولی نیازِ طرف مقابل را برطرف می\u200cکند. احتمالا'
Possible cached-text prefix matches: 0

parent_child_children
Records: 2721
Exact cache-text matches: 33
Unmatched records: 2688

First unmatched-content diagnostic:
Raw content length: 1316
Raw content preview: 'دویدم و دویدم،\nسر کوهی رسیدم،\nدو تا خاتونُ دیدم ...\nاین شعر از معروف\u200cترین شعرهای دوران کودکی ماست. ماجرای پسر بچه\u200cای که در مسیری طولانی با افرادی گوناگون مواجه می\u

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تعیین کلید متنی استاندارد برای Chunkها

فایل‌های Chunk شامل متن خام هستند، اما cache embedding با نسخهٔ استانداردشدهٔ متن ساخته شده است.

در این مرحله تابع استانداردسازی درست را تعیین می‌کنیم تا Benchmark اصلاح‌شده دقیقاً با همان cache معتبر اجرا شود.

</div>

In [76]:
# Code Cell 57
# NB13 — Canonical chunk-text key validation and selection

canonical_key_functions = {
    "canonical_exact_text": canonical_exact_text,
    "canonical_comparison_text": canonical_comparison_text,
}

canonical_key_validation = {}
all_candidate_raw_contents = []

print("=== CANONICAL CHUNK-TEXT KEY VALIDATION ===")

for function_name, canonicalize_text in canonical_key_functions.items():
    print(f"\nFunction: {function_name}")

    function_results = {}

    for candidate_name, candidate in retrieval_chunking_candidates.items():
        candidate_path = project_root / candidate["path"]

        candidate_records = [
            json.loads(line)
            for line in candidate_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

        if function_name == "canonical_exact_text":
            all_candidate_raw_contents.extend(
                record["content"]
                for record in candidate_records
            )

        matched_records = [
            record
            for record in candidate_records
            if canonicalize_text(record["content"]) in passage_hash_by_text
        ]

        function_results[candidate_name] = {
            "records": len(candidate_records),
            "canonical_matches": len(matched_records),
            "unmatched_records": len(candidate_records) - len(matched_records),
        }

        print(
            f"- {candidate_name}: "
            f"{len(matched_records)} / {len(candidate_records)} "
            f"canonical matches"
        )

    canonical_key_validation[function_name] = function_results

fully_matching_functions = [
    function_name
    for function_name, candidate_results in canonical_key_validation.items()
    if all(
        result["canonical_matches"] == result["records"]
        for result in candidate_results.values()
    )
]

assert "canonical_exact_text" in fully_matching_functions, (
    "canonical_exact_text does not fully match the existing embedding cache."
)

canonical_functions_are_equivalent = all(
    canonical_exact_text(content)
    == canonical_comparison_text(content)
    for content in all_candidate_raw_contents
)

active_chunk_cache_key_function_name = "canonical_exact_text"
active_chunk_cache_key_function = canonical_exact_text

print(
    "\nFunctions equivalent for all frozen candidate contents:",
    canonical_functions_are_equivalent,
)
print("Selected cache-key function:", active_chunk_cache_key_function_name)
print("\nCANONICAL CHUNK-TEXT KEY VALIDATION: PASS")

=== CANONICAL CHUNK-TEXT KEY VALIDATION ===

Function: canonical_exact_text
- baseline_phase1_char_safe: 3488 / 3488 canonical matches
- parent_child_children: 2721 / 2721 canonical matches
- structure_aware_medium: 1799 / 1799 canonical matches
- structure_aware_small: 2526 / 2526 canonical matches

Function: canonical_comparison_text
- baseline_phase1_char_safe: 3488 / 3488 canonical matches
- parent_child_children: 2721 / 2721 canonical matches
- structure_aware_medium: 1799 / 1799 canonical matches
- structure_aware_small: 2526 / 2526 canonical matches

Functions equivalent for all frozen candidate contents: True
Selected cache-key function: canonical_exact_text

CANONICAL CHUNK-TEXT KEY VALIDATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## Benchmark بازیابی پرسش با E5 اصلاح‌شده

E5 با پیشوندهای استاندارد `query:` و `passage:` اکنون با همان ۵۵۳ پرسش، همان ۲۷ seed واجد شرایط، همان برچسب‌های مثبت و همان چهار روش Chunking قبلی ارزیابی می‌شود.

نتیجه مستقیماً با E5 قبلی مقایسه خواهد شد. هیچ نتیجهٔ قبلی جایگزین نمی‌شود.

</div>

In [77]:
# Code Cell 58
# NB13 — Full question-retrieval benchmark for instructional E5

import time

instructional_benchmark_queries = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(instructional_benchmark_queries) == 553, (
    "Expected the same 553 eligible Golden questions as the prior benchmark."
)

assert all(
    record["question"] in query_hash_by_text
    for record in instructional_benchmark_queries
), "One or more Golden questions are missing from the instructional E5 cache."

instructional_query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in instructional_benchmark_queries
    ]
)

e5_instructional_benchmark_results = {}

print("=== E5 INSTRUCTIONAL FULL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(instructional_benchmark_queries))
print("Evaluation cutoffs: (1, 3, 5, 10)")

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]

    canonical_candidate_texts = [
        active_chunk_cache_key_function(record["content"])
        for record in candidate_records
    ]

    assert all(
        canonical_text in passage_hash_by_text
        for canonical_text in canonical_candidate_texts
    ), (
        f"{candidate_name} contains a Chunk not represented in "
        "the instructional E5 cache."
    )

    candidate_passage_matrix = np.vstack(
        [
            e5_instructional_embeddings["passage_vectors_by_text_hash"][
                passage_hash_by_text[canonical_text]
            ]
            for canonical_text in canonical_candidate_texts
        ]
    )

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]

        assert seed_entry["eligible"] is True, (
            f"{candidate_name}, seed {seed_id} is unexpectedly ineligible."
        )

        positive_ids = set(seed_entry["positive_chunk_ids"])
        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    started_at = time.perf_counter()
    similarity_matrix = instructional_query_matrix @ candidate_passage_matrix.T
    ranked_indexes = np.argsort(-similarity_matrix, axis=1)
    elapsed_seconds = time.perf_counter() - started_at

    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(instructional_benchmark_queries):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        ranked_candidate_indexes = ranked_indexes[query_index]
        positive_rank_positions = np.flatnonzero(
            np.isin(ranked_candidate_indexes, list(positive_indexes))
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    metrics = {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
        "seconds": round(elapsed_seconds, 3),
        "query_count": len(instructional_benchmark_queries),
        "candidate_chunk_count": len(candidate_records),
    }

    e5_instructional_benchmark_results[candidate_name] = metrics
    prior_metrics = embedding_benchmark_results["multilingual_e5_small"][candidate_name]

    print(f"\nmultilingual_e5_small_instructional × {candidate_name}")
    print(f"- Recall@1: {metrics['recall_at_1']:.4f}")
    print(f"- Recall@3: {metrics['recall_at_3']:.4f}")
    print(f"- Recall@5: {metrics['recall_at_5']:.4f}")
    print(f"- Recall@10: {metrics['recall_at_10']:.4f}")
    print(f"- MRR: {metrics['mrr']:.4f}")
    print(
        f"- Δ Recall@10 vs prior E5: "
        f"{metrics['recall_at_10'] - prior_metrics['recall_at_10']:+.4f}"
    )
    print(
        f"- Δ MRR vs prior E5: "
        f"{metrics['mrr'] - prior_metrics['mrr']:+.4f}"
    )
    print(f"- Seconds: {metrics['seconds']}")

e5_instructional_benchmark_report_path = (
    active_benchmark_run_directory
    / "multilingual_e5_small_instructional_retrieval_benchmark.json"
)

e5_instructional_benchmark_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": e5_instructional_embeddings["model_id"],
            "query_prefix": e5_instructional_embeddings["query_prefix"],
            "passage_prefix": e5_instructional_embeddings["passage_prefix"],
            "comparison_baseline": "multilingual_e5_small_without_prefixes",
            "common_eligible_seed_ids": common_eligible_seed_ids,
            "query_count": len(instructional_benchmark_queries),
            "results": e5_instructional_benchmark_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", e5_instructional_benchmark_report_path)
print("\nE5 INSTRUCTIONAL FULL RETRIEVAL BENCHMARK: PASS")

=== E5 INSTRUCTIONAL FULL RETRIEVAL BENCHMARK ===
Eligible seeds: 27
Eligible Golden queries: 553
Evaluation cutoffs: (1, 3, 5, 10)

multilingual_e5_small_instructional × baseline_phase1_char_safe
- Recall@1: 0.0090
- Recall@3: 0.0199
- Recall@5: 0.0253
- Recall@10: 0.0434
- MRR: 0.0225
- Δ Recall@10 vs prior E5: +0.0000
- Δ MRR vs prior E5: +0.0000
- Seconds: 0.162

multilingual_e5_small_instructional × parent_child_children
- Recall@1: 0.0054
- Recall@3: 0.0181
- Recall@5: 0.0181
- Recall@10: 0.0325
- MRR: 0.0190
- Δ Recall@10 vs prior E5: +0.0000
- Δ MRR vs prior E5: -0.0000
- Seconds: 0.098

multilingual_e5_small_instructional × structure_aware_medium
- Recall@1: 0.0036
- Recall@3: 0.0163
- Recall@5: 0.0181
- Recall@10: 0.0362
- MRR: 0.0177
- Δ Recall@10 vs prior E5: +0.0000
- Δ MRR vs prior E5: -0.0000
- Seconds: 0.067

multilingual_e5_small_instructional × structure_aware_small
- Recall@1: 0.0036
- Recall@3: 0.0181
- Recall@5: 0.0199
- Recall@10: 0.0307
- MRR: 0.0184
- Δ Recall@1

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی بازیابی واژگانی فارسی

نتایج قبلی نشان دادند که بازیابی embedding برای پرسش‌های Golden ضعیف‌تر از انتظار است.

در این مرحله یک بازیابی واژگانی مستقل بر پایهٔ TF-IDF فارسی اجرا می‌کنیم. این روش از شباهت واژه‌ها و عبارت‌های پرسش با متن Chunk استفاده می‌کند و نتیجهٔ آن با همان برچسب‌های مثبت و همان چهار روش Chunking ارزیابی می‌شود.

این فقط یک Baseline تشخیصی است و جایگزین embedding یا انتخاب نهایی سیستم نیست.

</div>

In [78]:
# Code Cell 59
# NB13 — Persian lexical retrieval baseline: TF-IDF word and phrase matching

import re
import time
from sklearn.feature_extraction.text import TfidfVectorizer

def persian_lexical_tokens(text):
    canonical_text = canonical_exact_text(text).lower()
    return re.findall(r"[\u0600-\u06FFa-z0-9]+", canonical_text)

lexical_benchmark_queries = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(lexical_benchmark_queries) == 553

lexical_query_texts = [
    record["question"]
    for record in lexical_benchmark_queries
]

lexical_retrieval_results = {}

print("=== PERSIAN LEXICAL RETRIEVAL BASELINE ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(lexical_benchmark_queries))
print("Tokenizer: Persian word tokens")
print("Features: word unigrams and bigrams")
print("Evaluation cutoffs: (1, 3, 5, 10)")

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]
        assert seed_entry["eligible"] is True

        positive_ids = set(seed_entry["positive_chunk_ids"])
        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    candidate_tfidf = vectorizer.fit_transform(candidate_texts)
    query_tfidf = vectorizer.transform(lexical_query_texts)

    similarity_matrix = (query_tfidf @ candidate_tfidf.T).toarray()
    ranked_indexes = np.argsort(-similarity_matrix, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(lexical_benchmark_queries):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        ranked_candidate_indexes = ranked_indexes[query_index]
        positive_rank_positions = np.flatnonzero(
            np.isin(ranked_candidate_indexes, list(positive_indexes))
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    metrics = {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
        "seconds": round(elapsed_seconds, 3),
        "query_count": len(lexical_benchmark_queries),
        "candidate_chunk_count": len(candidate_records),
        "vocabulary_size": int(len(vectorizer.vocabulary_)),
    }

    lexical_retrieval_results[candidate_name] = metrics

    print(f"\npersian_tfidf_lexical × {candidate_name}")
    print(f"- Recall@1: {metrics['recall_at_1']:.4f}")
    print(f"- Recall@3: {metrics['recall_at_3']:.4f}")
    print(f"- Recall@5: {metrics['recall_at_5']:.4f}")
    print(f"- Recall@10: {metrics['recall_at_10']:.4f}")
    print(f"- MRR: {metrics['mrr']:.4f}")
    print(f"- Vocabulary size: {metrics['vocabulary_size']}")
    print(f"- Seconds: {metrics['seconds']}")

print("\nPERSIAN LEXICAL RETRIEVAL BASELINE: PASS")

=== PERSIAN LEXICAL RETRIEVAL BASELINE ===
Eligible seeds: 27
Eligible Golden queries: 553
Tokenizer: Persian word tokens
Features: word unigrams and bigrams
Evaluation cutoffs: (1, 3, 5, 10)

persian_tfidf_lexical × baseline_phase1_char_safe
- Recall@1: 0.0054
- Recall@3: 0.0163
- Recall@5: 0.0271
- Recall@10: 0.0470
- MRR: 0.0226
- Vocabulary size: 297986
- Seconds: 2.976

persian_tfidf_lexical × parent_child_children
- Recall@1: 0.0054
- Recall@3: 0.0217
- Recall@5: 0.0325
- Recall@10: 0.0615
- MRR: 0.0264
- Vocabulary size: 293978
- Seconds: 2.527

persian_tfidf_lexical × structure_aware_medium
- Recall@1: 0.0072
- Recall@3: 0.0253
- Recall@5: 0.0362
- Recall@10: 0.0615
- MRR: 0.0289
- Vocabulary size: 294433
- Seconds: 2.397

persian_tfidf_lexical × structure_aware_small
- Recall@1: 0.0072
- Recall@3: 0.0235
- Recall@5: 0.0307
- Recall@10: 0.0506
- MRR: 0.0271
- Vocabulary size: 294081
- Seconds: 2.417

PERSIAN LEXICAL RETRIEVAL BASELINE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی Hybrid Retrieval با ترکیب E5 و بازیابی واژگانی

Embedding و بازیابی واژگانی هر کدام ممکن است بخشی از پرسش‌ها را بهتر بازیابی کنند.

در این مرحله رتبه‌های E5 و TF-IDF فارسی با روش Reciprocal Rank Fusion یا RRF ترکیب می‌شوند. هدف، بررسی این است که آیا ترکیب دو روش از هر روش به‌تنهایی بهتر است یا خیر.

این یک آزمایش Benchmark است و هنوز انتخاب نهایی سیستم نیست.

</div>

In [79]:
# Code Cell 60
# NB13 — Hybrid E5 + Persian lexical retrieval benchmark with RRF

from sklearn.feature_extraction.text import TfidfVectorizer
import time

def calculate_retrieval_metrics(
    ranked_indexes,
    query_records,
    positive_chunk_indexes_by_seed,
):
    first_positive_ranks = []
    hit_lists = {cutoff: [] for cutoff in (1, 3, 5, 10)}

    for query_index, query_record in enumerate(query_records):
        seed_id = query_record["source_seed_id"]
        positive_indexes = positive_chunk_indexes_by_seed[seed_id]

        positive_rank_positions = np.flatnonzero(
            np.isin(
                ranked_indexes[query_index],
                list(positive_indexes),
            )
        )

        first_rank = int(positive_rank_positions[0]) + 1
        first_positive_ranks.append(first_rank)

        for cutoff in hit_lists:
            hit_lists[cutoff].append(first_rank <= cutoff)

    return {
        "recall_at_1": float(np.mean(hit_lists[1])),
        "recall_at_3": float(np.mean(hit_lists[3])),
        "recall_at_5": float(np.mean(hit_lists[5])),
        "recall_at_10": float(np.mean(hit_lists[10])),
        "mrr": float(np.mean([1 / rank for rank in first_positive_ranks])),
        "mean_first_positive_rank": float(np.mean(first_positive_ranks)),
    }


hybrid_query_records = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(hybrid_query_records) == 553

hybrid_e5_query_matrix = np.vstack(
    [
        e5_instructional_embeddings["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in hybrid_query_records
    ]
)

hybrid_query_texts = [
    record["question"]
    for record in hybrid_query_records
]

rrf_constant = 60
hybrid_retrieval_results = {}

print("=== HYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(hybrid_query_records))
print("Fusion method: Reciprocal Rank Fusion")
print("RRF constant:", rrf_constant)

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        seed_entry = seed_registry[seed_id]
        positive_ids = set(seed_entry["positive_chunk_ids"])

        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    e5_candidate_matrix = np.vstack(
        [
            e5_instructional_embeddings["passage_vectors_by_text_hash"][
                passage_hash_by_text[text]
            ]
            for text in candidate_texts
        ]
    )

    lexical_vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    lexical_candidate_matrix = lexical_vectorizer.fit_transform(candidate_texts)
    lexical_query_matrix = lexical_vectorizer.transform(hybrid_query_texts)

    e5_scores = hybrid_e5_query_matrix @ e5_candidate_matrix.T
    lexical_scores = (
        lexical_query_matrix @ lexical_candidate_matrix.T
    ).toarray()

    e5_ranked_indexes = np.argsort(-e5_scores, axis=1)
    lexical_ranked_indexes = np.argsort(-lexical_scores, axis=1)

    e5_rank_positions = np.empty_like(e5_ranked_indexes)
    lexical_rank_positions = np.empty_like(lexical_ranked_indexes)

    row_indexes = np.arange(len(hybrid_query_records))[:, None]

    e5_rank_positions[row_indexes, e5_ranked_indexes] = (
        np.arange(len(candidate_records))[None, :] + 1
    )
    lexical_rank_positions[row_indexes, lexical_ranked_indexes] = (
        np.arange(len(candidate_records))[None, :] + 1
    )

    rrf_scores = (
        1.0 / (rrf_constant + e5_rank_positions)
        + 1.0 / (rrf_constant + lexical_rank_positions)
    )
    rrf_ranked_indexes = np.argsort(-rrf_scores, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    e5_metrics = calculate_retrieval_metrics(
        e5_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    lexical_metrics = calculate_retrieval_metrics(
        lexical_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    rrf_metrics = calculate_retrieval_metrics(
        rrf_ranked_indexes,
        hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )

    rrf_metrics["seconds"] = round(elapsed_seconds, 3)
    rrf_metrics["candidate_chunk_count"] = len(candidate_records)

    hybrid_retrieval_results[candidate_name] = {
        "e5_instructional": e5_metrics,
        "persian_tfidf_lexical": lexical_metrics,
        "rrf_e5_plus_lexical": rrf_metrics,
    }

    print(f"\nE5 + lexical RRF × {candidate_name}")
    print(f"- E5 Recall@10: {e5_metrics['recall_at_10']:.4f}")
    print(f"- Lexical Recall@10: {lexical_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid Recall@10: {rrf_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid MRR: {rrf_metrics['mrr']:.4f}")
    print(f"- Seconds: {rrf_metrics['seconds']}")

print("\nHYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS")

=== HYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===
Eligible seeds: 27
Eligible Golden queries: 553
Fusion method: Reciprocal Rank Fusion
RRF constant: 60

E5 + lexical RRF × baseline_phase1_char_safe
- E5 Recall@10: 0.0434
- Lexical Recall@10: 0.0470
- Hybrid Recall@10: 0.0633
- Hybrid MRR: 0.0267
- Seconds: 3.45

E5 + lexical RRF × parent_child_children
- E5 Recall@10: 0.0325
- Lexical Recall@10: 0.0615
- Hybrid Recall@10: 0.0597
- Hybrid MRR: 0.0270
- Seconds: 2.644

E5 + lexical RRF × structure_aware_medium
- E5 Recall@10: 0.0362
- Lexical Recall@10: 0.0615
- Hybrid Recall@10: 0.0524
- Hybrid MRR: 0.0272
- Seconds: 2.525

E5 + lexical RRF × structure_aware_small
- E5 Recall@10: 0.0307
- Lexical Recall@10: 0.0506
- Hybrid Recall@10: 0.0597
- Hybrid MRR: 0.0240
- Seconds: 3.24

HYBRID E5 + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## آماده‌سازی Hybrid Retrieval با Arvan و بازیابی واژگانی

مدل Arvan در بازیابی متن شواهد عملکرد خوبی نشان داد، اما در بازیابی مستقیم پرسش ضعیف‌تر بود.

در این مرحله ابتدا محل دقیق embeddingهای Arvan موجود در حافظهٔ Notebook را پیدا می‌کنیم تا در مرحلهٔ بعد، بدون اجرای دوبارهٔ API و بدون هزینهٔ جدید، آن را با بازیابی واژگانی ترکیب کنیم.

</div>

In [82]:
# Code Cell 61
# NB13 — Arvan vector-cache discovery for hybrid retrieval

arvan_vector_lookup_candidates = []

for global_name, global_value in list(globals().items()):
    if not isinstance(global_value, dict):
        continue

    if {
        "query_vectors_by_text_hash",
        "passage_vectors_by_text_hash",
    }.issubset(global_value.keys()):
        arvan_vector_lookup_candidates.append(
            {
                "container_name": global_name,
                "lookup_name": None,
                "lookup": global_value,
            }
        )

    for nested_name, nested_value in global_value.items():
        if not isinstance(nested_value, dict):
            continue

        if {
            "query_vectors_by_text_hash",
            "passage_vectors_by_text_hash",
        }.issubset(nested_value.keys()):
            arvan_vector_lookup_candidates.append(
                {
                    "container_name": global_name,
                    "lookup_name": nested_name,
                    "lookup": nested_value,
                }
            )

print("=== ARVAN VECTOR-CACHE DISCOVERY ===")
print("Vector-cache candidates found:", len(arvan_vector_lookup_candidates))

for candidate in arvan_vector_lookup_candidates:
    lookup = candidate["lookup"]
    model_identifier = (
        lookup.get("model_id")
        or lookup.get("provider")
        or candidate["lookup_name"]
        or candidate["container_name"]
    )

    print("\nContainer:", candidate["container_name"])
    print("Nested lookup:", candidate["lookup_name"])
    print("Model identifier:", model_identifier)
    print(
        "Query-vector count:",
        len(lookup["query_vectors_by_text_hash"]),
    )
    print(
        "Passage-vector count:",
        len(lookup["passage_vectors_by_text_hash"]),
    )
    print(
        "Dimension:",
        lookup.get("dimension", lookup.get("embedding_dimension")),
    )

arvan_matches = [
    candidate
    for candidate in arvan_vector_lookup_candidates
    if (
        "arvan" in str(candidate["lookup_name"]).lower()
        or "arvan" in str(candidate["container_name"]).lower()
        or "text-embedding-3-large"
        in str(candidate["lookup"].get("model_id", "")).lower()
        or candidate["lookup"].get("embedding_dimension") == 3072
        or candidate["lookup"].get("dimension") == 3072
    )
]

assert len(arvan_matches) == 1, (
    "Expected exactly one Arvan vector cache. Found: "
    f"{len(arvan_matches)}."
)

arvan_embedding_lookup = arvan_matches[0]["lookup"]

assert len(arvan_embedding_lookup["query_vectors_by_text_hash"]) == 573
assert len(arvan_embedding_lookup["passage_vectors_by_text_hash"]) == 7698

print("\nResolved Arvan model:", arvan_embedding_lookup.get("model_id"))
print("\nARVAN VECTOR-CACHE DISCOVERY: PASS")

=== ARVAN VECTOR-CACHE DISCOVERY ===
Vector-cache candidates found: 5

Container: benchmark_embeddings_by_model
Nested lookup: arvan_text_embedding_3_large
Model identifier: arvan_text_embedding_3_large
Query-vector count: 573
Passage-vector count: 7698
Dimension: 3072

Container: benchmark_embeddings_by_model
Nested lookup: multilingual_e5_small
Model identifier: multilingual_e5_small
Query-vector count: 573
Passage-vector count: 7698
Dimension: 384

Container: benchmark_embeddings_by_model
Nested lookup: all_minilm_l6_v2
Model identifier: all_minilm_l6_v2
Query-vector count: 573
Passage-vector count: 7698
Dimension: 384

Container: embedding_lookup
Nested lookup: None
Model identifier: embedding_lookup
Query-vector count: 573
Passage-vector count: 7698
Dimension: 384

Container: e5_instructional_embeddings
Nested lookup: None
Model identifier: intfloat/multilingual-e5-small
Query-vector count: 573
Passage-vector count: 7698
Dimension: 384

Resolved Arvan model: None

ARVAN VECTOR-CAC

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی Hybrid Retrieval با Arvan و بازیابی واژگانی

در این آزمایش، رتبه‌های مدل Arvan و بازیابی واژگانی فارسی با روش RRF ترکیب می‌شوند.

هدف این است که بررسی کنیم آیا مدل معنایی Arvan و تطبیق واژگانی، پرسش‌های Golden را مکمل یکدیگر بازیابی می‌کنند یا خیر.

</div>

In [83]:
# Code Cell 62
# NB13 — Hybrid Arvan + Persian lexical retrieval benchmark with RRF

arvan_hybrid_query_records = [
    record
    for record in benchmark_query_records
    if record["source_seed_id"] in common_eligible_seed_ids
]

assert len(arvan_hybrid_query_records) == 553

assert all(
    record["question"] in query_hash_by_text
    for record in arvan_hybrid_query_records
)

arvan_query_matrix = np.vstack(
    [
        arvan_embedding_lookup["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in arvan_hybrid_query_records
    ]
)

arvan_hybrid_query_texts = [
    record["question"]
    for record in arvan_hybrid_query_records
]

arvan_lexical_hybrid_results = {}
rrf_constant = 60

print("=== HYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===")
print("Eligible seeds:", len(common_eligible_seed_ids))
print("Eligible Golden queries:", len(arvan_hybrid_query_records))
print("Fusion method: Reciprocal Rank Fusion")
print("RRF constant:", rrf_constant)

for candidate_name, seed_registry in retrieval_evaluation_registry.items():
    candidate_path = project_root / retrieval_chunking_candidates[candidate_name]["path"]

    candidate_records = [
        json.loads(line)
        for line in candidate_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    candidate_chunk_ids = [
        record["chunk_id"]
        for record in candidate_records
    ]
    candidate_texts = [
        canonical_exact_text(record["content"])
        for record in candidate_records
    ]

    positive_chunk_indexes_by_seed = {}

    for seed_id in common_eligible_seed_ids:
        positive_ids = set(seed_registry[seed_id]["positive_chunk_ids"])

        positive_indexes = {
            index
            for index, chunk_id in enumerate(candidate_chunk_ids)
            if chunk_id in positive_ids
        }

        assert positive_indexes, (
            f"{candidate_name}, seed {seed_id} has no indexed positive Chunk."
        )

        positive_chunk_indexes_by_seed[seed_id] = positive_indexes

    arvan_candidate_matrix = np.vstack(
        [
            arvan_embedding_lookup["passage_vectors_by_text_hash"][
                passage_hash_by_text[text]
            ]
            for text in candidate_texts
        ]
    )

    lexical_vectorizer = TfidfVectorizer(
        tokenizer=persian_lexical_tokens,
        token_pattern=None,
        ngram_range=(1, 2),
        lowercase=False,
        norm="l2",
        sublinear_tf=True,
        dtype=np.float32,
    )

    started_at = time.perf_counter()

    lexical_candidate_matrix = lexical_vectorizer.fit_transform(candidate_texts)
    lexical_query_matrix = lexical_vectorizer.transform(
        arvan_hybrid_query_texts
    )

    arvan_scores = arvan_query_matrix @ arvan_candidate_matrix.T
    lexical_scores = (
        lexical_query_matrix @ lexical_candidate_matrix.T
    ).toarray()

    arvan_ranked_indexes = np.argsort(-arvan_scores, axis=1)
    lexical_ranked_indexes = np.argsort(-lexical_scores, axis=1)

    arvan_rank_positions = np.empty_like(arvan_ranked_indexes)
    lexical_rank_positions = np.empty_like(lexical_ranked_indexes)

    row_indexes = np.arange(len(arvan_hybrid_query_records))[:, None]
    candidate_rank_positions = (
        np.arange(len(candidate_records))[None, :] + 1
    )

    arvan_rank_positions[row_indexes, arvan_ranked_indexes] = (
        candidate_rank_positions
    )
    lexical_rank_positions[row_indexes, lexical_ranked_indexes] = (
        candidate_rank_positions
    )

    rrf_scores = (
        1.0 / (rrf_constant + arvan_rank_positions)
        + 1.0 / (rrf_constant + lexical_rank_positions)
    )
    rrf_ranked_indexes = np.argsort(-rrf_scores, axis=1)

    elapsed_seconds = time.perf_counter() - started_at

    arvan_metrics = calculate_retrieval_metrics(
        arvan_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    lexical_metrics = calculate_retrieval_metrics(
        lexical_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )
    rrf_metrics = calculate_retrieval_metrics(
        rrf_ranked_indexes,
        arvan_hybrid_query_records,
        positive_chunk_indexes_by_seed,
    )

    rrf_metrics["seconds"] = round(elapsed_seconds, 3)
    rrf_metrics["candidate_chunk_count"] = len(candidate_records)

    arvan_lexical_hybrid_results[candidate_name] = {
        "arvan": arvan_metrics,
        "persian_tfidf_lexical": lexical_metrics,
        "rrf_arvan_plus_lexical": rrf_metrics,
    }

    print(f"\nArvan + lexical RRF × {candidate_name}")
    print(f"- Arvan Recall@10: {arvan_metrics['recall_at_10']:.4f}")
    print(f"- Lexical Recall@10: {lexical_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid Recall@10: {rrf_metrics['recall_at_10']:.4f}")
    print(f"- Hybrid MRR: {rrf_metrics['mrr']:.4f}")
    print(f"- Seconds: {rrf_metrics['seconds']}")

print("\nHYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS")

=== HYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK ===
Eligible seeds: 27
Eligible Golden queries: 553
Fusion method: Reciprocal Rank Fusion
RRF constant: 60

Arvan + lexical RRF × baseline_phase1_char_safe
- Arvan Recall@10: 0.0362
- Lexical Recall@10: 0.0470
- Hybrid Recall@10: 0.0633
- Hybrid MRR: 0.0309
- Seconds: 3.393

Arvan + lexical RRF × parent_child_children
- Arvan Recall@10: 0.0325
- Lexical Recall@10: 0.0615
- Hybrid Recall@10: 0.0687
- Hybrid MRR: 0.0342
- Seconds: 2.705

Arvan + lexical RRF × structure_aware_medium
- Arvan Recall@10: 0.0344
- Lexical Recall@10: 0.0615
- Hybrid Recall@10: 0.0506
- Hybrid MRR: 0.0295
- Seconds: 2.566

Arvan + lexical RRF × structure_aware_small
- Arvan Recall@10: 0.0380
- Lexical Recall@10: 0.0506
- Hybrid Recall@10: 0.0615
- Hybrid MRR: 0.0265
- Seconds: 2.684

HYBRID ARVAN + PERSIAN LEXICAL RETRIEVAL BENCHMARK: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل شکست بازیابی در بهترین ترکیب فعلی

بهترین ترکیب فعلی، Arvan به‌همراه بازیابی واژگانی و Chunking والد–فرزند است.

در این مرحله عملکرد این ترکیب برای هر source seed جداگانه بررسی می‌شود تا مشخص شود کدام گروه از پرسش‌ها بازیابی می‌شوند، کدام گروه شکست می‌خورند و اصلاح بعدی باید روی چه بخش‌هایی متمرکز باشد.

</div>

In [84]:
# Code Cell 63
# NB13 — Per-seed failure profile for best current hybrid:
# Arvan + Persian lexical RRF × parent_child_children

best_candidate_name = "parent_child_children"
best_seed_registry = retrieval_evaluation_registry[best_candidate_name]
best_candidate_path = (
    project_root
    / retrieval_chunking_candidates[best_candidate_name]["path"]
)

best_candidate_records = [
    json.loads(line)
    for line in best_candidate_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

best_candidate_chunk_ids = [
    record["chunk_id"]
    for record in best_candidate_records
]
best_candidate_texts = [
    canonical_exact_text(record["content"])
    for record in best_candidate_records
]

best_positive_indexes_by_seed = {}

for seed_id in common_eligible_seed_ids:
    positive_ids = set(best_seed_registry[seed_id]["positive_chunk_ids"])

    positive_indexes = {
        index
        for index, chunk_id in enumerate(best_candidate_chunk_ids)
        if chunk_id in positive_ids
    }

    assert positive_indexes, (
        f"Seed {seed_id} has no indexed positive Chunk "
        f"for {best_candidate_name}."
    )

    best_positive_indexes_by_seed[seed_id] = positive_indexes

best_arvan_query_matrix = np.vstack(
    [
        arvan_embedding_lookup["query_vectors_by_text_hash"][
            query_hash_by_text[record["question"]]
        ]
        for record in arvan_hybrid_query_records
    ]
)

best_arvan_candidate_matrix = np.vstack(
    [
        arvan_embedding_lookup["passage_vectors_by_text_hash"][
            passage_hash_by_text[text]
        ]
        for text in best_candidate_texts
    ]
)

best_lexical_vectorizer = TfidfVectorizer(
    tokenizer=persian_lexical_tokens,
    token_pattern=None,
    ngram_range=(1, 2),
    lowercase=False,
    norm="l2",
    sublinear_tf=True,
    dtype=np.float32,
)

best_lexical_candidate_matrix = best_lexical_vectorizer.fit_transform(
    best_candidate_texts
)
best_lexical_query_matrix = best_lexical_vectorizer.transform(
    [
        record["question"]
        for record in arvan_hybrid_query_records
    ]
)

best_arvan_scores = (
    best_arvan_query_matrix @ best_arvan_candidate_matrix.T
)
best_lexical_scores = (
    best_lexical_query_matrix @ best_lexical_candidate_matrix.T
).toarray()

best_arvan_ranked_indexes = np.argsort(-best_arvan_scores, axis=1)
best_lexical_ranked_indexes = np.argsort(-best_lexical_scores, axis=1)

best_arvan_rank_positions = np.empty_like(best_arvan_ranked_indexes)
best_lexical_rank_positions = np.empty_like(best_lexical_ranked_indexes)

best_row_indexes = np.arange(
    len(arvan_hybrid_query_records)
)[:, None]
best_candidate_rank_positions = (
    np.arange(len(best_candidate_records))[None, :] + 1
)

best_arvan_rank_positions[
    best_row_indexes,
    best_arvan_ranked_indexes,
] = best_candidate_rank_positions

best_lexical_rank_positions[
    best_row_indexes,
    best_lexical_ranked_indexes,
] = best_candidate_rank_positions

best_rrf_scores = (
    1.0 / (60 + best_arvan_rank_positions)
    + 1.0 / (60 + best_lexical_rank_positions)
)
best_rrf_ranked_indexes = np.argsort(-best_rrf_scores, axis=1)

seed_failure_profile = {}

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]

    positive_rank_positions = np.flatnonzero(
        np.isin(
            best_rrf_ranked_indexes[query_index],
            list(positive_indexes),
        )
    )
    first_rank = int(positive_rank_positions[0]) + 1

    seed_failure_profile.setdefault(
        seed_id,
        {
            "seed_id": seed_id,
            "query_count": 0,
            "first_positive_ranks": [],
            "example_hit_questions": [],
            "example_miss_questions": [],
        },
    )

    seed_profile = seed_failure_profile[seed_id]
    seed_profile["query_count"] += 1
    seed_profile["first_positive_ranks"].append(first_rank)

    if first_rank <= 10 and len(seed_profile["example_hit_questions"]) < 2:
        seed_profile["example_hit_questions"].append(
            {
                "candidate_id": query_record["candidate_id"],
                "question": query_record["question"],
                "first_positive_rank": first_rank,
            }
        )

    if first_rank > 10 and len(seed_profile["example_miss_questions"]) < 2:
        seed_profile["example_miss_questions"].append(
            {
                "candidate_id": query_record["candidate_id"],
                "question": query_record["question"],
                "first_positive_rank": first_rank,
            }
        )

for seed_profile in seed_failure_profile.values():
    ranks = seed_profile["first_positive_ranks"]

    seed_profile["recall_at_1"] = float(
        np.mean([rank <= 1 for rank in ranks])
    )
    seed_profile["recall_at_3"] = float(
        np.mean([rank <= 3 for rank in ranks])
    )
    seed_profile["recall_at_5"] = float(
        np.mean([rank <= 5 for rank in ranks])
    )
    seed_profile["recall_at_10"] = float(
        np.mean([rank <= 10 for rank in ranks])
    )
    seed_profile["mrr"] = float(np.mean([1 / rank for rank in ranks]))
    seed_profile["mean_first_positive_rank"] = float(np.mean(ranks))
    seed_profile["miss_count_at_10"] = int(
        np.sum([rank > 10 for rank in ranks])
    )

seed_failure_profile_rows = sorted(
    seed_failure_profile.values(),
    key=lambda item: (
        item["recall_at_10"],
        item["mrr"],
        item["seed_id"],
    ),
)

seed_failure_profile_report_path = (
    active_benchmark_run_directory
    / "arvan_lexical_parent_child_seed_failure_profile.json"
)

seed_failure_profile_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": "arvan_text_embedding_3_large_plus_persian_tfidf",
            "fusion_method": "RRF",
            "rrf_constant": 60,
            "candidate": best_candidate_name,
            "query_count": len(arvan_hybrid_query_records),
            "seed_profiles": seed_failure_profile_rows,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== BEST HYBRID PER-SEED FAILURE PROFILE ===")
print("Configuration: Arvan + lexical RRF × parent_child_children")
print("Seeds evaluated:", len(seed_failure_profile_rows))
print("Queries evaluated:", len(arvan_hybrid_query_records))

print("\nPer-seed results, weakest first:")
for profile in seed_failure_profile_rows:
    print(
        f"- seed {profile['seed_id']}: "
        f"Recall@10={profile['recall_at_10']:.4f}, "
        f"MRR={profile['mrr']:.4f}, "
        f"misses@10={profile['miss_count_at_10']}/{profile['query_count']}"
    )

print("\nSeeds with zero Recall@10:")
zero_recall_seed_ids = [
    profile["seed_id"]
    for profile in seed_failure_profile_rows
    if profile["recall_at_10"] == 0.0
]
print(zero_recall_seed_ids)

print("\nReport:", seed_failure_profile_report_path)
print("\nBEST HYBRID PER-SEED FAILURE PROFILE: PASS")

=== BEST HYBRID PER-SEED FAILURE PROFILE ===
Configuration: Arvan + lexical RRF × parent_child_children
Seeds evaluated: 27
Queries evaluated: 553

Per-seed results, weakest first:
- seed 22: Recall@10=0.0000, MRR=0.0010, misses@10=20/20
- seed 15: Recall@10=0.0000, MRR=0.0029, misses@10=20/20
- seed 14: Recall@10=0.0000, MRR=0.0033, misses@10=20/20
- seed 4: Recall@10=0.0000, MRR=0.0035, misses@10=21/21
- seed 11: Recall@10=0.0000, MRR=0.0059, misses@10=21/21
- seed 26: Recall@10=0.0000, MRR=0.0085, misses@10=20/20
- seed 1: Recall@10=0.0000, MRR=0.0106, misses@10=21/21
- seed 2: Recall@10=0.0000, MRR=0.0164, misses@10=21/21
- seed 28: Recall@10=0.0000, MRR=0.0191, misses@10=20/20
- seed 7: Recall@10=0.0476, MRR=0.0152, misses@10=20/21
- seed 10: Recall@10=0.0476, MRR=0.0192, misses@10=20/21
- seed 8: Recall@10=0.0476, MRR=0.0200, misses@10=20/21
- seed 19: Recall@10=0.0500, MRR=0.0093, misses@10=19/20
- seed 17: Recall@10=0.0500, MRR=0.0122, misses@10=19/20
- seed 25: Recall@10=0.050

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی نمونه‌های واقعی شکست بازیابی

در این مرحله، برای seedهایی که Recall@10 آن‌ها صفر است، یک پرسش ناموفق، شاهد طلایی، پاسخ مورد انتظار و اولین Chunk بازیابی‌شده کنار هم نمایش داده می‌شوند.

این بررسی فقط برای تشخیص علت شکست است. پاسخ مورد انتظار و شاهد طلایی در ساخت query یا رتبه‌بندی استفاده نمی‌شوند.

</div>

In [85]:
# Code Cell 64
# NB13 — Human-readable inspection of zero-recall retrieval failures

active_golden_records = [
    json.loads(line)
    for line in golden_dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in active_golden_records
}

assert len(golden_record_by_candidate_id) == len(active_golden_records) == 573

zero_recall_seed_ids = [
    profile["seed_id"]
    for profile in seed_failure_profile_rows
    if profile["recall_at_10"] == 0.0
]

failure_examples = []

for seed_id in zero_recall_seed_ids:
    seed_query_indexes = [
        query_index
        for query_index, query_record in enumerate(arvan_hybrid_query_records)
        if query_record["source_seed_id"] == seed_id
    ]

    seed_query_rank_pairs = []

    for query_index in seed_query_indexes:
        positive_rank_positions = np.flatnonzero(
            np.isin(
                best_rrf_ranked_indexes[query_index],
                list(best_positive_indexes_by_seed[seed_id]),
            )
        )

        seed_query_rank_pairs.append(
            (
                int(positive_rank_positions[0]) + 1,
                query_index,
            )
        )

    best_failed_rank, selected_query_index = min(seed_query_rank_pairs)
    selected_query = arvan_hybrid_query_records[selected_query_index]
    selected_golden_record = golden_record_by_candidate_id[
        selected_query["candidate_id"]
    ]

    top_chunk_index = int(best_rrf_ranked_indexes[selected_query_index][0])
    top_chunk_record = best_candidate_records[top_chunk_index]

    failure_examples.append(
        {
            "seed_id": seed_id,
            "candidate_id": selected_query["candidate_id"],
            "first_positive_rank": best_failed_rank,
            "question": selected_golden_record["question"],
            "expected_answer": selected_golden_record["expected_answer"],
            "source_evidence": selected_golden_record["source_evidence"],
            "top_retrieved_chunk_id": top_chunk_record["chunk_id"],
            "top_retrieved_source_id": top_chunk_record["source_id"],
            "top_retrieved_content": top_chunk_record["content"],
        }
    )

failure_example_report_path = (
    active_benchmark_run_directory
    / "arvan_lexical_parent_child_zero_recall_examples.json"
)

failure_example_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "zero_recall_seed_ids": zero_recall_seed_ids,
            "examples": failure_examples,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== ZERO-RECALL FAILURE EXAMPLES ===")
print("Configuration: Arvan + lexical RRF × parent_child_children")
print("Seeds inspected:", zero_recall_seed_ids)

for example in failure_examples:
    print("\n" + "=" * 80)
    print(
        f"SEED {example['seed_id']} | "
        f"best positive rank: {example['first_positive_rank']}"
    )
    print("\nQuestion:")
    print(example["question"])

    print("\nGolden evidence preview:")
    print(example["source_evidence"][:600])

    print("\nExpected-answer preview (diagnostic only):")
    print(example["expected_answer"][:500])

    print("\nTop retrieved Chunk:")
    print("Chunk ID:", example["top_retrieved_chunk_id"])
    print("Source ID:", example["top_retrieved_source_id"])
    print("Content preview:", example["top_retrieved_content"][:600])

print("\nReport:", failure_example_report_path)
print("\nZERO-RECALL FAILURE EXAMPLES: PASS")

=== ZERO-RECALL FAILURE EXAMPLES ===
Configuration: Arvan + lexical RRF × parent_child_children
Seeds inspected: [22, 15, 14, 4, 11, 26, 1, 2, 28]

SEED 22 | best positive rank: 315

Question:
در شرایط عملی زندگی، درباره زندگی چه اشتباهی باید جلوگیری شود؟ از نگاه آموزشی چگونه تحلیل می‌شود؟

Golden evidence preview:
پلکان توانمندی مالی پنج مرحله دارد: 1. ناتوانی مالی: فرد برای گذران زندگی دائماً بدهکار است، بدون قرض نمی‌تواند زندگی کند و حتی برای پرداخت بدهی قبلی دوباره قرض می‌گیرد. 2. تقلّای مالی: فرد برای زندگی قرض می‌گیرد، اما می‌تواند بدهی را از درآمد آینده پرداخت کند؛ یعنی درآمد آینده خود را از قبل خرج می‌کند. 3. تعادل مالی: فرد امور مالی روزانه را اداره می‌کند و درآمد و هزینه‌اش متعادل است، اما برای اهداف کوتاه‌مدت یا بلندمدت پس‌انداز ندارد و در برابر شوک‌های مالی آسیب‌پذیر است. 4. تاب‌آوری مالی (امنیت مالی): علاوه بر تعادل دخل‌وخرج، فرد توان تحمل و عبور از شوک‌های مالی مانند بیکاری یا هزینه‌های ناگ

Expected-answer preview (diagnostic only):
پلکان توانمندی مالی پنج مرحله دارد: 1.

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل اطلاعات متمایزکننده در پرسش‌های Golden Dataset

در این مرحله بررسی می‌کنیم هر پرسش چند واژه یا عبارت متمایزکننده دارد؛ یعنی واژه‌هایی که در کل Chunkهای کاندیدا کم‌تکرارترند و می‌توانند به تشخیص منبع درست کمک کنند.

این تحلیل تشخیصی است، Golden Dataset را تغییر نمی‌دهد و از پاسخ مورد انتظار برای بازیابی استفاده نمی‌کند.

</div>

In [87]:
# Code Cell 65
# NB13 — Query information-profile diagnostic using corpus-derived IDF

active_golden_records = [
    json.loads(line)
    for line in golden_dataset_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

golden_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in active_golden_records
}

assert len(active_golden_records) == 573
assert len(question_rank_by_candidate_id) == 553

idf_values = best_lexical_vectorizer.idf_
high_idf_threshold = float(np.quantile(idf_values, 0.75))

information_profile_query_matrix = best_lexical_vectorizer.transform(
    [
        golden_record_by_candidate_id[candidate_id]["question"]
        for candidate_id in question_rank_by_candidate_id
    ]
)

information_profile_rows = []

for row_index, candidate_id in enumerate(question_rank_by_candidate_id):
    golden_record = golden_record_by_candidate_id[candidate_id]
    query_vector = information_profile_query_matrix[row_index]

    feature_indexes = query_vector.indices
    query_idf_values = idf_values[feature_indexes]

    high_idf_feature_count = int(
        np.sum(query_idf_values >= high_idf_threshold)
    )

    first_positive_rank = question_rank_by_candidate_id[candidate_id]

    information_profile_rows.append(
        {
            "candidate_id": candidate_id,
            "source_seed_id": golden_record["source_seed_id"],
            "question": golden_record["question"],
            "query_feature_count": int(len(feature_indexes)),
            "high_idf_feature_count": high_idf_feature_count,
            "maximum_query_idf": (
                float(np.max(query_idf_values))
                if len(query_idf_values)
                else 0.0
            ),
            "mean_query_idf": (
                float(np.mean(query_idf_values))
                if len(query_idf_values)
                else 0.0
            ),
            "information_status": (
                "NO_HIGH_IDF_SIGNAL"
                if high_idf_feature_count == 0
                else "HAS_HIGH_IDF_SIGNAL"
            ),
            "first_positive_rank": first_positive_rank,
            "hit_at_10": first_positive_rank <= 10,
        }
    )

assert len(information_profile_rows) == 553

information_summary = {}

for status in ("NO_HIGH_IDF_SIGNAL", "HAS_HIGH_IDF_SIGNAL"):
    rows = [
        row
        for row in information_profile_rows
        if row["information_status"] == status
    ]

    information_summary[status] = {
        "query_count": len(rows),
        "share_of_queries": len(rows) / len(information_profile_rows),
        "recall_at_10": (
            float(np.mean([row["hit_at_10"] for row in rows]))
            if rows
            else None
        ),
        "mean_first_positive_rank": (
            float(np.mean([row["first_positive_rank"] for row in rows]))
            if rows
            else None
        ),
    }

information_profile_report_path = (
    active_benchmark_run_directory
    / "active_golden_query_information_profile.json"
)

information_profile_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "candidate": best_candidate_name,
            "configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "idf_policy": {
                "source": "parent_child_children_tfidf_vocabulary",
                "high_idf_percentile": 0.75,
                "high_idf_threshold": high_idf_threshold,
            },
            "summary": information_summary,
            "rows": information_profile_rows,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== ACTIVE GOLDEN QUERY INFORMATION PROFILE ===")
print("Queries analysed:", len(information_profile_rows))
print(f"High-IDF threshold: {high_idf_threshold:.4f}")

for status, summary in information_summary.items():
    print(f"\n{status}")
    print("- Query count:", summary["query_count"])
    print(f"- Share: {summary['share_of_queries']:.4f}")

    if summary["recall_at_10"] is None:
        print("- Recall@10: N/A (no queries in this group)")
        print("- Mean first-positive rank: N/A (no queries in this group)")
    else:
        print(f"- Recall@10: {summary['recall_at_10']:.4f}")
        print(
            "- Mean first-positive rank: "
            f"{summary['mean_first_positive_rank']:.2f}"
        )

weak_information_examples = sorted(
    information_profile_rows,
    key=lambda row: (
        row["high_idf_feature_count"],
        row["maximum_query_idf"],
        row["first_positive_rank"],
    ),
)

print("\nLowest-information example questions, first 12:")
for example in weak_information_examples[:12]:
    print(
        f"- seed {example['source_seed_id']} | "
        f"rank={example['first_positive_rank']} | "
        f"high_idf_features={example['high_idf_feature_count']} | "
        f"question={example['question']}"
    )

print("\nReport:", information_profile_report_path)
print("\nACTIVE GOLDEN QUERY INFORMATION PROFILE: PASS")

=== ACTIVE GOLDEN QUERY INFORMATION PROFILE ===
Queries analysed: 553
High-IDF threshold: 8.2160

NO_HIGH_IDF_SIGNAL
- Query count: 67
- Share: 0.1212
- Recall@10: 0.1194
- Mean first-positive rank: 593.97

HAS_HIGH_IDF_SIGNAL
- Query count: 486
- Share: 0.8788
- Recall@10: 0.0617
- Mean first-positive rank: 598.71

Lowest-information example questions, first 12:
- seed 6 | rank=1 | high_idf_features=0 | question=برای یک فرد آگاه مالی، درباره استاندارد چه نکته‌ای باید بررسی شود؟
- seed 18 | rank=1 | high_idf_features=0 | question=برای یک فرد آگاه مالی، درباره نقدینگی چه نکته‌ای باید بررسی شود؟
- seed 24 | rank=2 | high_idf_features=0 | question=برای یک فرد آگاه مالی، درباره پرسشنامه چه نکته‌ای باید بررسی شود؟
- seed 9 | rank=3 | high_idf_features=0 | question=برای یک فرد آگاه مالی، درباره نوجوان چه نکته‌ای باید بررسی شود؟
- seed 9 | rank=4 | high_idf_features=0 | question=برای یک فرد آگاه مالی، درباره مدیرعامل چه نکته‌ای باید بررسی شود؟
- seed 13 | rank=4 | high_idf_features=0 | questi

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی سیاست بودجهٔ Context در روش Parent–Child

در Parent–Child، ابتدا Chunkهای کوچک بازیابی می‌شوند و سپس متن والد آن‌ها برای تولید پاسخ استفاده می‌شود.

در این مرحله چند سیاست متفاوت برای تعداد Childهای بازیابی‌شده، تعداد والدها و سقف tokenهای Context بررسی می‌شود تا هزینه و حجم Context قابل‌کنترل باشد، بدون آنکه اتصال Child به Parent از بین برود.

</div>

In [88]:
# Code Cell 66
# NB13 — Parent-child context-budget policy evaluation

assert "parent_child_parents" in chunking_candidates, (
    "The parent_child_parents candidate is unavailable."
)

parent_context_path = (
    project_root
    / chunking_candidates["parent_child_parents"]["path"]
)

assert parent_context_path.is_file(), (
    f"Missing parent-context file: {parent_context_path}"
)

parent_context_records = [
    json.loads(line)
    for line in parent_context_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

parent_record_by_chunk_id = {
    record["chunk_id"]: record
    for record in parent_context_records
}

assert len(parent_record_by_chunk_id) == len(parent_context_records)

context_budget_policies = [
    {
        "policy_id": "conservative",
        "retrieved_child_k": 5,
        "max_parent_chunks": 3,
        "max_context_tokens": 3000,
    },
    {
        "policy_id": "balanced",
        "retrieved_child_k": 10,
        "max_parent_chunks": 4,
        "max_context_tokens": 5000,
    },
    {
        "policy_id": "expansive",
        "retrieved_child_k": 10,
        "max_parent_chunks": 6,
        "max_context_tokens": 8000,
    },
]

context_budget_results = {}

print("=== PARENT–CHILD CONTEXT-BUDGET POLICY EVALUATION ===")
print("Queries evaluated:", len(arvan_hybrid_query_records))
print("Parent chunks available:", len(parent_context_records))

for policy in context_budget_policies:
    assembled_contexts = []
    missing_parent_references = 0

    for query_index, _ in enumerate(arvan_hybrid_query_records):
        ranked_child_indexes = best_rrf_ranked_indexes[query_index]
        selected_child_indexes = ranked_child_indexes[
            :policy["retrieved_child_k"]
        ]

        selected_parent_ids = []
        selected_parent_records = []
        current_token_total = 0

        for child_index in selected_child_indexes:
            child_record = best_candidate_records[int(child_index)]
            parent_chunk_id = child_record["parent_chunk_id"]

            if parent_chunk_id not in parent_record_by_chunk_id:
                missing_parent_references += 1
                continue

            if parent_chunk_id in selected_parent_ids:
                continue

            parent_record = parent_record_by_chunk_id[parent_chunk_id]
            parent_token_count = parent_record["token_count"]

            if len(selected_parent_records) >= policy["max_parent_chunks"]:
                continue

            if (
                current_token_total + parent_token_count
                > policy["max_context_tokens"]
            ):
                continue

            selected_parent_ids.append(parent_chunk_id)
            selected_parent_records.append(parent_record)
            current_token_total += parent_token_count

        assembled_contexts.append(
            {
                "parent_chunk_count": len(selected_parent_records),
                "token_count": current_token_total,
                "source_count": len(
                    {
                        parent_record["source_id"]
                        for parent_record in selected_parent_records
                    }
                ),
                "selected_parent_chunk_ids": selected_parent_ids,
            }
        )

    token_counts = np.array(
        [context["token_count"] for context in assembled_contexts]
    )
    parent_chunk_counts = np.array(
        [context["parent_chunk_count"] for context in assembled_contexts]
    )
    source_counts = np.array(
        [context["source_count"] for context in assembled_contexts]
    )

    result = {
        **policy,
        "query_count": len(assembled_contexts),
        "missing_parent_references": missing_parent_references,
        "empty_context_count": int(np.sum(token_counts == 0)),
        "mean_context_tokens": float(np.mean(token_counts)),
        "p95_context_tokens": float(np.percentile(token_counts, 95)),
        "maximum_context_tokens": int(np.max(token_counts)),
        "mean_parent_chunk_count": float(np.mean(parent_chunk_counts)),
        "mean_unique_source_count": float(np.mean(source_counts)),
    }

    context_budget_results[policy["policy_id"]] = result

    print(f"\nPolicy: {policy['policy_id']}")
    print("- Retrieved children:", policy["retrieved_child_k"])
    print("- Maximum parent chunks:", policy["max_parent_chunks"])
    print("- Maximum context tokens:", policy["max_context_tokens"])
    print(f"- Mean context tokens: {result['mean_context_tokens']:.1f}")
    print(f"- P95 context tokens: {result['p95_context_tokens']:.1f}")
    print(f"- Maximum actual tokens: {result['maximum_context_tokens']}")
    print(
        "- Mean parent chunks: "
        f"{result['mean_parent_chunk_count']:.2f}"
    )
    print(
        "- Mean unique sources: "
        f"{result['mean_unique_source_count']:.2f}"
    )
    print("- Empty contexts:", result["empty_context_count"])
    print(
        "- Missing parent references: "
        f"{result['missing_parent_references']}"
    )

assert all(
    result["missing_parent_references"] == 0
    for result in context_budget_results.values()
), "A selected Child referenced a missing Parent."

context_budget_report_path = (
    active_benchmark_run_directory
    / "parent_child_context_budget_policy_evaluation.json"
)

context_budget_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "child_candidate": "parent_child_children",
            "parent_candidate": "parent_child_parents",
            "results": context_budget_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", context_budget_report_path)
print("\nPARENT–CHILD CONTEXT-BUDGET POLICY EVALUATION: PASS")

=== PARENT–CHILD CONTEXT-BUDGET POLICY EVALUATION ===
Queries evaluated: 553
Parent chunks available: 1447

Policy: conservative
- Retrieved children: 5
- Maximum parent chunks: 3
- Maximum context tokens: 3000
- Mean context tokens: 2117.6
- P95 context tokens: 2837.0
- Maximum actual tokens: 2964
- Mean parent chunks: 2.98
- Mean unique sources: 2.92
- Empty contexts: 0
- Missing parent references: 0

Policy: balanced
- Retrieved children: 10
- Maximum parent chunks: 4
- Maximum context tokens: 5000
- Mean context tokens: 2918.5
- P95 context tokens: 4005.2
- Maximum actual tokens: 4615
- Mean parent chunks: 4.00
- Mean unique sources: 3.87
- Empty contexts: 0
- Missing parent references: 0

Policy: expansive
- Retrieved children: 10
- Maximum parent chunks: 6
- Maximum context tokens: 8000
- Mean context tokens: 4375.8
- P95 context tokens: 5728.2
- Maximum actual tokens: 6557
- Mean parent chunks: 6.00
- Mean unique sources: 5.66
- Empty contexts: 0
- Missing parent references: 0



<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## سنجش حفظ شواهد مثبت در بودجهٔ Context

برای هر سیاست Parent–Child بررسی می‌کنیم:

- آیا Child مثبت در نتایج بازیابی‌شده وجود داشته است؟
- اگر وجود داشته، آیا Parent متناظر آن واقعاً در Context نهایی باقی مانده است؟

این سنجش نشان می‌دهد محدودیت token و تعداد والدها چه مقدار از شواهد بازیابی‌شده را حفظ یا حذف می‌کند.

</div>

In [89]:
# Code Cell 67
# NB13 — Positive-evidence retention under parent-child context budgets

context_budget_evidence_retention = {}

print("=== PARENT–CHILD POSITIVE-EVIDENCE RETENTION ===")
print("Queries evaluated:", len(arvan_hybrid_query_records))

for policy in context_budget_policies:
    retrieved_positive_query_count = 0
    retained_positive_parent_query_count = 0
    missing_parent_references = 0

    for query_index, query_record in enumerate(arvan_hybrid_query_records):
        seed_id = query_record["source_seed_id"]
        positive_child_indexes = best_positive_indexes_by_seed[seed_id]

        selected_child_indexes = best_rrf_ranked_indexes[query_index][
            :policy["retrieved_child_k"]
        ]

        positive_child_was_retrieved = any(
            int(child_index) in positive_child_indexes
            for child_index in selected_child_indexes
        )

        if positive_child_was_retrieved:
            retrieved_positive_query_count += 1

        positive_parent_ids = {
            best_candidate_records[child_index]["parent_chunk_id"]
            for child_index in positive_child_indexes
        }

        selected_parent_ids = []
        selected_parent_token_total = 0

        for child_index in selected_child_indexes:
            child_record = best_candidate_records[int(child_index)]
            parent_chunk_id = child_record["parent_chunk_id"]

            if parent_chunk_id not in parent_record_by_chunk_id:
                missing_parent_references += 1
                continue

            if parent_chunk_id in selected_parent_ids:
                continue

            parent_record = parent_record_by_chunk_id[parent_chunk_id]
            parent_token_count = parent_record["token_count"]

            if len(selected_parent_ids) >= policy["max_parent_chunks"]:
                continue

            if (
                selected_parent_token_total + parent_token_count
                > policy["max_context_tokens"]
            ):
                continue

            selected_parent_ids.append(parent_chunk_id)
            selected_parent_token_total += parent_token_count

        if positive_parent_ids.intersection(selected_parent_ids):
            retained_positive_parent_query_count += 1

    retrieval_rate = (
        retrieved_positive_query_count
        / len(arvan_hybrid_query_records)
    )
    parent_retention_rate = (
        retained_positive_parent_query_count
        / len(arvan_hybrid_query_records)
    )
    conditional_retention_rate = (
        retained_positive_parent_query_count
        / retrieved_positive_query_count
        if retrieved_positive_query_count
        else None
    )

    result = {
        **policy,
        "query_count": len(arvan_hybrid_query_records),
        "retrieved_positive_query_count": retrieved_positive_query_count,
        "retrieval_recall_at_child_k": retrieval_rate,
        "retained_positive_parent_query_count": (
            retained_positive_parent_query_count
        ),
        "parent_evidence_recall_after_budget": parent_retention_rate,
        "conditional_parent_retention_after_retrieval": (
            conditional_retention_rate
        ),
        "missing_parent_references": missing_parent_references,
    }

    context_budget_evidence_retention[policy["policy_id"]] = result

    print(f"\nPolicy: {policy['policy_id']}")
    print(
        "- Positive Child retrieved: "
        f"{retrieved_positive_query_count}/{len(arvan_hybrid_query_records)} "
        f"({retrieval_rate:.4f})"
    )
    print(
        "- Positive Parent retained in final Context: "
        f"{retained_positive_parent_query_count}/"
        f"{len(arvan_hybrid_query_records)} "
        f"({parent_retention_rate:.4f})"
    )
    print(
        "- Conditional retention after positive retrieval: "
        f"{conditional_retention_rate:.4f}"
    )
    print(
        "- Missing parent references: "
        f"{missing_parent_references}"
    )

assert all(
    result["missing_parent_references"] == 0
    for result in context_budget_evidence_retention.values()
), "A Child referenced a missing Parent."

context_budget_evidence_retention_report_path = (
    active_benchmark_run_directory
    / "parent_child_context_budget_evidence_retention.json"
)

context_budget_evidence_retention_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "results": context_budget_evidence_retention,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", context_budget_evidence_retention_report_path)
print("\nPARENT–CHILD POSITIVE-EVIDENCE RETENTION: PASS")

=== PARENT–CHILD POSITIVE-EVIDENCE RETENTION ===
Queries evaluated: 553

Policy: conservative
- Positive Child retrieved: 26/553 (0.0470)
- Positive Parent retained in final Context: 17/553 (0.0307)
- Conditional retention after positive retrieval: 0.6538
- Missing parent references: 0

Policy: balanced
- Positive Child retrieved: 38/553 (0.0687)
- Positive Parent retained in final Context: 23/553 (0.0416)
- Conditional retention after positive retrieval: 0.6053
- Missing parent references: 0

Policy: expansive
- Positive Child retrieved: 38/553 (0.0687)
- Positive Parent retained in final Context: 31/553 (0.0561)
- Conditional retention after positive retrieval: 0.8158
- Missing parent references: 0

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729/parent_child_context_budget_evidence_retention.json

PARENT–CHILD POSITIVE-EVIDENCE RETENTION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ثبت پیکربندی موقت برای ادامهٔ ارزیابی

بهترین پیکربندی فعلی برای ادامهٔ آزمایش‌ها ثبت می‌شود، اما به دلیل پایین‌بودن Recall بازیابی پرسش و نبود انتخاب نهایی، برای استفادهٔ Production تأیید نمی‌شود.

این ثبت نسخه‌محور است و با تغییر Golden Dataset، در اجرای مجدد به‌صورت مستقل ارزیابی خواهد شد.

</div>

In [90]:
# Code Cell 68
# NB13 — Provisional retrieval configuration registration

best_hybrid_metrics = arvan_lexical_hybrid_results[
    "parent_child_children"
]["rrf_arvan_plus_lexical"]

expansive_context_metrics = context_budget_evidence_retention["expansive"]
expansive_context_profile = context_budget_results["expansive"]

provisional_retrieval_configuration = {
    "active_golden_dataset_sha256": golden_dataset_context["sha256"],
    "status": "PROVISIONAL_FOR_FURTHER_EVALUATION_NOT_PRODUCTION",
    "retrieval": {
        "dense_model": "arvan_text_embedding_3_large",
        "lexical_model": "persian_tfidf_word_unigram_bigram",
        "fusion_method": "reciprocal_rank_fusion",
        "rrf_constant": 60,
        "retrieval_candidate": "parent_child_children",
        "query_recall_at_10": best_hybrid_metrics["recall_at_10"],
        "query_mrr": best_hybrid_metrics["mrr"],
    },
    "context_assembly": {
        "parent_candidate": "parent_child_parents",
        "policy_id": "expansive",
        "retrieved_child_k": expansive_context_profile[
            "retrieved_child_k"
        ],
        "max_parent_chunks": expansive_context_profile[
            "max_parent_chunks"
        ],
        "max_context_tokens": expansive_context_profile[
            "max_context_tokens"
        ],
        "p95_context_tokens": expansive_context_profile[
            "p95_context_tokens"
        ],
        "conditional_parent_retention_after_retrieval": (
            expansive_context_metrics[
                "conditional_parent_retention_after_retrieval"
            ]
        ),
    },
    "not_approved_for_production_because": [
        (
            "Question retrieval remains too weak: only "
            f"{best_hybrid_metrics['recall_at_10']:.4f} Recall@10."
        ),
        (
            "Context assembly cannot recover evidence that was not "
            "retrieved among the top Child results."
        ),
        (
            "Final model-and-chunking selection requires a completed "
            "query-side remediation evaluation."
        ),
    ],
    "required_next_work": [
        "query_reformulation_or_reranking_experiment",
        "post_remediation_retrieval_benchmark",
        "rerun_with_active_golden_dataset_after_feedback_integration",
        "final_tradeoff_decision",
    ],
}

provisional_retrieval_configuration_path = (
    active_benchmark_run_directory
    / "provisional_retrieval_configuration.json"
)

provisional_retrieval_configuration_path.write_text(
    json.dumps(
        provisional_retrieval_configuration,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== PROVISIONAL RETRIEVAL CONFIGURATION ===")
print(
    "Status:",
    provisional_retrieval_configuration["status"],
)
print(
    "Retrieval Recall@10:",
    f"{best_hybrid_metrics['recall_at_10']:.4f}",
)
print(
    "Retrieval MRR:",
    f"{best_hybrid_metrics['mrr']:.4f}",
)
print(
    "Context policy:",
    provisional_retrieval_configuration["context_assembly"]["policy_id"],
)
print(
    "Conditional parent retention:",
    f"{expansive_context_metrics['conditional_parent_retention_after_retrieval']:.4f}",
)
print("\nReport:", provisional_retrieval_configuration_path)
print("\nPROVISIONAL RETRIEVAL CONFIGURATION: PASS")

=== PROVISIONAL RETRIEVAL CONFIGURATION ===
Status: PROVISIONAL_FOR_FURTHER_EVALUATION_NOT_PRODUCTION
Retrieval Recall@10: 0.0687
Retrieval MRR: 0.0342
Context policy: expansive
Conditional parent retention: 0.8158

Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729/provisional_retrieval_configuration.json

PROVISIONAL RETRIEVAL CONFIGURATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## کشف قابلیت Reranking

Reranker پس از بازیابی اولیه، پرسش و تعداد محدودی Chunk برتر را با هم بررسی می‌کند و می‌تواند ترتیب نتایج را دقیق‌تر کند.

در این مرحله فقط مدل‌های محلی و تنظیمات موجود بررسی می‌شوند. هیچ دانلود، نصب، فراخوانی API یا هزینه‌ای انجام نمی‌شود.

</div>

In [91]:
# Code Cell 69
# NB13 — Read-only reranking capability discovery

import importlib.util
import os
from pathlib import Path

huggingface_cache_roots = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path.home() / "Library" / "Caches" / "huggingface" / "hub",
]

local_model_directories = []

for cache_root in huggingface_cache_roots:
    if not cache_root.is_dir():
        continue

    local_model_directories.extend(
        sorted(
            path
            for path in cache_root.glob("models--*")
            if path.is_dir()
        )
    )

reranker_name_indicators = (
    "reranker",
    "cross-encoder",
    "cross_encoder",
    "bge",
    "ms-marco",
)

local_reranker_directories = [
    model_directory
    for model_directory in local_model_directories
    if any(
        indicator in model_directory.name.lower()
        for indicator in reranker_name_indicators
    )
]

arvan_configuration_keys = sorted(
    environment_key
    for environment_key in os.environ
    if environment_key.startswith("ARVAN")
)

print("=== RERANKING CAPABILITY DISCOVERY ===")
print(
    "sentence_transformers available:",
    importlib.util.find_spec("sentence_transformers") is not None,
)
print(
    "CrossEncoder class available:",
    end=" ",
)

try:
    from sentence_transformers import CrossEncoder
    print(True)
except Exception:
    print(False)

print("\nLocal Hugging Face model directories:", len(local_model_directories))

print("\nLocal reranker-like model directories:")
if local_reranker_directories:
    for model_directory in local_reranker_directories:
        print("-", model_directory.name)
else:
    print("NONE FOUND")

print("\nConfigured Arvan environment-variable names:")
for environment_key in arvan_configuration_keys:
    print("-", environment_key)

known_reranking_environment_keys = [
    key
    for key in arvan_configuration_keys
    if any(
        indicator in key.lower()
        for indicator in (
            "rerank",
            "chat",
            "completion",
            "generation",
        )
    )
]

print("\nConfigured reranking-or-generation endpoint names:")
if known_reranking_environment_keys:
    for environment_key in known_reranking_environment_keys:
        print("-", environment_key)
else:
    print("NONE FOUND")

print("\nRERANKING CAPABILITY DISCOVERY: PASS")

=== RERANKING CAPABILITY DISCOVERY ===
sentence_transformers available: True
CrossEncoder class available: True

Local Hugging Face model directories: 3

Local reranker-like model directories:
NONE FOUND

Configured Arvan environment-variable names:
- ARVAN_AI_API_KEY
- ARVAN_CHAT_MODEL
- ARVAN_CHAT_URL
- ARVAN_EMBEDDING_URL

Configured reranking-or-generation endpoint names:
- ARVAN_CHAT_MODEL
- ARVAN_CHAT_URL

RERANKING CAPABILITY DISCOVERY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی آماده‌بودن سرویس گفت‌وگویی Arvan برای Reranking

مدل گفت‌وگویی می‌تواند در مرحلهٔ بعد برای بازنویسی کنترل‌شدهٔ پرسش یا رتبه‌بندی دوبارهٔ تعداد محدودی Chunk برتر آزمایش شود.

در این مرحله فقط وجود تنظیمات و سازگاری اتصال بررسی می‌شود. هیچ درخواست API و هیچ هزینه‌ای ایجاد نمی‌شود.

</div>

In [92]:
# Code Cell 70
# NB13 — Arvan chat configuration preflight for reranking experiments

from urllib.parse import urlparse
from dotenv import load_dotenv
import importlib.util
import os

arvan_env_path = project_root / ".env"

assert arvan_env_path.is_file(), (
    f"Arvan environment file is missing: {arvan_env_path}"
)

load_dotenv(arvan_env_path, override=False)

arvan_chat_api_key = os.getenv("ARVAN_AI_API_KEY")
arvan_chat_url = os.getenv("ARVAN_CHAT_URL")
arvan_chat_model = os.getenv("ARVAN_CHAT_MODEL")

assert arvan_chat_api_key, "ARVAN_AI_API_KEY is not configured."
assert arvan_chat_url, "ARVAN_CHAT_URL is not configured."
assert arvan_chat_model, "ARVAN_CHAT_MODEL is not configured."

parsed_chat_url = urlparse(arvan_chat_url)

assert parsed_chat_url.scheme == "https", (
    "Arvan chat endpoint must use HTTPS."
)
assert parsed_chat_url.netloc, (
    "Arvan chat endpoint has no valid host."
)
assert importlib.util.find_spec("openai") is not None, (
    "The OpenAI-compatible client package is unavailable."
)

arvan_chat_configuration = {
    "model": arvan_chat_model,
    "endpoint_scheme": parsed_chat_url.scheme,
    "endpoint_host": parsed_chat_url.netloc,
    "endpoint_path_configured": bool(parsed_chat_url.path),
    "api_key_available": True,
}

print("=== ARVAN CHAT CONFIGURATION PREFLIGHT ===")
print("OpenAI-compatible client available: True")
print("Arvan API key available: True")
print("Chat model:", arvan_chat_configuration["model"])
print("Gateway scheme:", arvan_chat_configuration["endpoint_scheme"])
print("Gateway host:", arvan_chat_configuration["endpoint_host"])
print(
    "Gateway path configured:",
    arvan_chat_configuration["endpoint_path_configured"],
)
print("\nARVAN CHAT CONFIGURATION PREFLIGHT: PASS")

=== ARVAN CHAT CONFIGURATION PREFLIGHT ===
OpenAI-compatible client available: True
Arvan API key available: True
Chat model: Qwen3-30B-A3B
Gateway scheme: https
Gateway host: arvancloudai.ir
Gateway path configured: True

ARVAN CHAT CONFIGURATION PREFLIGHT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## اعتبارسنجی اتصال واقعی مدل گفت‌وگویی Arvan

در این مرحله با یک درخواست کوتاه و بدون دادهٔ پروژه، اتصال واقعی به مدل گفت‌وگویی Arvan بررسی می‌شود.

این آزمایش فقط برای اطمینان از آماده‌بودن زیرساخت Query Reformulation و Reranking است.

</div>

In [95]:
# Code Cell 71
# NB13 — Arvan chat live connection validation (corrected)

from openai import OpenAI
from datetime import datetime, timezone

arvan_chat_client = OpenAI(
    api_key=arvan_chat_api_key,
    base_url=arvan_chat_url,
)

live_chat_started_at = datetime.now(timezone.utc)

live_chat_response = arvan_chat_client.chat.completions.create(
    model=arvan_chat_model,
    messages=[
        {
            "role": "user",
            "content": (
                "Return exactly one word: READY. "
                "Do not explain your answer."
            ),
        }
    ],
    temperature=0,
    max_tokens=128,
)

assert live_chat_response.choices, (
    "Arvan chat returned no completion choices."
)

live_chat_choice = live_chat_response.choices[0]
live_chat_message = (
    live_chat_choice.message.content or ""
).strip()

arvan_chat_connection_report = {
    "checked_at_utc": live_chat_started_at.isoformat(),
    "model": arvan_chat_model,
    "endpoint_host": parsed_chat_url.netloc,
    "response_received": True,
    "finish_reason": live_chat_choice.finish_reason,
    "visible_text_received": bool(live_chat_message),
    "visible_text_character_count": len(live_chat_message),
}

arvan_chat_connection_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_connection_report.json"
)

arvan_chat_connection_report_path.write_text(
    json.dumps(
        arvan_chat_connection_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("=== ARVAN CHAT LIVE CONNECTION VALIDATION ===")
print("Model:", arvan_chat_model)
print("Response received:", True)
print("Finish reason:", live_chat_choice.finish_reason)
print("Visible text received:", bool(live_chat_message))
print("Response:", repr(live_chat_message))
print("Report:", arvan_chat_connection_report_path)
print("\nARVAN CHAT LIVE CONNECTION VALIDATION: PASS")

=== ARVAN CHAT LIVE CONNECTION VALIDATION ===
Model: Qwen3-30B-A3B
Response received: True
Finish reason: stop
Visible text received: True
Response: 'READY'
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729/arvan_chat_connection_report.json

ARVAN CHAT LIVE CONNECTION VALIDATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## بررسی ساختار پاسخ مدل گفت‌وگویی Arvan

در درخواست قبلی پاسخ از سرویس دریافت شد، اما متن در فیلد معمول `content` قرار نگرفت.

در این مرحله بدون ارسال درخواست جدید، ساختار همان پاسخ بررسی می‌شود تا روش درست خواندن خروجی مدل مشخص شود.

</div>

In [94]:
# Code Cell 72
# NB13 — Read-only inspection of the existing Arvan chat response

assert "live_chat_response" in globals(), (
    "The previous live-chat response is unavailable. "
    "Run Code Cell 71 once before this cell."
)

assert live_chat_response.choices, (
    "Arvan chat returned no choices."
)

live_chat_choice = live_chat_response.choices[0]
live_chat_response_message = live_chat_choice.message

message_fields = [
    field_name
    for field_name in (
        "content",
        "reasoning_content",
        "refusal",
        "tool_calls",
        "function_call",
    )
    if hasattr(live_chat_response_message, field_name)
]

print("=== ARVAN CHAT RESPONSE STRUCTURE DISCOVERY ===")
print("Response model:", getattr(live_chat_response, "model", None))
print("Finish reason:", getattr(live_chat_choice, "finish_reason", None))
print("Message fields available:", message_fields)

for field_name in message_fields:
    field_value = getattr(live_chat_response_message, field_name)
    print(f"\n{field_name}:")
    print(repr(field_value))

print("\nUsage:")
print(getattr(live_chat_response, "usage", None))

print("\nARVAN CHAT RESPONSE STRUCTURE DISCOVERY: PASS")

=== ARVAN CHAT RESPONSE STRUCTURE DISCOVERY ===
Response model: qwen/qwen3-30b-a3b
Finish reason: length
Message fields available: ['content', 'refusal', 'tool_calls', 'function_call']

content:
None

refusal:
None

tool_calls:
None

function_call:
None

Usage:
CompletionUsage(completion_tokens=10, prompt_tokens=20, total_tokens=30, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=11, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=7.4e-06, is_byok=False, cost_details={'upstream_inference_cost': 7.4e-06, 'upstream_inference_prompt_cost': 2.4e-06, 'upstream_inference_completions_cost': 5e-06})

ARVAN CHAT RESPONSE STRUCTURE DISCOVERY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## پایلوت Reranking با مدل گفت‌وگویی Arvan

در این آزمایش محدود، مدل گفت‌وگویی فقط پرسش و ده Chunk برتر بازیابی‌شده را می‌بیند و آن‌ها را دوباره رتبه‌بندی می‌کند.

پاسخ مورد انتظار، شواهد طلایی و اطلاعات label به مدل داده نمی‌شود. از آن‌ها فقط پس از پاسخ مدل، برای اندازه‌گیری کیفیت استفاده می‌شود.

هدف این است که قبل از هر اجرای بزرگ، بررسی کنیم آیا Reranking می‌تواند شواهدی را که در ده نتیجهٔ اول هستند به رتبه‌های بالاتر منتقل کند یا خیر.

</div>

In [96]:
# Code Cell 73
# NB13 — Controlled Arvan-chat reranking pilot:
# 20 queries with at least one positive Child already in the top 10

import re
import time

reranker_top_k = 10
reranker_pilot_size = 20
reranker_excerpt_char_limit = 1000

eligible_reranker_query_indexes = []

for query_index, query_record in enumerate(arvan_hybrid_query_records):
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][:reranker_top_k]

    if any(int(index) in positive_indexes for index in top_k_indexes):
        eligible_reranker_query_indexes.append(query_index)

assert eligible_reranker_query_indexes, (
    "No query has a positive Child inside the current top-k retrieval set."
)

# One query per seed first, then fill remaining pilot positions deterministically.
selected_reranker_query_indexes = []
selected_seed_ids = set()

for query_index in eligible_reranker_query_indexes:
    seed_id = arvan_hybrid_query_records[query_index]["source_seed_id"]

    if seed_id not in selected_seed_ids:
        selected_reranker_query_indexes.append(query_index)
        selected_seed_ids.add(seed_id)

    if len(selected_reranker_query_indexes) == reranker_pilot_size:
        break

if len(selected_reranker_query_indexes) < reranker_pilot_size:
    for query_index in eligible_reranker_query_indexes:
        if query_index not in selected_reranker_query_indexes:
            selected_reranker_query_indexes.append(query_index)

        if len(selected_reranker_query_indexes) == reranker_pilot_size:
            break

assert len(selected_reranker_query_indexes) == min(
    reranker_pilot_size,
    len(eligible_reranker_query_indexes),
)

def extract_reranked_labels(response_text, allowed_labels):
    match = re.search(r"\{.*\}", response_text, flags=re.DOTALL)

    if not match:
        return None

    try:
        parsed_response = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    proposed_labels = parsed_response.get("ranking")

    if not isinstance(proposed_labels, list):
        return None

    normalized_labels = [
        str(label).strip()
        for label in proposed_labels
        if str(label).strip() in allowed_labels
    ]

    deduplicated_labels = list(dict.fromkeys(normalized_labels))

    if set(deduplicated_labels) != set(allowed_labels):
        return None

    return deduplicated_labels

reranker_pilot_results = []

print("=== ARVAN CHAT RERANKING PILOT ===")
print("Top-k candidate passages:", reranker_top_k)
print("Eligible queries with a positive Child in top-k:", len(eligible_reranker_query_indexes))
print("Pilot queries:", len(selected_reranker_query_indexes))

for pilot_position, query_index in enumerate(
    selected_reranker_query_indexes,
    start=1,
):
    query_record = arvan_hybrid_query_records[query_index]
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][:reranker_top_k]

    passage_labels = [
        f"P{rank:02d}"
        for rank in range(1, reranker_top_k + 1)
    ]

    passage_by_label = {
        label: best_candidate_records[int(candidate_index)]
        for label, candidate_index in zip(passage_labels, top_k_indexes)
    }

    prompt_passages = "\n\n".join(
        (
            f"[{label}]\n"
            f"{passage_by_label[label]['content'][:reranker_excerpt_char_limit]}"
        )
        for label in passage_labels
    )

    reranker_prompt = f"""
وظیفه: فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار در خروجی بیاور.
- خروجی فقط JSON معتبر باشد و دقیقاً این ساختار را داشته باشد:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{query_record["question"]}

Passageها:
{prompt_passages}
""".strip()

    started_at = time.perf_counter()

    reranker_response = arvan_chat_client.chat.completions.create(
        model=arvan_chat_model,
        messages=[
            {
                "role": "user",
                "content": reranker_prompt,
            }
        ],
        temperature=0,
        max_tokens=256,
    )

    elapsed_seconds = time.perf_counter() - started_at

    raw_response_text = (
        reranker_response.choices[0].message.content or ""
    ).strip()

    reranked_labels = extract_reranked_labels(
        raw_response_text,
        allowed_labels=set(passage_labels),
    )

    original_positive_rank = next(
        rank
        for rank, candidate_index in enumerate(top_k_indexes, start=1)
        if int(candidate_index) in positive_indexes
    )

    result = {
        "pilot_position": pilot_position,
        "candidate_id": query_record["candidate_id"],
        "source_seed_id": seed_id,
        "question": query_record["question"],
        "original_positive_rank": original_positive_rank,
        "response_parse_status": (
            "PASS"
            if reranked_labels is not None
            else "FAIL"
        ),
        "seconds": round(elapsed_seconds, 3),
        "raw_response": raw_response_text,
    }

    if reranked_labels is not None:
        reranked_candidate_indexes = [
            int(top_k_indexes[passage_labels.index(label)])
            for label in reranked_labels
        ]

        reranked_positive_rank = next(
            rank
            for rank, candidate_index in enumerate(
                reranked_candidate_indexes,
                start=1,
            )
            if candidate_index in positive_indexes
        )

        result["reranked_positive_rank"] = reranked_positive_rank
        result["rank_change"] = (
            original_positive_rank - reranked_positive_rank
        )

    reranker_pilot_results.append(result)

    print(
        f"Pilot {pilot_position}/{len(selected_reranker_query_indexes)} | "
        f"seed={seed_id} | "
        f"original_rank={original_positive_rank} | "
        f"parse={result['response_parse_status']}"
    )

parsed_results = [
    result
    for result in reranker_pilot_results
    if result["response_parse_status"] == "PASS"
]

reranker_pilot_summary = {
    "pilot_query_count": len(reranker_pilot_results),
    "parse_success_count": len(parsed_results),
    "parse_success_rate": (
        len(parsed_results) / len(reranker_pilot_results)
    ),
    "mean_original_positive_rank": (
        float(
            np.mean(
                [
                    result["original_positive_rank"]
                    for result in parsed_results
                ]
            )
        )
        if parsed_results
        else None
    ),
    "mean_reranked_positive_rank": (
        float(
            np.mean(
                [
                    result["reranked_positive_rank"]
                    for result in parsed_results
                ]
            )
        )
        if parsed_results
        else None
    ),
    "improved_query_count": (
        int(
            np.sum(
                [
                    result["rank_change"] > 0
                    for result in parsed_results
                ]
            )
        )
        if parsed_results
        else 0
    ),
    "worsened_query_count": (
        int(
            np.sum(
                [
                    result["rank_change"] < 0
                    for result in parsed_results
                ]
            )
        )
        if parsed_results
        else 0
    ),
}

reranker_pilot_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_pilot.json"
)

reranker_pilot_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": arvan_chat_model,
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "top_k": reranker_top_k,
            "excerpt_char_limit": reranker_excerpt_char_limit,
            "summary": reranker_pilot_summary,
            "results": reranker_pilot_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nPilot parse success:", f"{reranker_pilot_summary['parse_success_rate']:.4f}")
print("Improved queries:", reranker_pilot_summary["improved_query_count"])
print("Worsened queries:", reranker_pilot_summary["worsened_query_count"])
print(
    "Mean original positive rank:",
    reranker_pilot_summary["mean_original_positive_rank"],
)
print(
    "Mean reranked positive rank:",
    reranker_pilot_summary["mean_reranked_positive_rank"],
)
print("Report:", reranker_pilot_report_path)
print("\nARVAN CHAT RERANKING PILOT: PASS")

=== ARVAN CHAT RERANKING PILOT ===
Top-k candidate passages: 10
Eligible queries with a positive Child in top-k: 38
Pilot queries: 20
Pilot 1/20 | seed=21 | original_rank=3 | parse=PASS
Pilot 2/20 | seed=12 | original_rank=1 | parse=PASS
Pilot 3/20 | seed=3 | original_rank=2 | parse=PASS
Pilot 4/20 | seed=5 | original_rank=7 | parse=PASS
Pilot 5/20 | seed=6 | original_rank=1 | parse=FAIL
Pilot 6/20 | seed=7 | original_rank=4 | parse=FAIL
Pilot 7/20 | seed=8 | original_rank=4 | parse=FAIL
Pilot 8/20 | seed=9 | original_rank=4 | parse=PASS
Pilot 9/20 | seed=10 | original_rank=4 | parse=FAIL
Pilot 10/20 | seed=13 | original_rank=4 | parse=PASS
Pilot 11/20 | seed=17 | original_rank=6 | parse=FAIL
Pilot 12/20 | seed=18 | original_rank=1 | parse=FAIL
Pilot 13/20 | seed=19 | original_rank=8 | parse=FAIL
Pilot 14/20 | seed=20 | original_rank=5 | parse=PASS
Pilot 15/20 | seed=23 | original_rank=6 | parse=PASS
Pilot 16/20 | seed=24 | original_rank=2 | parse=FAIL
Pilot 17/20 | seed=25 | original_

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## تحلیل پاسخ‌های نامعتبر پایلوت Reranking

در این مرحله پاسخ‌های دریافت‌شده از مدل که قابل‌خواندن نبودند بررسی می‌شوند.

هدف، تشخیص تفاوت میان دو حالت است:

- مدل رتبه‌بندی درست ساخته اما قالب JSON را رعایت نکرده است؛
- مدل دستور رتبه‌بندی را به‌درستی انجام نداده است.

هیچ درخواست جدیدی به سرویس ارسال نمی‌شود.

</div>

In [97]:
# Code Cell 74
# NB13 — Read-only analysis of malformed reranking-pilot responses

failed_reranker_pilot_results = [
    result
    for result in reranker_pilot_results
    if result["response_parse_status"] == "FAIL"
]

print("=== RERANKING PILOT RESPONSE-FORMAT ANALYSIS ===")
print("Total pilot responses:", len(reranker_pilot_results))
print("Malformed or incomplete responses:", len(failed_reranker_pilot_results))

for result in failed_reranker_pilot_results:
    raw_response = result["raw_response"]

    contains_json_object = bool(
        re.search(r"\{.*\}", raw_response, flags=re.DOTALL)
    )
    contains_json_array = bool(
        re.search(r"\[.*\]", raw_response, flags=re.DOTALL)
    )
    contains_passage_label = bool(
        re.search(r"\bP\d{2}\b", raw_response)
    )

    print("\n" + "=" * 80)
    print(
        f"Pilot {result['pilot_position']} | "
        f"seed={result['source_seed_id']} | "
        f"original_positive_rank={result['original_positive_rank']}"
    )
    print("Contains JSON object:", contains_json_object)
    print("Contains JSON array:", contains_json_array)
    print("Contains passage label:", contains_passage_label)
    print("Raw response:")
    print(repr(raw_response[:2000]))

reranker_format_analysis_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_reranking_pilot_format_analysis.json"
)

reranker_format_analysis_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "pilot_query_count": len(reranker_pilot_results),
            "failed_response_count": len(failed_reranker_pilot_results),
            "failed_responses": failed_reranker_pilot_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nReport:", reranker_format_analysis_report_path)
print("\nRERANKING PILOT RESPONSE-FORMAT ANALYSIS: PASS")

=== RERANKING PILOT RESPONSE-FORMAT ANALYSIS ===
Total pilot responses: 20
Malformed or incomplete responses: 10

Pilot 5 | seed=6 | original_positive_rank=1
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 6 | seed=7 | original_positive_rank=4
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 7 | seed=8 | original_positive_rank=4
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 9 | seed=10 | original_positive_rank=4
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 11 | seed=17 | original_positive_rank=6
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''

Pilot 12 | seed=18 | original_positive_rank=1
Contains JSON object: False
Contains JSON array: False
Contains passage label: False
Raw response:
''


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## آزمون کنترل‌شدهٔ سقف پاسخ Reranker

برخی پاسخ‌های پایلوت کاملاً خالی بودند. در این مرحله فقط یک نمونه از آن‌ها با سقف پاسخ بالاتر تکرار می‌شود تا مشخص شود آیا مدل پس از پایان reasoning داخلی، خروجی رتبه‌بندی قابل‌خواندن تولید می‌کند یا خیر.

</div>

In [98]:
# Code Cell 75
# NB13 — One controlled reranking retry with a larger completion budget

assert failed_reranker_pilot_results, (
    "No failed pilot response is available for the controlled retry."
)

retry_source_result = failed_reranker_pilot_results[0]
retry_query_index = next(
    query_index
    for query_index in selected_reranker_query_indexes
    if arvan_hybrid_query_records[query_index]["candidate_id"]
    == retry_source_result["candidate_id"]
)

retry_query_record = arvan_hybrid_query_records[retry_query_index]
retry_top_k_indexes = best_rrf_ranked_indexes[retry_query_index][:reranker_top_k]
retry_passage_labels = [
    f"P{rank:02d}"
    for rank in range(1, reranker_top_k + 1)
]

retry_passage_by_label = {
    label: best_candidate_records[int(candidate_index)]
    for label, candidate_index in zip(
        retry_passage_labels,
        retry_top_k_indexes,
    )
}

retry_prompt_passages = "\n\n".join(
    (
        f"[{label}]\n"
        f"{retry_passage_by_label[label]['content'][:reranker_excerpt_char_limit]}"
    )
    for label in retry_passage_labels
)

retry_reranker_prompt = f"""
فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار بیاور.
- پاسخ را فوراً و فقط به شکل JSON معتبر بده.
- هیچ توضیح، تحلیل یا متن دیگری ننویس.

ساختار دقیق خروجی:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{retry_query_record["question"]}

Passageها:
{retry_prompt_passages}
""".strip()

retry_started_at = time.perf_counter()

retry_response = arvan_chat_client.chat.completions.create(
    model=arvan_chat_model,
    messages=[
        {
            "role": "user",
            "content": retry_reranker_prompt,
        }
    ],
    temperature=0,
    max_tokens=1024,
)

retry_elapsed_seconds = time.perf_counter() - retry_started_at
retry_choice = retry_response.choices[0]
retry_raw_response_text = (
    retry_choice.message.content or ""
).strip()

retry_reranked_labels = extract_reranked_labels(
    retry_raw_response_text,
    allowed_labels=set(retry_passage_labels),
)

retry_usage = getattr(retry_response, "usage", None)

print("=== CONTROLLED RERANKING RETRY ===")
print("Source pilot position:", retry_source_result["pilot_position"])
print("Seed:", retry_query_record["source_seed_id"])
print("Finish reason:", retry_choice.finish_reason)
print("Visible text received:", bool(retry_raw_response_text))
print("Valid ranking parsed:", retry_reranked_labels is not None)
print("Seconds:", round(retry_elapsed_seconds, 3))
print("Usage:", retry_usage)
print("Raw response:", repr(retry_raw_response_text[:2000]))
print("\nCONTROLLED RERANKING RETRY: PASS")

=== CONTROLLED RERANKING RETRY ===
Source pilot position: 5
Seed: 6
Finish reason: stop
Visible text received: True
Valid ranking parsed: True
Seconds: 6.402
Usage: CompletionUsage(completion_tokens=919, prompt_tokens=5433, total_tokens=6352, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=1031, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=0.00111146, is_byok=False, cost_details={'upstream_inference_cost': 0.00111146, 'upstream_inference_prompt_cost': 0.00065196, 'upstream_inference_completions_cost': 0.0004595})
Raw response: '{"ranking":["P01","P02","P03","P04","P06","P08","P09","P05","P07","P10"]}'

CONTROLLED RERANKING RETRY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## پایلوت کالیبره‌شدهٔ Reranking

پس از تأیید سقف مناسب پاسخ، Reranking روی ده پرسش از seedهای متفاوت اجرا می‌شود.

هدف این مرحله سنجش هم‌زمان سه معیار است:

- پایداری قالب خروجی؛
- توان مدل در حفظ Passage مثبت؛
- بهبود یا افت رتبهٔ Passage مثبت نسبت به رتبه‌بندی اولیه.

</div>

In [99]:
# Code Cell 76
# NB13 — Calibrated 10-query Arvan-chat reranking pilot

calibrated_reranker_pilot_size = 10
calibrated_query_indexes = []
calibrated_seed_ids = set()

for query_index in eligible_reranker_query_indexes:
    seed_id = arvan_hybrid_query_records[query_index]["source_seed_id"]

    if seed_id not in calibrated_seed_ids:
        calibrated_query_indexes.append(query_index)
        calibrated_seed_ids.add(seed_id)

    if len(calibrated_query_indexes) == calibrated_reranker_pilot_size:
        break

assert len(calibrated_query_indexes) == calibrated_reranker_pilot_size

calibrated_reranker_results = []

print("=== CALIBRATED ARVAN CHAT RERANKING PILOT ===")
print("Pilot queries:", len(calibrated_query_indexes))
print("Top-k passages:", reranker_top_k)
print("Maximum completion tokens: 1024")

for pilot_position, query_index in enumerate(
    calibrated_query_indexes,
    start=1,
):
    query_record = arvan_hybrid_query_records[query_index]
    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][:reranker_top_k]

    passage_labels = [
        f"P{rank:02d}"
        for rank in range(1, reranker_top_k + 1)
    ]

    passage_by_label = {
        label: best_candidate_records[int(candidate_index)]
        for label, candidate_index in zip(
            passage_labels,
            top_k_indexes,
        )
    }

    prompt_passages = "\n\n".join(
        (
            f"[{label}]\n"
            f"{passage_by_label[label]['content'][:reranker_excerpt_char_limit]}"
        )
        for label in passage_labels
    )

    reranker_prompt = f"""
فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار بیاور.
- پاسخ را فوراً و فقط به شکل JSON معتبر بده.
- هیچ توضیح، تحلیل یا متن دیگری ننویس.

ساختار دقیق خروجی:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{query_record["question"]}

Passageها:
{prompt_passages}
""".strip()

    started_at = time.perf_counter()

    response = arvan_chat_client.chat.completions.create(
        model=arvan_chat_model,
        messages=[
            {
                "role": "user",
                "content": reranker_prompt,
            }
        ],
        temperature=0,
        max_tokens=1024,
    )

    elapsed_seconds = time.perf_counter() - started_at
    choice = response.choices[0]
    raw_response_text = (choice.message.content or "").strip()

    reranked_labels = extract_reranked_labels(
        raw_response_text,
        allowed_labels=set(passage_labels),
    )

    original_positive_rank = next(
        rank
        for rank, candidate_index in enumerate(top_k_indexes, start=1)
        if int(candidate_index) in positive_indexes
    )

    result = {
        "pilot_position": pilot_position,
        "candidate_id": query_record["candidate_id"],
        "source_seed_id": seed_id,
        "original_positive_rank": original_positive_rank,
        "finish_reason": choice.finish_reason,
        "response_parse_status": (
            "PASS"
            if reranked_labels is not None
            else "FAIL"
        ),
        "seconds": round(elapsed_seconds, 3),
        "cost": getattr(getattr(response, "usage", None), "cost", None),
        "raw_response": raw_response_text,
    }

    if reranked_labels is not None:
        reranked_candidate_indexes = [
            int(top_k_indexes[passage_labels.index(label)])
            for label in reranked_labels
        ]

        reranked_positive_rank = next(
            rank
            for rank, candidate_index in enumerate(
                reranked_candidate_indexes,
                start=1,
            )
            if candidate_index in positive_indexes
        )

        result["reranked_positive_rank"] = reranked_positive_rank
        result["rank_change"] = (
            original_positive_rank - reranked_positive_rank
        )

    calibrated_reranker_results.append(result)

    print(
        f"Pilot {pilot_position}/{len(calibrated_query_indexes)} | "
        f"seed={seed_id} | "
        f"original_rank={original_positive_rank} | "
        f"parse={result['response_parse_status']}"
    )

calibrated_parsed_results = [
    result
    for result in calibrated_reranker_results
    if result["response_parse_status"] == "PASS"
]

calibrated_reranker_summary = {
    "pilot_query_count": len(calibrated_reranker_results),
    "parse_success_rate": (
        len(calibrated_parsed_results)
        / len(calibrated_reranker_results)
    ),
    "mean_original_positive_rank": (
        float(
            np.mean(
                [
                    result["original_positive_rank"]
                    for result in calibrated_parsed_results
                ]
            )
        )
        if calibrated_parsed_results
        else None
    ),
    "mean_reranked_positive_rank": (
        float(
            np.mean(
                [
                    result["reranked_positive_rank"]
                    for result in calibrated_parsed_results
                ]
            )
        )
        if calibrated_parsed_results
        else None
    ),
    "improved_query_count": int(
        np.sum(
            [
                result["rank_change"] > 0
                for result in calibrated_parsed_results
            ]
        )
    ),
    "unchanged_query_count": int(
        np.sum(
            [
                result["rank_change"] == 0
                for result in calibrated_parsed_results
            ]
        )
    ),
    "worsened_query_count": int(
        np.sum(
            [
                result["rank_change"] < 0
                for result in calibrated_parsed_results
            ]
        )
    ),
    "total_observed_cost": float(
        np.sum(
            [
                result["cost"] or 0.0
                for result in calibrated_reranker_results
            ]
        )
    ),
}

calibrated_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_calibrated_reranking_pilot.json"
)

calibrated_reranker_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": arvan_chat_model,
            "candidate": best_candidate_name,
            "top_k": reranker_top_k,
            "max_tokens": 1024,
            "summary": calibrated_reranker_summary,
            "results": calibrated_reranker_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "\nParse success:",
    f"{calibrated_reranker_summary['parse_success_rate']:.4f}",
)
print(
    "Mean original positive rank:",
    calibrated_reranker_summary["mean_original_positive_rank"],
)
print(
    "Mean reranked positive rank:",
    calibrated_reranker_summary["mean_reranked_positive_rank"],
)
print(
    "Improved / unchanged / worsened:",
    calibrated_reranker_summary["improved_query_count"],
    "/",
    calibrated_reranker_summary["unchanged_query_count"],
    "/",
    calibrated_reranker_summary["worsened_query_count"],
)
print(
    "Total observed cost:",
    f"{calibrated_reranker_summary['total_observed_cost']:.6f}",
)
print("Report:", calibrated_reranker_report_path)
print("\nCALIBRATED ARVAN CHAT RERANKING PILOT: PASS")

=== CALIBRATED ARVAN CHAT RERANKING PILOT ===
Pilot queries: 10
Top-k passages: 10
Maximum completion tokens: 1024
Pilot 1/10 | seed=21 | original_rank=3 | parse=PASS
Pilot 2/10 | seed=12 | original_rank=1 | parse=PASS
Pilot 3/10 | seed=3 | original_rank=2 | parse=PASS
Pilot 4/10 | seed=5 | original_rank=7 | parse=PASS
Pilot 5/10 | seed=6 | original_rank=1 | parse=PASS
Pilot 6/10 | seed=7 | original_rank=4 | parse=PASS
Pilot 7/10 | seed=8 | original_rank=4 | parse=PASS
Pilot 8/10 | seed=9 | original_rank=4 | parse=PASS
Pilot 9/10 | seed=10 | original_rank=4 | parse=FAIL
Pilot 10/10 | seed=13 | original_rank=4 | parse=PASS

Parse success: 0.9000
Mean original positive rank: 3.3333333333333335
Mean reranked positive rank: 2.0
Improved / unchanged / worsened: 5 / 2 / 2
Total observed cost: 0.010899
Report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/embedding_benchmark/df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729/arvan_cha

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

## ارزیابی کامل مشروط Reranking

Reranker نمی‌تواند Chunkی را که در ده نتیجهٔ اولیه وجود ندارد بازیابی کند. بنابراین در این ارزیابی فقط پرسش‌هایی بررسی می‌شوند که حداقل یک شاهد مثبت از قبل در Top-10 آن‌ها وجود دارد.

هدف، سنجش دقیق اثر Reranking بر رتبهٔ شاهدهای موجود است. شواهد طلایی فقط برای محاسبهٔ معیار پس از پاسخ مدل استفاده می‌شوند و به مدل داده نمی‌شوند.

خروجی هر درخواست به‌صورت checkpoint ذخیره می‌شود تا اجرای مجدد از نقطهٔ قبلی ادامه پیدا کند.

</div>

In [100]:
# Code Cell 77
# NB13 — Resumable complete conditional reranking evaluation:
# all queries with a positive Child already present in top 10

conditional_reranker_top_k = 10
conditional_reranker_max_tokens = 1536
conditional_reranker_excerpt_char_limit = 1000

conditional_reranker_cache_path = (
    active_benchmark_run_directory
    / "arvan_chat_conditional_reranking_cache.json"
)

conditional_reranker_report_path = (
    active_benchmark_run_directory
    / "arvan_chat_conditional_reranking_evaluation.json"
)

if conditional_reranker_cache_path.is_file():
    conditional_reranker_cache = json.loads(
        conditional_reranker_cache_path.read_text(encoding="utf-8")
    )
else:
    conditional_reranker_cache = {
        "active_golden_dataset_sha256": golden_dataset_context["sha256"],
        "model": arvan_chat_model,
        "candidate": best_candidate_name,
        "top_k": conditional_reranker_top_k,
        "max_tokens": conditional_reranker_max_tokens,
        "results_by_candidate_id": {},
    }

assert (
    conditional_reranker_cache["active_golden_dataset_sha256"]
    == golden_dataset_context["sha256"]
), "The existing reranker cache belongs to a different Golden Dataset."

assert conditional_reranker_cache["model"] == arvan_chat_model, (
    "The existing reranker cache belongs to a different chat model."
)

conditional_query_indexes = eligible_reranker_query_indexes

print("=== RESUMABLE CONDITIONAL RERANKING EVALUATION ===")
print("Eligible queries:", len(conditional_query_indexes))
print(
    "Previously cached:",
    len(conditional_reranker_cache["results_by_candidate_id"]),
)
print("Top-k passages:", conditional_reranker_top_k)
print("Maximum completion tokens:", conditional_reranker_max_tokens)

for execution_position, query_index in enumerate(
    conditional_query_indexes,
    start=1,
):
    query_record = arvan_hybrid_query_records[query_index]
    candidate_id = query_record["candidate_id"]

    if candidate_id in conditional_reranker_cache["results_by_candidate_id"]:
        print(
            f"Query {execution_position}/{len(conditional_query_indexes)} "
            f"| cached | candidate_id={candidate_id}"
        )
        continue

    seed_id = query_record["source_seed_id"]
    positive_indexes = best_positive_indexes_by_seed[seed_id]
    top_k_indexes = best_rrf_ranked_indexes[query_index][
        :conditional_reranker_top_k
    ]

    passage_labels = [
        f"P{rank:02d}"
        for rank in range(1, conditional_reranker_top_k + 1)
    ]

    passage_by_label = {
        label: best_candidate_records[int(candidate_index)]
        for label, candidate_index in zip(
            passage_labels,
            top_k_indexes,
        )
    }

    prompt_passages = "\n\n".join(
        (
            f"[{label}]\n"
            f"{passage_by_label[label]['content'][:conditional_reranker_excerpt_char_limit]}"
        )
        for label in passage_labels
    )

    reranker_prompt = f"""
فقط Passageهای زیر را برای پاسخ‌گویی به پرسش رتبه‌بندی کن.

قواعد:
- فقط بر اساس متن Passageها تصمیم بگیر.
- از دانش بیرونی استفاده نکن.
- همهٔ برچسب‌ها را دقیقاً یک‌بار بیاور.
- پاسخ را فوراً و فقط به شکل JSON معتبر بده.
- هیچ توضیح، تحلیل یا متن دیگری ننویس.

ساختار دقیق خروجی:
{{"ranking":["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10"]}}

پرسش:
{query_record["question"]}

Passageها:
{prompt_passages}
""".strip()

    original_positive_rank = next(
        rank
        for rank, candidate_index in enumerate(top_k_indexes, start=1)
        if int(candidate_index) in positive_indexes
    )

    attempts = []

    for attempt_number, max_tokens in enumerate(
        (conditional_reranker_max_tokens, 2048),
        start=1,
    ):
        started_at = time.perf_counter()

        response = arvan_chat_client.chat.completions.create(
            model=arvan_chat_model,
            messages=[
                {
                    "role": "user",
                    "content": reranker_prompt,
                }
            ],
            temperature=0,
            max_tokens=max_tokens,
        )

        elapsed_seconds = time.perf_counter() - started_at
        choice = response.choices[0]
        raw_response_text = (choice.message.content or "").strip()

        reranked_labels = extract_reranked_labels(
            raw_response_text,
            allowed_labels=set(passage_labels),
        )

        attempts.append(
            {
                "attempt_number": attempt_number,
                "max_tokens": max_tokens,
                "finish_reason": choice.finish_reason,
                "seconds": round(elapsed_seconds, 3),
                "cost": getattr(
                    getattr(response, "usage", None),
                    "cost",
                    None,
                ),
                "raw_response": raw_response_text,
                "parsed": reranked_labels is not None,
            }
        )

        if reranked_labels is not None:
            break

    result = {
        "candidate_id": candidate_id,
        "source_seed_id": seed_id,
        "question": query_record["question"],
        "original_positive_rank": original_positive_rank,
        "attempts": attempts,
        "response_parse_status": (
            "PASS"
            if reranked_labels is not None
            else "FAIL"
        ),
    }

    if reranked_labels is not None:
        reranked_candidate_indexes = [
            int(top_k_indexes[passage_labels.index(label)])
            for label in reranked_labels
        ]

        reranked_positive_rank = next(
            rank
            for rank, candidate_index in enumerate(
                reranked_candidate_indexes,
                start=1,
            )
            if candidate_index in positive_indexes
        )

        result["reranked_positive_rank"] = reranked_positive_rank
        result["rank_change"] = (
            original_positive_rank - reranked_positive_rank
        )

    conditional_reranker_cache["results_by_candidate_id"][
        candidate_id
    ] = result

    conditional_reranker_cache_path.write_text(
        json.dumps(
            conditional_reranker_cache,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"Query {execution_position}/{len(conditional_query_indexes)} | "
        f"seed={seed_id} | "
        f"original_rank={original_positive_rank} | "
        f"parse={result['response_parse_status']}"
    )

conditional_reranker_results = list(
    conditional_reranker_cache["results_by_candidate_id"].values()
)

parsed_conditional_results = [
    result
    for result in conditional_reranker_results
    if result["response_parse_status"] == "PASS"
]

conditional_reranker_summary = {
    "eligible_query_count": len(conditional_query_indexes),
    "completed_query_count": len(conditional_reranker_results),
    "parse_success_rate": (
        len(parsed_conditional_results)
        / len(conditional_reranker_results)
        if conditional_reranker_results
        else 0.0
    ),
    "mean_original_positive_rank": (
        float(
            np.mean(
                [
                    result["original_positive_rank"]
                    for result in parsed_conditional_results
                ]
            )
        )
        if parsed_conditional_results
        else None
    ),
    "mean_reranked_positive_rank": (
        float(
            np.mean(
                [
                    result["reranked_positive_rank"]
                    for result in parsed_conditional_results
                ]
            )
        )
        if parsed_conditional_results
        else None
    ),
    "improved_query_count": int(
        np.sum(
            [
                result["rank_change"] > 0
                for result in parsed_conditional_results
            ]
        )
    ),
    "unchanged_query_count": int(
        np.sum(
            [
                result["rank_change"] == 0
                for result in parsed_conditional_results
            ]
        )
    ),
    "worsened_query_count": int(
        np.sum(
            [
                result["rank_change"] < 0
                for result in parsed_conditional_results
            ]
        )
    ),
    "total_observed_cost": float(
        np.sum(
            [
                attempt["cost"] or 0.0
                for result in conditional_reranker_results
                for attempt in result["attempts"]
            ]
        )
    ),
}

conditional_reranker_report_path.write_text(
    json.dumps(
        {
            "active_golden_dataset_sha256": golden_dataset_context["sha256"],
            "model": arvan_chat_model,
            "retrieval_configuration": (
                "arvan_text_embedding_3_large_plus_persian_tfidf_rrf"
            ),
            "candidate": best_candidate_name,
            "top_k": conditional_reranker_top_k,
            "summary": conditional_reranker_summary,
            "results": conditional_reranker_results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nCompleted:", conditional_reranker_summary["completed_query_count"])
print(
    "Parse success:",
    f"{conditional_reranker_summary['parse_success_rate']:.4f}",
)
print(
    "Mean original positive rank:",
    conditional_reranker_summary["mean_original_positive_rank"],
)
print(
    "Mean reranked positive rank:",
    conditional_reranker_summary["mean_reranked_positive_rank"],
)
print(
    "Improved / unchanged / worsened:",
    conditional_reranker_summary["improved_query_count"],
    "/",
    conditional_reranker_summary["unchanged_query_count"],
    "/",
    conditional_reranker_summary["worsened_query_count"],
)
print(
    "Total observed cost:",
    f"{conditional_reranker_summary['total_observed_cost']:.6f}",
)
print("Report:", conditional_reranker_report_path)
print("\nRESUMABLE CONDITIONAL RERANKING EVALUATION: PASS")

=== RESUMABLE CONDITIONAL RERANKING EVALUATION ===
Eligible queries: 38
Previously cached: 0
Top-k passages: 10
Maximum completion tokens: 1536
Query 1/38 | seed=21 | original_rank=3 | parse=PASS
Query 2/38 | seed=21 | original_rank=1 | parse=PASS
Query 3/38 | seed=21 | original_rank=1 | parse=PASS
Query 4/38 | seed=21 | original_rank=3 | parse=PASS
Query 5/38 | seed=21 | original_rank=6 | parse=PASS
Query 6/38 | seed=21 | original_rank=7 | parse=PASS
Query 7/38 | seed=21 | original_rank=2 | parse=PASS
Query 8/38 | seed=21 | original_rank=5 | parse=PASS
Query 9/38 | seed=21 | original_rank=1 | parse=PASS
Query 10/38 | seed=12 | original_rank=1 | parse=PASS
Query 11/38 | seed=12 | original_rank=3 | parse=PASS
Query 12/38 | seed=3 | original_rank=2 | parse=PASS
Query 13/38 | seed=3 | original_rank=2 | parse=PASS
Query 14/38 | seed=3 | original_rank=10 | parse=PASS
Query 15/38 | seed=3 | original_rank=2 | parse=PASS
Query 16/38 | seed=5 | original_rank=7 | parse=PASS
Query 17/38 | seed=5 